# Unified Runner — VCM 3.4

Single entry point for every workflow in this repo. Set `CASE` in the next
cell and run the notebook top to bottom; the dispatcher at the bottom calls
the matching `run_<case>()` function.

Available cases (matching the per-case notebooks in this directory):
- `bem_displacement`
- `crack_net`
- `direct_method_verification`
- `disk_compression_2`
- `disk_compression_inclined`
- `disk_energetics`
- `disk_experiments`
- `function_diagnostics`
- `graph_net`
- `propagation`
- `validation`

Each `run_<case>()` is a faithful inlining of the corresponding notebook's
code cells. Markdown cells from the originals become the function docstring.
If a case notebook changes, regenerate this file with `_build_unified_runner.py`.


## 1. Case selector


In [ ]:
# Pick which workflow to run. Must be one of the keys in CASES at the bottom.
CASE = "bem_displacement"

# Safety guard: keep DRY_RUN = True the first time you open this notebook so
# Run-All only defines the case functions without executing them. Set to
# False to actually invoke run_<CASE>() at the bottom.
DRY_RUN = True


## 2. Common preamble (imports + repo path)


In [ ]:
from __future__ import annotations  # hoisted from per-case notebooks

from pathlib import Path
import os, sys
import numpy as np  # noqa: F401  (every case imports it)
import matplotlib.pyplot as plt  # noqa: F401  (most cases need it)

nb_dir = Path(os.getcwd()).resolve()
repo_root = nb_dir.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Several legacy notebooks did `from preamble import *` at the top of every
# cell, which is illegal inside a function body. Hoist that wildcard import
# here at module scope so every case function inherits the same namespace,
# and the per-case bodies (stripped of their own `from preamble import *`
# lines by _build_unified_runner.py) continue to work unchanged.
from preamble import *  # noqa: F401,F403

print(f'Running case: {CASE!r}')
print(f'Repo root:    {repo_root}')


## 3. Per-case workflows

Each cell below defines `run_<case>()`. Only the function matching `CASE`
is invoked by the dispatcher; the others are defined but not executed.


### Case `bem_displacement` (from `BEM_displacement.ipynb`)


In [ ]:
def run_bem_displacement():
    """Case extracted from BEM_displacement.ipynb."""
    from pathlib import Path
    import os, sys
    import numpy as np

    # -----------------------
    # Repo import path
    # -----------------------
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    from fracture_utils.Ubem.brazilian_disk_bem import BrazilianDiskParams
    from fracture_utils.Ubem.boundary_conditions import build_boundary, add_boundary_to_solver
    from fracture_utils.Ubem.bem_solver import BEMSolver2D
    from fracture_utils.Ubem.bem_stress_field import circle_inside
    from fracture_utils.Ubem.bem_stress_plotter import (
        ContourOpts,
        eval_and_plot_stress_components_contours_separate,
    )
    from fracture_utils.Ubem.direct_kkt_coupling import build_bem_dense_operators


    # -----------------------
    # User config
    # -----------------------
    # Choose one:
    #   "traction"   -> external pressure tractions on top/bottom arcs
    #   "imposed_uy" -> impose u_y on top/bottom arcs, keep u_x free (traction-free in x)
    BC_MODE = "imposed_uy"

    # Used only when BC_MODE == "imposed_uy" (meters)
    # Top arc displacement (compressive: negative)
    U_Y_TOP_IMPOSED = -4.0e-6
    # Bottom arc displacement to suppress rigid-body vertical translation
    U_Y_BOTTOM_IMPOSED = 0.0

    # Same disk geometry/material set used in previous simulations
    disk = BrazilianDiskParams(
        R=25.4e-3 / 2,
        P_total=3.8e3,
        E=231.52e9,
        nu=0.3,
        n_elem=120,
        gauss_n=6,
        plane_strain=True,
        arc_half_angle_deg=15.0,
        h=6.35e-3,
        n_grid=120,
        pad_frac=0.03,
    )

    out_dir = repo_root / "output" / "BEM_Brazilian_disk_BC_selector"
    out_dir.mkdir(parents=True, exist_ok=True)


    # -----------------------
    # Utilities
    # -----------------------
    def _build_disk_mesh(params: BrazilianDiskParams):
        return build_boundary({"type": "circle", "R": float(params.R), "n_boundary": int(params.n_elem), "center": (0.0, 0.0)})


    def _arc_masks(mesh, params: BrazilianDiskParams):
        theta_deg = np.asarray(mesh.theta_deg, dtype=float)
        arc_half = float(params.arc_half_angle_deg)
        top = (theta_deg >= 90.0 - arc_half) & (theta_deg <= 90.0 + arc_half)
        bot = (theta_deg >= -90.0 - arc_half) & (theta_deg <= -90.0 + arc_half)
        return top, bot


    def _disk_external_boundary_tractions(mesh, params: BrazilianDiskParams):
        theta_deg = np.asarray(mesh.theta_deg, dtype=float)
        L = np.asarray(mesh.length, dtype=float)

        arc_half = float(params.arc_half_angle_deg)
        top = (theta_deg >= 90 - arc_half) & (theta_deg <= 90 + arc_half)
        bot = (theta_deg >= -90 - arc_half) & (theta_deg <= -90 + arc_half)

        L_top = float(np.sum(L[top]))
        L_bot = float(np.sum(L[bot]))
        if L_top <= 0 or L_bot <= 0:
            raise RuntimeError("Top/bottom loaded arc has zero length.")

        h = float(params.h)
        pressure_top = float(params.P_total) / (L_top * h)
        pressure_bot = float(params.P_total) / (L_bot * h)

        tx = np.zeros(mesh.n_seg, float)
        ty = np.zeros(mesh.n_seg, float)

        # pressure_normal => traction = -p * n_out
        tx[top] += -pressure_top * np.asarray(mesh.nx, float)[top]
        ty[top] += -pressure_top * np.asarray(mesh.ny, float)[top]

        tx[bot] += -pressure_bot * np.asarray(mesh.nx, float)[bot]
        ty[bot] += -pressure_bot * np.asarray(mesh.ny, float)[bot]

        return tx, ty, top, bot


    def _solve_with_tractions(params: BrazilianDiskParams):
        mesh = _build_disk_mesh(params)
        tx, ty, _, _ = _disk_external_boundary_tractions(mesh, params)

        solver = BEMSolver2D(E=float(params.E), nu=float(params.nu), h=float(params.h), plane_strain=bool(params.plane_strain))
        is_traction = np.ones(mesh.n_seg, dtype=bool)
        add_boundary_to_solver(solver, mesh, is_traction, tx, ty)
        solver.solve(gauss_n=int(params.gauss_n))
        return solver, mesh


    def _solve_with_imposed_uy(params: BrazilianDiskParams, uy_top: float, uy_bottom: float = 0.0):
        mesh = _build_disk_mesh(params)
        top, bot = _arc_masks(mesh, params)
        n = int(mesh.n_seg)

        # Build geometric segments for dense operators
        solver = BEMSolver2D(E=float(params.E), nu=float(params.nu), h=float(params.h), plane_strain=bool(params.plane_strain))
        for i in range(n):
            solver.add_element(
                float(mesh.x1[i]), float(mesh.y1[i]),
                float(mesh.x2[i]), float(mesh.y2[i]),
                True, 0.0, 0.0,
            )

        CplusH, G = build_bem_dense_operators(
            segments=solver.segs,
            E=float(params.E),
            nu=float(params.nu),
            plane_strain=bool(params.plane_strain),
            gauss_n=int(params.gauss_n),
        )

        # Unknown y = [u(2n), t(2n)] with interleaved components per segment
        C_bem = np.hstack([CplusH, -G])

        # BC rows (2n):
        #  - all segments: t_x = 0  (u_x free)
        #  - top arc:      u_y = uy_top
        #  - bottom arc:   u_y = uy_bottom (default 0 to suppress rigid-body translation)
        #  - remaining arc: t_y = 0
        C_bc = np.zeros((2 * n, 4 * n), float)
        d = np.zeros((2 * n,), float)

        for i in range(n):
            r0 = 2 * i
            r1 = 2 * i + 1

            # t_x = 0
            C_bc[r0, 2 * n + 2 * i + 0] = 1.0

            if top[i]:
                C_bc[r1, 2 * i + 1] = 1.0
                d[r1] = float(uy_top)
            elif bot[i]:
                C_bc[r1, 2 * i + 1] = 1.0
                d[r1] = float(uy_bottom)
            else:
                C_bc[r1, 2 * n + 2 * i + 1] = 1.0

        A = np.vstack([C_bem, C_bc])
        b = np.concatenate([np.zeros((2 * n,), float), d])

        # Least-squares handles the remaining x-translation gauge mode robustly.
        y, *_ = np.linalg.lstsq(A, b, rcond=None)

        u = y[: 2 * n]
        t = y[2 * n :]

        solver.u_x = np.asarray(u[0::2], float).copy()
        solver.u_y = np.asarray(u[1::2], float).copy()
        solver.t_x = np.asarray(t[0::2], float).copy()
        solver.t_y = np.asarray(t[1::2], float).copy()
        solver._colloc_xy = np.column_stack([np.asarray(mesh.xm, float), np.asarray(mesh.ym, float)])

        return solver, mesh


    def _plot_stress_components(solver, params: BrazilianDiskParams, out: Path, basename: str):
        bbox = (-params.R, params.R, -params.R, params.R)
        inside = circle_inside(float(params.R), center=(0.0, 0.0), pad=float(params.pad_frac) * float(params.R))

        opts_common = dict(
            dpi=300,
            show=True,
            robust=True,
            robust_pct=97.0,
            n_levels=30,
            n_line_levels=20,
            x_scale=1e3,
            y_scale=1e3,
            x_label="x [mm]",
            y_label="y [mm]",
            value_scale=1e-6,
            cbar_label="Stress [MPa]",
            cmap="jet",
        )

        opts_xx = ContourOpts(**opts_common, title=r"$\sigma_{xx}$", symmetric=True)
        opts_yy = ContourOpts(**opts_common, title=r"$\sigma_{yy}$", symmetric=True)
        opts_xy = ContourOpts(**opts_common, title=r"$\sigma_{xy}$", symmetric=True)

        xs, ys, Sxx, Syy, Sxy, _ = eval_and_plot_stress_components_contours_separate(
            solver,
            bbox=bbox,
            out_dir=out,
            basename=basename,
            n=int(params.n_grid),
            inside=inside,
            normalize_by=None,
            opts_xx=opts_xx,
            opts_yy=opts_yy,
            opts_xy=opts_xy,
        )

        np.save(out / f"{basename}_xs.npy", xs)
        np.save(out / f"{basename}_ys.npy", ys)
        np.save(out / f"{basename}_Sxx.npy", Sxx)
        np.save(out / f"{basename}_Syy.npy", Syy)
        np.save(out / f"{basename}_Sxy.npy", Sxy)

        return xs, ys, Sxx, Syy, Sxy


    # -----------------------
    # Run selected boundary condition mode
    # -----------------------
    mode = str(BC_MODE).strip().lower()
    if mode == "traction":
        solver, mesh = _solve_with_tractions(disk)
        base = "disk_traction"
    elif mode == "imposed_uy":
        solver, mesh = _solve_with_imposed_uy(
            disk,
            uy_top=float(U_Y_TOP_IMPOSED),
            uy_bottom=float(U_Y_BOTTOM_IMPOSED),
        )
        base = "disk_imposed_uy"
    else:
        raise ValueError("BC_MODE must be 'traction' or 'imposed_uy'.")

    xs, ys, Sxx, Syy, Sxy = _plot_stress_components(solver, disk, out_dir, basename=base)

    print(f"[done] mode={mode}")
    print(f"[saved] contours + arrays in: {out_dir}")

    return locals()


### Case `crack_net` (from `crack_net.ipynb`)


In [ ]:
def run_crack_net():
    """Case extracted from crack_net.ipynb."""
    import os, sys
    from pathlib import Path

    # Add repo root (one level above notebooks/) to sys.path
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    # --- Preamble re-exports
    out_dir = Path(repo_root) / "output" / "90DegKink"
    out_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # Define Y network: vertices + connectivity
    # Units: meters
    # -------------------------
    mm = 1e-3
    vertices = np.array([
        [0,   0.0*mm,  0.0*mm],   # v0 junction
        [1,   5.0*mm,  0*mm], # v1 upper-right
        [2,   6*mm, -20.0*mm], # v2 lower-right
        # [3,   5.0*mm, -4.0*mm], # v3 upper-left
    ], dtype=float)

    # edges: (edge_id, v0, v1)
    connectivity = np.array([
        [0, 1],
        [1, 2],
        # [1, 3],
    ], dtype=int)

    net = CrackNetworkV4.from_vertices_connectivity(
        vertices=vertices,
        connectivity=connectivity,
        Nv_max=3,
        validate=True,
    )

    material = Material(E=200e9, nu=0.30, plane_stress=False)
    applied  = AppliedStress(sigma_xx=0e6, sigma_yy=-100e6, sigma_xy=0.0)

    calc = DCENetworkStaticV4(material, net, applied)

    sol  = calc.solve(
        ne_half=60,
        representation="cheb_quad",
        node_distribution="uniform",
        solver_option="parametrized_crack",
        parametrization="polyline",
        nq_col=12,
        nq_stress=16,
        crack_mode="half",
        enforce_cod_nonnegative=True,
        cod_tol=1e-10,
        cod_max_iter=25,
    )


    res = DCEResultsNetworkV4(calc, sol)
    plotter = DCEPlotterV4(res, out_dir=out_dir)

    # Graph topology
    plotter.plot_network_graph()

    # Displacement vector field
    plotter.plot_displacement_vector_field(
        extent_factor=1.2,
        n_grid=41,
        scale=1.0,
        disp_scale=5e4,
        mask_cracks=False,
    )

    # Stress fields (global)
    opts = StressPlotOptsV4(
        extent_factor=3,
        n_grid=400,
        add_remote=True,
        mask_cracks=False,
        cmap="jet",
        n_bands=40,
        label_contours=False,
        vmax_factor=2
    )
    plotter.plot_stress_components_global(opts=opts, components=("sxx","syy","sxy"))

    plt.show()
    print("Saved figures to:", out_dir)


    # ----------------------------
    # Plot: deformed network (trim junction faces; smooth COD/CSD if desired)
    # ----------------------------
    pl_def = DCEPlotterDeformedV4(res, out_dir=out_dir)
    pl_def.plot_deformed_network(
        n_pts_per_edge=200,
        scale=1e2,
        show_faces=True,
        units="mm",
        cod_smooth_window=5,
        csd_smooth_window=5,
        junction_model="core",
    )


    # ---- next cell ----

    import os, sys
    from pathlib import Path

    # Add repo root (one level above notebooks/) to sys.path
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    # --- Preamble re-exports
    out_dir = Path(repo_root) / "output" / "StarJunctionPolyline"
    out_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # Define star-junction network: vertices + connectivity
    mm = 1e-3
    vertices = np.array([
        [0,  0.0*mm,  0.0*mm],   # junction
        [1,  5.0*mm, - 5.0*mm],   # left tip
        [2, 5.0*mm,  5.0*mm],   # upper tip
        [3, -5.0*mm, 5.0*mm],
        [4, -5.0*mm, -5.0*mm], 
        [5, 0.0*mm, -15.0*mm], 
        [6, -5.0*mm, -20.0*mm],   # lower tip
        [7, -15.0*mm, -20.0*mm],   # upper tip
        [8, -10.0*mm, -10.0*mm],
        [9, -0.5*mm, -10.0*mm],
        ], dtype=float)

    # Each edge: [start_node, end_node]
    connectivity = [
        [0, 1],  # Edge 0: from node 0 to node 1
        [0, 2],  # Edge 1: from node 0 to node 2
        [0, 3],  # Edge 2: from node 0 to node 3
        [0, 4],  # Edge 3: from node 0 to node 4
        [0, 5],  # Edge 4: from node 0 to node 5
        [5, 6],  # Edge 5: from node 5 to node 6
        [6, 7],  # Edge 6: from node 6 to node 7
        [8, 9],  # Edge 7: from node 8 to node 9
    ]

    generator = CrackNetworkGenerator(domain_size=(25e-3, 25e-3))
    net = CrackNetworkV4.from_vertices_connectivity(vertices=vertices, connectivity=connectivity, Nv_max=8, validate=True)
    material = Material(E=200e9, nu=0.30, plane_stress=False)
    applied  = AppliedStress(sigma_xx=0e6, sigma_yy=100e6, sigma_xy=0.0)

    calc = DCENetworkStaticV4(material, net, applied)
    sol  = calc.solve(
        ne_half=50,
        representation="cheb_quad",
        node_distribution="uniform",
        solver_option="parametrized_crack",
        parametrization="cubic_spline",
        nq_col=12,
        nq_stress=16,
        crack_mode="half",
        junction_model= "core"
    )

    res = DCEResultsNetworkV4(calc, sol)

    # ----------------------------
    # Plot: network graph + stress + displacements
    # ----------------------------
    plotter = DCEPlotterV4(res, out_dir=out_dir)

    _call(plotter, "plot_network_graph")

    opts = StressPlotOptsV4(
        extent_factor=3,
        n_grid=301,
        add_remote=True,
        mask_cracks=False,
        cmap="jet",
        n_bands=40,
        label_contours=False,
        vmax_factor=2,
    )
    _call(plotter, "plot_stress_components_global", opts=opts, components=("sxx","syy","sxy"))

    # If your core plotter has displacement methods, this will use them automatically.
    # Otherwise, it will just skip.
    _call(plotter, "plot_displacement_vector_field", disp_scale=5e4, mask_cracks=False)

    # ----------------------------
    # Plot: deformed network (trim junction faces; smooth COD/CSD if desired)
    # ----------------------------
    pl_def = DCEPlotterDeformedV4(res, out_dir=out_dir)
    _call(
        pl_def,
        "plot_deformed_network",
        scale=2e1,
        trim_core_junction_faces=True,
    )

    # ----------------------------
    # Vectors: B-content and PK forces
    # ----------------------------
    pkplt = DCEPlotterPK(res, out_dir=out_dir)

    _call(
        pkplt,
        "plot_b_content_vectors",
        style=VecPlotStyle(max_len_factor=0.18),
        user_scale=1e-3,
        units="mm",
        show_tips=True,
        show_junctions=True,
        show_sum_at_junction=True,
        annotate=True,
    )

    _call(
        pkplt,
        "plot_pk_force_vectors",
        style=VecPlotStyle(max_len_factor=0.18),
        user_scale=1e-3,
        units="mm",
        exclude_self=True,
        show_tips=True,
        show_junctions=True,
        n_samples_tip=32,
        annotate=True,
    )

    plt.show()
    print("Saved figures to:", out_dir)

    # ----------------------------
    # Diagnostics (console)
    # ----------------------------
    #diagnostics.junction_constraint_diagnostics(calc=calc, sol=sol, min_degree=1)
    #diagnostics.jump_vector_diagnostics(calc=calc, sol=sol, min_degree=1, only_vertices=None)

    # ---- next cell ----

    import os, sys
    from pathlib import Path

    # Add repo root (one level above notebooks/) to sys.path
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    # --- Preamble re-exports
    out_dir = Path(repo_root) / "output" / "TreeBranchesSpline"
    out_dir.mkdir(parents=True, exist_ok=True)


    # ----------------------------
    # Problem definition
    # ----------------------------
    mm = 1e-3
    vertices = np.array([
        [1,  0.0*mm,  0.0*mm],
        [2,  5.0*mm,  5.0*mm],
        [3, -5.0*mm,  5.0*mm],
        [4,  8.0*mm,  5.0*mm],
        [5,  0.0*mm,  8.0*mm],
        [6, -8.0*mm,  5.0*mm],
        [7,  0.0*mm, -4.0*mm],
        [8, -4.0*mm, -8.0*mm],
    ], dtype=float)

    connectivity = np.array([[1,2],[1,3],[1,7],[7,8],[2,4],[2,5],[3,6]], dtype=int)

    net = CrackNetworkV4.from_vertices_connectivity(
        vertices=vertices,
        connectivity=connectivity,
        Nv_max=4,
        validate=True
    )

    material = Material(E=200e9, nu=0.30, plane_stress=False)
    applied  = AppliedStress(sigma_xx=50e6, sigma_yy=100e6, sigma_xy=0.0)

    calc = DCENetworkStaticV4(material, net, applied)

    sol  = calc.solve(
        ne_half=50,
        representation="cheb_quad",
        node_distribution="tip_dense",
        solver_option="parametrized_crack",
        parametrization="cspline",
        nq_col=12,
        nq_stress=16,
        crack_mode="half",
        junction_model="core",
    )

    res = DCEResultsNetworkV4(calc, sol)

    # ----------------------------
    # Plot: network graph + stress + displacements
    # ----------------------------
    plotter = DCEPlotterV4(res, out_dir=out_dir)

    _call(plotter, "plot_network_graph")

    opts = StressPlotOptsV4(
        extent_factor=3,
        n_grid=301,
        add_remote=True,
        mask_cracks=False,
        cmap="jet",
        n_bands=40,
        label_contours=False,
        vmax_factor=2,
    )
    _call(plotter, "plot_stress_components_global", opts=opts, components=("sxx","syy","sxy"))

    # If your core plotter has displacement methods, this will use them automatically.
    # Otherwise, it will just skip.
    _call(plotter, "plot_displacement_vector_field", disp_scale=5e4, mask_cracks=False)

    # ----------------------------
    # Plot: deformed network (trim junction faces; smooth COD/CSD if desired)
    # ----------------------------
    pl_def = DCEPlotterDeformedV4(res, out_dir=out_dir)
    _call(
        pl_def,
        "plot_deformed_network",
        scale=0.8e1,
        trim_core_junction_faces=True,
    )

    # ----------------------------
    # Vectors: B-content and PK forces
    # ----------------------------
    pkplt = DCEPlotterPK(res, out_dir=out_dir)

    _call(
        pkplt,
        "plot_b_content_vectors",
        style=VecPlotStyle(max_len_factor=0.18),
        user_scale=1e-3,
        units="mm",
        show_tips=True,
        show_junctions=True,
        show_sum_at_junction=True,
        annotate=True,
    )

    _call(
        pkplt,
        "plot_pk_force_vectors",
        style=VecPlotStyle(max_len_factor=0.18),
        user_scale=1e-3,
        units="mm",
        exclude_self=True,
        show_tips=True,
        show_junctions=True,
        n_samples_tip=32,
        annotate=True,
    )

    plt.show()
    print("Saved figures to:", out_dir)

    # ----------------------------
    # Diagnostics (console)
    # ----------------------------
    #diagnostics.junction_constraint_diagnostics(calc=calc, sol=sol, min_degree=1)
    #diagnostics.jump_vector_diagnostics(calc=calc, sol=sol, min_degree=1, only_vertices=None)


    # ---- next cell ----

    import os, sys
    from pathlib import Path

    # Add repo root (one level above notebooks/) to sys.path
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    # --- Preamble re-exports
    out_dir = Path(repo_root) / "output" / "3SegmentCrackSpline"
    out_dir.mkdir(parents=True, exist_ok=True)


    # -------------------------
    # Define Y network: vertices + connectivity
    # Units: meters
    # -------------------------
    mm = 1e-3
    vertices = np.array([
        [0,   0.0*mm,  0.0*mm],   # v0 junction
        [1,   5.0*mm,  0*mm], # v1 upper-right
        [2,   5.0*mm, -5.0*mm], # v2 lower-right
        [3,  0*mm,  -10.0*mm], # v3 extended lower-right
    ], dtype=float)

    # edges: (edge_id, v0, v1)
    connectivity = np.array([
        [0, 1],
        [1, 2],
        [2, 3]
    ], dtype=int)

    net = CrackNetworkV4.from_vertices_connectivity(
        vertices=vertices,
        connectivity=connectivity,
        Nv_max=4,
        validate=True
    )

    material = Material(E=200e9, nu=0.30, plane_stress=False)
    applied  = AppliedStress(sigma_xx=50e6, sigma_yy=100e6, sigma_xy=0.0)

    calc = DCENetworkStaticV4(material, net, applied)

    sol  = calc.solve(
        ne_half=50,
        representation="cheb_quad",
        node_distribution="tip_dense",
        solver_option="parametrized_crack",
        parametrization="cspline",
        nq_col=12,
        nq_stress=16,
        crack_mode="half",
        junction_model="core",
    )

    res = DCEResultsNetworkV4(calc, sol)

    # ----------------------------
    # Plot: network graph + stress + displacements
    # ----------------------------
    plotter = DCEPlotterV4(res, out_dir=out_dir)

    _call(plotter, "plot_network_graph")

    opts = StressPlotOptsV4(
        extent_factor=3,
        n_grid=301,
        add_remote=True,
        mask_cracks=False,
        cmap="jet",
        n_bands=40,
        label_contours=False,
        vmax_factor=2,
    )
    _call(plotter, "plot_stress_components_global", opts=opts, components=("sxx","syy","sxy"))

    # If your core plotter has displacement methods, this will use them automatically.
    # Otherwise, it will just skip.
    _call(plotter, "plot_displacement_vector_field", disp_scale=5e4, mask_cracks=False)

    # ----------------------------
    # Plot: deformed network (trim junction faces; smooth COD/CSD if desired)
    # ----------------------------
    pl_def = DCEPlotterDeformedV4(res, out_dir=out_dir)
    _call(
        pl_def,
        "plot_deformed_network",
        scale=5e1,
        trim_core_junction_faces=True,
    )

    # ----------------------------
    # Vectors: B-content and PK forces
    # ----------------------------
    pkplt = DCEPlotterPK(res, out_dir=out_dir)

    _call(
        pkplt,
        "plot_b_content_vectors",
        style=VecPlotStyle(max_len_factor=0.18),
        user_scale=1e-3,
        units="mm",
        show_tips=True,
        show_junctions=True,
        show_sum_at_junction=True,
        annotate=True,
    )

    _call(
        pkplt,
        "plot_pk_force_vectors",
        style=VecPlotStyle(max_len_factor=0.18),
        user_scale=1e-3,
        units="mm",
        exclude_self=True,
        show_tips=True,
        show_junctions=True,
        n_samples_tip=32,
        annotate=True,
    )

    plt.show()
    print("Saved figures to:", out_dir)


    # ----------------------------
    # Diagnostics (console)
    # ----------------------------
    #diagnostics.junction_constraint_diagnostics(calc=calc, sol=sol, min_degree=1)
    #diagnostics.jump_vector_diagnostics(calc=calc, sol=sol, min_degree=1, only_vertices=None)

    # ---- next cell ----


    import os, sys
    from pathlib import Path

    # Add repo root (one level above notebooks/) to sys.path
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    # --- Preamble re-exports
    out_dir = Path(repo_root) / "output" / "IntersectingNetwork"
    out_dir.mkdir(parents=True, exist_ok=True)

    # ----------------------------
    # Problem definition
    # ----------------------------
    mm = 1e-3
    generator = CrackNetworkGenerator(
        domain_size=(20*mm, 20*mm),
        seed=42
    )

    generator.generate_network(
        n_junctions=0,              # No junctions
        n_tips=0,                   # No isolated tips
        edge_length_mean=2*mm,
        edge_length_std=0.5*mm,
        min_edge_length=1*mm,
        max_edge_length=3*mm,
        
        # Connected crack paths
        n_crack_paths=5,                    # 5 long cracks
        segments_per_path_mean=10,          # Average 10 segments each
        segments_per_path_std=3,            # ±3 segments variation
        path_angle_change_mean_deg=0.0,     # Straight (no systematic curvature)
        path_angle_change_std_deg=10.0,     # Small random wiggles (±10°)
        
        allow_intersections=True,
        detect_intersections=True,
        min_edge_distance=1.0*mm
    )

    fig, ax = generator.visualize(node_size=60, edge_width=2.0)
    plt.show()

    # Print statistics
    generator.print_statistics()

    fig2, ax2 = generator.visualize(
        figsize=(12, 12),
        node_size=20,
        edge_width=2.0,
        show_labels=True,
        show_edge_labels=True,
        edge_color='black'
    )
    plt.show()
    vertices, connectivity, vertex_types = generator.to_arrays()

    net = CrackNetworkV4.from_vertices_connectivity(
        vertices=vertices,
        connectivity=connectivity,
        Nv_max=5,
        validate=True,
    )

    material = Material(E=200e9, nu=0.30, plane_stress=False)
    applied  = AppliedStress(sigma_xx=50e6, sigma_yy=100e6, sigma_xy=0.0)

    calc = DCENetworkStaticV4(material, net, applied)

    sol  = calc.solve(
        ne_half=40,
        representation="cheb_quad",
        node_distribution="tip_dense",
        solver_option="parametrized_crack",
        parametrization="polyline",
        nq_col=12,
        nq_stress=16,
        crack_mode="half",
        junction_model="core",
    )

    res = DCEResultsNetworkV4(calc, sol)

    # ----------------------------
    # Plot: network graph + stress + displacements
    # ----------------------------
    plotter = DCEPlotterV4(res, out_dir=out_dir)

    _call(plotter, "plot_network_graph")

    opts = StressPlotOptsV4(
        extent_factor=3,
        n_grid=301,
        add_remote=True,
        mask_cracks=False,
        cmap="jet",
        n_bands=40,
        label_contours=False,
        vmax_factor=2,
    )
    _call(plotter, "plot_stress_components_global", opts=opts, components=("sxx","syy","sxy"))

    # If your core plotter has displacement methods, this will use them automatically.
    # Otherwise, it will just skip.
    _call(plotter, "plot_displacement_vector_field", disp_scale=5e4, mask_cracks=False)

    # ----------------------------
    # Plot: deformed network (trim junction faces; smooth COD/CSD if desired)
    # ----------------------------
    pl_def = DCEPlotterDeformedV4(res, out_dir=out_dir)
    _call(
        pl_def,
        "plot_deformed_network",
        scale=6e-1,
        trim_core_junction_faces=True,
        # trim_junction_apply_mode="True",
        cod_smooth_window="5",   # change to 7/11/15 if you want manual control
        csd_smooth_window="5",
    )

    # ----------------------------
    # Vectors: B-content and PK forces
    # ----------------------------
    pkplt = DCEPlotterPK(res, out_dir=out_dir)

    _call(
        pkplt,
        "plot_b_content_vectors",
        style=VecPlotStyle(max_len_factor=0.18),
        user_scale=1e-3,
        units="mm",
        show_tips=True,
        show_junctions=True,
        show_sum_at_junction=True,
        annotate=True,
    )

    _call(
        pkplt,
        "plot_pk_force_vectors",
        style=VecPlotStyle(max_len_factor=0.18),
        user_scale=1e-3,
        units="mm",
        exclude_self=True,
        show_tips=True,
        show_junctions=True,
        n_samples_tip=32,
        annotate=True,
    )

    plt.show()
    print("Saved figures to:", out_dir)

    # ----------------------------
    # Diagnostics (console)
    # ----------------------------
    #diagnostics.junction_constraint_diagnostics(calc=calc, sol=sol, min_degree=1)
    #diagnostics.jump_vector_diagnostics(calc=calc, sol=sol, min_degree=1, only_vertices=None)



    # ---- next cell ----

    import os, sys
    from pathlib import Path

    # Add repo root (one level above notebooks/) to sys.path
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    # --- Preamble re-exports
    out_dir = Path(repo_root) / "output" / "Y_Junction"
    out_dir.mkdir(parents=True, exist_ok=True)


    # ----------------------------
    # Problem definition
    # ----------------------------
    mm = 1e-3
    vertices = np.array([
        [0,   0.0*mm,  0.0*mm],   # v0 (left end)
        [1,  10.0*mm,  0.0*mm],   # v1 (kink / interior vertex, degree 2)
        [2,  20.0*mm,  10*mm],   # v2 (right end)
        [3,  20.0*mm,  -10*mm],   # v3 (extra vertex, not used)
    ], dtype=float)

    connectivity = np.array([
        [0, 1],   # e0
        [1, 2],   # e1
        [1, 3]
    ], dtype=int)


    net = CrackNetworkV4.from_vertices_connectivity(
        vertices=vertices,
        connectivity=connectivity,
        Nv_max=4,
        validate=True
    )

    material = Material(E=200e9, nu=0.30, plane_stress=False)
    applied  = AppliedStress(sigma_xx=100e6, sigma_yy=-100e6, sigma_xy=0.0)

    calc = DCENetworkStaticV4(material, net, applied)

    sol  = calc.solve(
        ne_half=60,
        representation="cheb_quad",
        node_distribution="tip_dense",
        solver_option="parametrized_crack",
        parametrization="polyline",
        nq_col=12,
        nq_stress=16,
        crack_mode="half",
        junction_model="strict",
    )

    res = DCEResultsNetworkV4(calc, sol)


    # ----------------------------
    # Plot: network graph + stress + displacements
    # ----------------------------
    plotter = DCEPlotterV4(res, out_dir=out_dir)

    _call(plotter, "plot_network_graph")

    opts = StressPlotOptsV4(
        extent_factor=3,
        n_grid=301,
        add_remote=True,
        mask_cracks=False,
        cmap="jet",
        n_bands=40,
        label_contours=False,
        vmax_factor=2,
    )
    _call(plotter, "plot_stress_components_global", opts=opts, components=("sxx","syy","sxy"))
    _call(plotter, "plot_displacement_vector_field", disp_scale=2e4, mask_cracks=False)

    # ----------------------------
    # Plot: deformed network (trim junction faces; smooth COD/CSD if desired)
    # ----------------------------
    pl_def = DCEPlotterDeformedV4(res, out_dir=out_dir)
    _call(
        pl_def,
        "plot_deformed_network",
        scale=2e1,
        junction_model="strict",
    )

    # ----------------------------
    # Vectors: B-content and PK forces
    # ----------------------------
    pkplt = DCEPlotterPK(res, out_dir=out_dir)

    _call(
        pkplt,
        "plot_b_content_vectors",
        style=VecPlotStyle(max_len_factor=0.18),
        user_scale=1e-3,
        units="mm",
        show_tips=True,
        show_junctions=True,
        show_sum_at_junction=True,
        annotate=True,
    )

    _call(
        pkplt,
        "plot_pk_force_vectors",
        style=VecPlotStyle(max_len_factor=0.18),
        user_scale=1e-3,
        units="mm",
        exclude_self=True,
        show_tips=True,
        show_junctions=True,
        n_samples_tip=32,
        annotate=True,
    )

    plt.show()
    print("Saved figures to:", out_dir)

    # ----------------------------
    # Diagnostics (console)
    # ----------------------------
    #diagnostics.junction_constraint_diagnostics(calc=calc, sol=sol, min_degree=1)
    #diagnostics.jump_vector_diagnostics(calc=calc, sol=sol, min_degree=1, only_vertices=None)


    # ---- next cell ----

    import os, sys
    from pathlib import Path

    # Add repo root (one level above notebooks/) to sys.path
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    # --- Preamble re-exports
    out_dir = Path(repo_root) / "output" / "CircularKink"
    out_dir.mkdir(parents=True, exist_ok=True)

    # ----------------------------
    # Problem definition
    # ----------------------------
    mm = 1e-3
    V = np.array([
        [0, -10*mm,    0.0],     # endpoint (node)
        [1, 10*mm,  0.0],     # endpoint (node)
        [2,  0*mm,  10*mm],    # center (control)
        [3,  -10*mm, -30*mm],    
        [4, 10*mm,  30*mm],    
    ], float)

    E = np.array([
        [0, 1], 
        [3,0],
        [1,4],   
    ], int)


    net = CrackNetworkV4.from_vertices_connectivity(
        vertices=V,
        connectivity=E,
        Nv_max=4,
        validate=True,
        edge_kinds=["arc", "line", "line"],      # one per edge in E, same order
        edge_ctrl_vids=[(2,), (), ()],           # one per edge; only arc gets a center
        vertex_roles={2: "control"},
    )


    material = Material(E=200e9, nu=0.30, plane_stress=False)
    applied  = AppliedStress(sigma_xx=100.0e6, sigma_yy=200.0e6, sigma_xy=0.0)

    calc = DCENetworkStaticV4(material, net, applied)
    sol  = calc.solve(ne_half=40, parametrization="polyline", crack_mode="half",nq_col=12, nq_stress=16, junction_model="strict")


    res = DCEResultsNetworkV4(calc, sol)

    # ----------------------------
    # Plot: network graph + stress + displacements
    # ----------------------------
    plotter = DCEPlotterV4(res, out_dir=out_dir)

    _call(plotter, "plot_network_graph")

    opts = StressPlotOptsV4(
        extent_factor=3,
        n_grid=301,
        add_remote=True,
        mask_cracks=False,
        cmap="jet",
        n_bands=40,
        label_contours=False,
        vmax_factor=2,
    )
    _call(plotter, "plot_stress_components_global", opts=opts, components=("sxx","syy","sxy"))

    # If your core plotter has displacement methods, this will use them automatically.
    # Otherwise, it will just skip.
    _call(plotter, "plot_displacement_vector_field", disp_scale=3e4, mask_cracks=False)

    # ----------------------------
    # Plot: deformed network (trim junction faces; smooth COD/CSD if desired)
    # ----------------------------
    pl_def = DCEPlotterDeformedV4(res, out_dir=out_dir)
    pl_def.plot_deformed_network(
        n_pts_per_edge=200,
        scale=2e1,
        show_faces=True,
        units="mm",
        junction_model="strict",
    )

    # ----------------------------
    # Vectors: B-content and PK forces
    # ----------------------------
    pkplt = DCEPlotterPK(res, out_dir=out_dir)

    _call(
        pkplt,
        "plot_b_content_vectors",
        style=VecPlotStyle(max_len_factor=0.18),
        user_scale=1e-3,
        units="mm",
        show_tips=True,
        show_junctions=True,
        show_sum_at_junction=True,
        annotate=True,
    )

    _call(
        pkplt,
        "plot_pk_force_vectors",
        style=VecPlotStyle(max_len_factor=0.18),
        user_scale=1e-3,
        units="mm",
        exclude_self=True,
        show_tips=True,
        show_junctions=True,
        n_samples_tip=32,
        annotate=True,
    )

    plt.show()
    print("Saved figures to:", out_dir)

    # ----------------------------
    # Diagnostics (console)
    # ----------------------------
    #diagnostics.junction_constraint_diagnostics(calc=calc, sol=sol, min_degree=1)
    #diagnostics.jump_vector_diagnostics(calc=calc, sol=sol, min_degree=1, only_vertices=None)


    # ---- next cell ----

    import os, sys
    from pathlib import Path

    # Add repo root (one level above notebooks/) to sys.path
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    # --- Preamble re-exports
    out_dir = Path(repo_root) / "output" / "PolylineBranchedCrack"
    out_dir.mkdir(parents=True, exist_ok=True)

    # ----------------------------
    # Problem definition
    # ----------------------------
    mm = 1e-3
    vertices = np.array([
        [0,   0.0*mm,  0.0*mm],   
        [1,  5.0*mm,  0.0*mm],   
        [2,  10.0*mm,  5*mm],   
        [3,  20.0*mm,  10*mm],   
        [4,  50.0*mm, 12.0*mm],
        [5,  60.0*mm, -5.0*mm],
        [6,  65.0*mm, -10*mm],
        [7,  80.0*mm, -40.0*mm],
        [8,  120.0*mm, -80.0*mm],
        [9, 60.0*mm, -60.0*mm]
    ], dtype=float)

    connectivity = np.array([
        [0, 1],   # e0
        [1, 2],   # e1
        [2, 3],
        [3, 4],
        [4, 5],
        [5, 6],
        [6, 7],
        [7, 8],
        [7, 9],
    ], dtype=int)

    net = CrackNetworkV4.from_vertices_connectivity(
        vertices=vertices,
        connectivity=connectivity,
        Nv_max=4,
        validate=True,
    )

    material = Material(E=200e9, nu=0.30, plane_stress=False)
    applied  = AppliedStress(sigma_xx=0.0, sigma_yy=100e6, sigma_xy=0.0)

    calc = DCENetworkStaticV4(material, net, applied)


    sol  = calc.solve(
        ne_half=40,
        representation="cheb_quad",
        node_distribution="uniform",
        solver_option="parametrized_crack",
        parametrization="polyline",
        nq_col=3,
        nq_stress=6,
        crack_mode="half",
        enforce_cod_nonnegative=True,
        cod_tol=1e-10,
        cod_max_iter=25,
        end_point_min_nodes=4,
    )

    res = DCEResultsNetworkV4(calc, sol)

    # ----------------------------
    # Plot: network graph + stress + displacements
    # ----------------------------
    plotter = DCEPlotterV4(res, out_dir=out_dir)

    _call(plotter, "plot_network_graph")

    opts = StressPlotOptsV4(
        extent_factor=3,
        n_grid=301,
        add_remote=True,
        mask_cracks=False,
        cmap="jet",
        n_bands=40,
        label_contours=False,
        vmax_factor=2,
    )
    _call(plotter, "plot_stress_components_global", opts=opts, components=("sxx","syy","sxy"))

    _call(plotter, "plot_displacement_vector_field", disp_scale=5e4, mask_cracks=False)

    # ----------------------------
    # Plot: deformed network (trim junction faces; smooth COD/CSD if desired)
    # ----------------------------
    pl_def = DCEPlotterDeformedV4(res, out_dir=out_dir)
    _call(
        pl_def,
        "plot_deformed_network",
        scale=1e2,
        # trim_core_junction_faces=True,
    )

    # ----------------------------
    # Vectors: B-content and PK forces
    # ----------------------------
    pkplt = DCEPlotterPK(res, out_dir=out_dir)

    _call(
        pkplt,
        "plot_b_content_vectors",
        style=VecPlotStyle(max_len_factor=0.18),
        user_scale=1e-3,
        units="mm",
        show_tips=True,
        show_junctions=True,
        show_sum_at_junction=True,
        annotate=True,
    )

    _call(
        pkplt,
        "plot_pk_force_vectors",
        style=VecPlotStyle(max_len_factor=0.18),
        user_scale=1e-3,
        units="mm",
        exclude_self=True,
        show_tips=True,
        show_junctions=True,
        n_samples_tip=32,
        annotate=True,
    )

    plt.show()
    print("Saved figures to:", out_dir)

    # ----------------------------
    # Diagnostics (console)
    # ----------------------------
    #diagnostics.junction_constraint_diagnostics(calc=calc, sol=sol, min_degree=1)
    #diagnostics.jump_vector_diagnostics(calc=calc, sol=sol, min_degree=1, only_vertices=None)

    return locals()


### Case `direct_method_verification` (from `direct_method_verification.ipynb`)


In [ ]:
def run_direct_method_verification():
    """Case extracted from direct_method_verification.ipynb."""
    from pathlib import Path
    import os, sys
    import shutil
    import numpy as np
    import matplotlib.pyplot as plt

    # -----------------------
    # Repo import path
    # -----------------------
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    try:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
    except Exception:
        pass

    from fracture_utils.Ubem.brazilian_disk_bem import BrazilianDiskParams, ensure_bem_field
    from fracture_utils.Ubem.disk_network_propagation import CrackGrowthParams, run_network_growth_uncoupled
    from fracture_utils.Ubem.disk_crack_plotting import PlotParams, make_standard_plot_hook, plot_total_field
    from fracture_utils.Ubem.coupling_blocks_v2 import BEMBlockV2, compose_solver_kwargs_v2
    from fracture_utils.Ubem.disk_iterative_coupling import (
        compute_crack_boundary_tractions_direct,
        solve_bem_with_extra_boundary_tractions,
    )
    from fracture_utils.Ubem.boundary_conditions import build_boundary


    # -----------------------
    # Utility helpers
    # -----------------------
    def _l2norm2(tx, ty):
        tx = np.asarray(tx, float).reshape(-1)
        ty = np.asarray(ty, float).reshape(-1)
        return float(np.sqrt(np.dot(tx, tx) + np.dot(ty, ty)))


    def _cycle_index_from_tag(tag: str):
        parts = str(tag).split("_")
        if len(parts) >= 2 and parts[0] == "cycle":
            try:
                return int(parts[1])
            except Exception:
                return None
        return None


    def _vertex_degree_map(network):
        deg = {int(v.id): 0 for v in network.vertices}
        for e in network.edges:
            deg[int(e.v0)] = deg.get(int(e.v0), 0) + 1
            deg[int(e.v1)] = deg.get(int(e.v1), 0) + 1
        return deg


    def _tip_points(network):
        deg = _vertex_degree_map(network)
        pts = []
        for v in network.vertices:
            if deg.get(int(v.id), 0) == 1:
                pts.append([float(v.x), float(v.y)])
        if not pts:
            return np.zeros((0, 2), float)
        return np.asarray(pts, float)


    def _tip_edge_lengths(network):
        deg = _vertex_degree_map(network)
        edge_by_id = {int(e.id): e for e in network.edges}
        lengths = []

        for v in network.vertices:
            vid = int(v.id)
            if deg.get(vid, 0) != 1 or len(v.edges) == 0:
                continue

            e = edge_by_id.get(int(v.edges[0]))
            if e is None:
                continue

            other = int(e.v1) if int(e.v0) == vid else int(e.v0)
            vo = network.V(other)
            L = float(np.hypot(float(v.x) - float(vo.x), float(v.y) - float(vo.y)))
            if np.isfinite(L) and L > 0:
                lengths.append(L)

        return np.asarray(lengths, float)


    def _min_tip_to_boundary_distance(res, boundary_mesh):
        net = getattr(getattr(res, "calc", None), "network", None)
        if net is None:
            return np.inf

        tips = _tip_points(net)
        if tips.shape[0] == 0:
            return np.inf

        bxy = np.column_stack([np.asarray(boundary_mesh.xm, float), np.asarray(boundary_mesh.ym, float)])
        d = tips[:, None, :] - bxy[None, :, :]
        return float(np.min(np.sqrt(np.sum(d * d, axis=2))))


    def _characteristic_tip_length(res):
        net = getattr(getattr(res, "calc", None), "network", None)
        if net is None:
            return np.inf
        L = _tip_edge_lengths(net)
        if L.size == 0:
            return np.inf
        return float(np.median(L))


    def _disk_external_boundary_tractions(mesh, disk_params):
        # External traction BC on the disk boundary (matching solve_bem_with_extra_boundary_tractions)
        theta_deg = np.asarray(mesh.theta_deg, dtype=float)
        L = np.asarray(mesh.length, dtype=float)

        arc_half = float(getattr(disk_params, "arc_half_angle_deg", 15.0))
        top = (theta_deg >= 90 - arc_half) & (theta_deg <= 90 + arc_half)
        bot = (theta_deg >= -90 - arc_half) & (theta_deg <= -90 + arc_half)

        L_top = float(np.sum(L[top]))
        L_bot = float(np.sum(L[bot]))
        if L_top <= 0 or L_bot <= 0:
            raise RuntimeError("Top/bottom loaded arc has zero length.")

        h = float(getattr(disk_params, "h", 1.0))
        pressure_top = float(disk_params.P_total) / (L_top * h)
        pressure_bot = float(disk_params.P_total) / (L_bot * h)

        tx = np.zeros(mesh.n_seg, float)
        ty = np.zeros(mesh.n_seg, float)

        # pressure_normal => traction = -p * n_out
        tx[top] += -pressure_top * np.asarray(mesh.nx, float)[top]
        ty[top] += -pressure_top * np.asarray(mesh.ny, float)[top]

        tx[bot] += -pressure_bot * np.asarray(mesh.nx, float)[bot]
        ty[bot] += -pressure_bot * np.asarray(mesh.ny, float)[bot]

        return tx, ty


    def _copy_bem_arrays(src_dir: Path, dst_dir: Path):
        dst_dir.mkdir(parents=True, exist_ok=True)
        for fn in ("xs.npy", "ys.npy", "Sxx.npy", "Syy.npy", "Sxy.npy"):
            shutil.copy2(Path(src_dir) / fn, Path(dst_dir) / fn)


    # ---- next cell ----

    # -----------------------
    # Output directory
    # -----------------------
    bem_dir = repo_root / "output" / "BEM_Brazilian_disk_2_direct"
    bem_dir.mkdir(parents=True, exist_ok=True)

    # -----------------------
    # Network (YOU control)
    # -----------------------
    mm = 1e-3
    vertices = np.array(
        [
            [0,  0.0 * mm, -4.0 * mm],
            [1,  0.0 * mm,  4.0 * mm],
        ],
        float,
    )
    connectivity = np.array([[0, 1]], int)

    # -----------------------
    # (1) BEM (fast reuse option)
    # -----------------------
    SKIP_BEM_SOLVE = True # set False to recompute

    disk = BrazilianDiskParams(
        R=25.4e-3 / 2,
        P_total=3.8e3,
        E=231.52e9,
        nu=0.3,
        n_elem=30,
        n_grid=120,
        h=6.35e-3,
    )

    bem_block = BEMBlockV2(disk=disk, out_dir=bem_dir)
    _bext = bem_block.ensure_external_only(
        recompute=(not SKIP_BEM_SOLVE),
        show=True,
        save_contours=True,
    )
    print(f"[bem-baseline] max|sxx| = {np.nanmax(np.abs(_bext.Sxx))*1e-6:.3f} MPa")

    # Build boundary mesh ONCE (used for traction extraction each outer cycle)
    boundary_mesh = build_boundary(
        {"type": "circle", "R": disk.R, "n_boundary": disk.n_elem, "center": (0.0, 0.0)}
    )

    # External traction BC norm (denominator for mismatch indicator)
    tx_ext, ty_ext = _disk_external_boundary_tractions(boundary_mesh, disk)
    ext_traction_norm = _l2norm2(tx_ext, ty_ext)


    # ---- next cell ----

    # -----------------------
    # (2) Three coupling modes (simple names)
    # -----------------------
    # Modes requested:
    #   1) no_coupling : external BEM field drives crack only
    #   2) iterative   : reverse crack boundary traction into BEM, then re-solve crack
    #   3) direct      : augmented KKT monolithic coupling

    RUN_MODES = ("iterative",)
    MAX_CYCLES_PER_CASE = 0   # 0 => initial state only, fastest comparison
    SHOW_TOTAL_PLOTS = True

    # Direct KKT controls
    # Coupled correction form: baseline BEM load drives crack; KKT enforces boundary correction compatibility.
    # Recommended stabilized direct setup: crack block is driven by baseline BEM applied field,
    # while augmented constraints enforce boundary correction compatibility.
    DIRECT_D_MODE = "correction_zero"
    DIRECT_KEEP_BEM_APPLIED = True
    DIRECT_RIDGE_Q = 1e-2
    DIRECT_RIDGE_Y = 1e-6
    DIRECT_C2_PROJECT_TAU = None

    plot_params = PlotParams(
        cmap="jet",
        match_limits_to_crack=False,
        robust=True,
        robust_pct=97.0,
        symmetric=True,
        vmax_factor=2.0,
        n_levels=30,
        n_line_levels=30,
        dpi=300,
        fixed_vlim_mpa=30.0,
        augmented_correction_stride=6,
    )


    # ---- next cell ----

    # -----------------------
    # (3) Run the three cases + compare sigma_xx along y=0
    # -----------------------
    cases_root = bem_dir / "three_mode_compare"
    cases_root.mkdir(parents=True, exist_ok=True)

    # Keep a frozen baseline external-only BEM field
    baseline_bem = cases_root / "baseline_external"
    _copy_bem_arrays(bem_dir, baseline_bem)


    def _run_case(case_name: str, case_bem_dir: Path, solver_kwargs: dict, run_tag: str = "run"):
        case_root = cases_root / case_name
        run_dir = case_root / run_tag
        run_dir.mkdir(parents=True, exist_ok=True)

        grow = CrackGrowthParams(
            max_cycles=int(MAX_CYCLES_PER_CASE),
            L_limit_mm=20.0,
            vertex_high=18,
            solver_kwargs=dict(solver_kwargs),
        )

        res = run_network_growth_uncoupled(
            bem_dir=case_bem_dir,
            out_dir=run_dir,
            vertices=vertices,
            connectivity=connectivity,
            params=grow,
            plot_hook=None,
        )
        return res


    case_data = {}

    # -----------------------
    # Case 1: no_coupling
    # -----------------------
    if "no_coupling" in RUN_MODES:
        case_name = "no_coupling"
        case_root = cases_root / case_name
        case_bem = case_root / "bem"
        _copy_bem_arrays(baseline_bem, case_bem)

        solver_kwargs = compose_solver_kwargs_v2("one_way", base_solver_kwargs=None)
        res = _run_case(case_name, case_bem, solver_kwargs, run_tag="run")

        Sxx_tot, Syy_tot, Sxy_tot = plot_total_field(
            case_bem,
            res,
            out_dir=case_root / "plots",
            tag="initial",
            params=plot_params,
            show=SHOW_TOTAL_PLOTS,
        )
        case_data[case_name] = {
            "res": res,
            "bem_dir": case_bem,
            "xs": np.load(case_bem / "xs.npy"),
            "ys": np.load(case_bem / "ys.npy"),
            "Sxx_tot": Sxx_tot,
            "Syy_tot": Syy_tot,
            "Sxy_tot": Sxy_tot,
        }

    # -----------------------
    # Case 2: iterative
    # -----------------------
    if "iterative" in RUN_MODES:
        case_name = "iterative"
        case_root = cases_root / case_name
        case_bem = case_root / "bem"
        _copy_bem_arrays(baseline_bem, case_bem)

        solver_kwargs = compose_solver_kwargs_v2("one_way", base_solver_kwargs=None)

        # Predictor crack solve under external-only BEM
        res_pred = _run_case(case_name, case_bem, solver_kwargs, run_tag="predictor")

        # Crack-induced tractions on boundary, then reverse into BEM
        tx_cr, ty_cr = compute_crack_boundary_tractions_direct(
            res=res_pred,
            boundary_mesh=boundary_mesh,
        )
        print(f"[iterative] ||t_cr||/||t_ext|| = {_l2norm2(tx_cr, ty_cr)/max(ext_traction_norm,1e-30):.3e}")

        solve_bem_with_extra_boundary_tractions(
            disk_params=disk,
            bem_dir=case_bem,
            tx_extra=-tx_cr,
            ty_extra=-ty_cr,
            show=False,
        )

        # Corrected crack solve under updated BEM field
        res = _run_case(case_name, case_bem, solver_kwargs, run_tag="corrected")

        Sxx_tot, Syy_tot, Sxy_tot = plot_total_field(
            case_bem,
            res,
            out_dir=case_root / "plots",
            tag="initial",
            params=plot_params,
            show=SHOW_TOTAL_PLOTS,
        )
        case_data[case_name] = {
            "res": res,
            "bem_dir": case_bem,
            "xs": np.load(case_bem / "xs.npy"),
            "ys": np.load(case_bem / "ys.npy"),
            "Sxx_tot": Sxx_tot,
            "Syy_tot": Syy_tot,
            "Sxy_tot": Sxy_tot,
        }

    # -----------------------
    # Case 3: direct (augmented KKT)
    # -----------------------
    if "direct" in RUN_MODES:
        case_name = "direct"
        case_root = cases_root / case_name
        case_bem = case_root / "bem"
        _copy_bem_arrays(baseline_bem, case_bem)

        aug_payload = bem_block.build_augmented_payload(
            d_mode=DIRECT_D_MODE,
            gauss_n=4,
        )
        solver_kwargs = compose_solver_kwargs_v2(
            "full_kkt",
            base_solver_kwargs={"augmented_equilibrate": True, "augmented_equilibrate_eps": 1e-14, "augmented_equilibrate_cols": False, "augmented_physical_scaling": True, "augmented_enforce_crack_equilibrium": True},
            augmented_payload=aug_payload,
            augmented_ridge_q=float(DIRECT_RIDGE_Q),
            augmented_ridge_y=float(DIRECT_RIDGE_Y),
            augmented_keep_bem_applied=bool(DIRECT_KEEP_BEM_APPLIED),
        )

        res = _run_case(case_name, case_bem, solver_kwargs, run_tag="run")

        Sxx_tot, Syy_tot, Sxy_tot = plot_total_field(
            case_bem,
            res,
            out_dir=case_root / "plots",
            tag="initial",
            params=plot_params,
            show=SHOW_TOTAL_PLOTS,
        )
        case_data[case_name] = {
            "res": res,
            "bem_dir": case_bem,
            "xs": np.load(case_bem / "xs.npy"),
            "ys": np.load(case_bem / "ys.npy"),
            "Sxx_tot": Sxx_tot,
            "Syy_tot": Syy_tot,
            "Sxy_tot": Sxy_tot,
        }


    # -----------------------
    # y=0 line comparison: sigma_xx for all three modes
    # -----------------------
    fig, ax = plt.subplots(figsize=(8.0, 5.0))
    colors = {
        "no_coupling": "tab:blue",
        "iterative": "tab:orange",
        "direct": "tab:green",
    }

    for mode in RUN_MODES:
        if mode not in case_data:
            continue
        xs = np.asarray(case_data[mode]["xs"], float)
        ys = np.asarray(case_data[mode]["ys"], float)
        Sxx_tot = np.asarray(case_data[mode]["Sxx_tot"], float)

        iy0 = int(np.argmin(np.abs(ys - 0.0)))
        y_sel = float(ys[iy0])
        sline_mpa = Sxx_tot[iy0, :] * 1e-6

        ax.plot(xs * 1e3, sline_mpa, lw=2.0, color=colors.get(mode, None), label=f"{mode} (y={y_sel*1e3:.2f} mm)")

    ax.set_xlabel("x [mm]")
    ax.set_ylabel(r"$\sigma_{xx}$ [MPa]")
    ax.set_title(r"$\sigma_{xx}$ along y=0 for three coupling modes")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

    line_png = cases_root / "sigma_xx_line_y0_three_modes.png"
    fig.savefig(line_png, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"[done] saved line comparison: {line_png}")
    print(f"[done] case outputs root: {cases_root}")


    # -----------------------
    # Numeric table for y=0 sigma_xx comparison
    # -----------------------
    summary_rows = []
    for mode in RUN_MODES:
        if mode not in case_data:
            continue
        xs = np.asarray(case_data[mode]["xs"], float)
        ys = np.asarray(case_data[mode]["ys"], float)
        Sxx_tot = np.asarray(case_data[mode]["Sxx_tot"], float)

        iy0 = int(np.argmin(np.abs(ys - 0.0)))
        y_sel_mm = float(ys[iy0] * 1e3)
        sline_mpa = np.asarray(Sxx_tot[iy0, :] * 1e-6, float)

        smin = float(np.nanmin(sline_mpa))
        smax = float(np.nanmax(sline_mpa))
        spp = float(smax - smin)
        summary_rows.append((mode, y_sel_mm, smin, smax, spp))

    if summary_rows:
        print()
        print("Sigma_xx along y=0 summary [MPa]")
        print("mode         y_used[mm]    min       max     peak_to_peak")
        for mode, ymm, smin, smax, spp in summary_rows:
            print(f"{mode:<12} {ymm:>10.3f} {smin:>9.3f} {smax:>9.3f} {spp:>13.3f}")

        csv_path = cases_root / "sigma_xx_line_y0_summary.csv"
        with open(csv_path, "w", encoding="utf-8") as f:
            f.write("mode,y_used_mm,sxx_min_mpa,sxx_max_mpa,sxx_peak_to_peak_mpa\n")
            for mode, ymm, smin, smax, spp in summary_rows:
                f.write(f"{mode},{ymm:.6f},{smin:.6f},{smax:.6f},{spp:.6f}\n")
        print(f"[done] saved numeric summary: {csv_path}")


    # ---- next cell ----

    # -----------------------
    # (4) Direct-mode scaling diagnostics (no patching)
    # -----------------------
    # Run this AFTER cell (3).
    # It reports constraint/traction magnitudes to localize unit-scaling issues.

    if "direct" not in case_data:
        raise RuntimeError("direct case is not available. Ensure RUN_MODES includes 'direct' and rerun cell (3).")

    res_direct = case_data["direct"]["res"]
    sol_d = getattr(res_direct, "sol", {})
    cpl = sol_d.get("coupling", {}) if isinstance(sol_d, dict) else {}
    if (not isinstance(cpl, dict)) or (not bool(cpl.get("augmented", False))):
        raise RuntimeError("No augmented coupling metadata found in direct result.")

    y = np.asarray(cpl.get("y_bc", []), float).reshape(-1)
    nbd = int(cpl.get("n_boundary", 0))
    if nbd <= 0 or y.size != 4 * nbd:
        raise RuntimeError(f"Unexpected y_bc shape: len(y)={y.size}, n_boundary={nbd}")

    # Rebuild augmented payload with same direct settings used in cell (3)
    aug_diag = bem_block.build_augmented_payload(
        d_mode=DIRECT_D_MODE,
        gauss_n=4,
    )
    C_bem = np.asarray(aug_diag["C_bem"], float)
    C_bc = np.asarray(aug_diag["C_bc"], float)
    d = np.asarray(aug_diag["d"], float).reshape(-1)

    # y traction block from augmented unknowns
    # y = [u_bc(2N), t_bc(2N)]
    t_y = y[2 * nbd :]

    # Crack-induced boundary tractions from direct crack solution
    tx_cr_d, ty_cr_d = compute_crack_boundary_tractions_direct(
        res=res_direct,
        boundary_mesh=boundary_mesh,
    )
    t_cr_d = np.empty((2 * nbd,), float)
    t_cr_d[0::2] = np.asarray(tx_cr_d, float)
    t_cr_d[1::2] = np.asarray(ty_cr_d, float)

    # Residuals of augmented constraints
    r_bem = C_bem @ y
    r_eq = (C_bc @ y) + t_cr_d - d   # should be ~0 for exact traction compatibility

    # External traction norm reference
    tx_ext_ref, ty_ext_ref = _disk_external_boundary_tractions(boundary_mesh, disk)
    t_ext = np.empty((2 * nbd,), float)
    t_ext[0::2] = tx_ext_ref
    t_ext[1::2] = ty_ext_ref


    def _nrm(v):
        v = np.asarray(v, float).reshape(-1)
        return float(np.linalg.norm(v))

    print("[direct diag] norms (Pa on traction vectors):")
    print(f"  ||y||                 = {_nrm(y):.6e}")
    print(f"  ||t_y||               = {_nrm(t_y):.6e}")
    print(f"  ||t_cr(direct)||      = {_nrm(t_cr_d):.6e}")
    print(f"  ||d||                 = {_nrm(d):.6e}")
    print(f"  ||t_ext||             = {_nrm(t_ext):.6e}")
    print(f"  ||C_bem y||           = {_nrm(r_bem):.6e}")
    print(f"  ||C_bc y + t_cr - d|| = {_nrm(r_eq):.6e}")

    if _nrm(d) > 0:
        print(f"  rel ||eq_res||/||d||  = {_nrm(r_eq)/_nrm(d):.6e}")
    if _nrm(t_ext) > 0:
        print(f"  ||t_cr(direct)||/||t_ext|| = {_nrm(t_cr_d)/_nrm(t_ext):.6e}")

    # Compare direct crack traction norm to no_coupling / iterative crack traction norms
    for mode_ref in ("no_coupling", "iterative"):
        if mode_ref not in case_data:
            continue
        res_ref = case_data[mode_ref]["res"]
        tx_r, ty_r = compute_crack_boundary_tractions_direct(res=res_ref, boundary_mesh=boundary_mesh)
        t_r = np.empty((2 * nbd,), float)
        t_r[0::2] = tx_r
        t_r[1::2] = ty_r
        nr = _nrm(t_r)
        if nr > 0:
            print(f"  ||t_cr(direct)||/||t_cr({mode_ref})|| = {_nrm(t_cr_d)/nr:.6e}")

    # y=0 sigma_xx factor checks vs expected signatures 1/h and 10/h
    if "no_coupling" in case_data:
        ys_d = np.asarray(case_data["direct"]["ys"], float)
        iy0_d = int(np.argmin(np.abs(ys_d - 0.0)))
        sxx_d = np.asarray(case_data["direct"]["Sxx_tot"], float)[iy0_d, :] * 1e-6

        ys_n = np.asarray(case_data["no_coupling"]["ys"], float)
        iy0_n = int(np.argmin(np.abs(ys_n - 0.0)))
        sxx_n = np.asarray(case_data["no_coupling"]["Sxx_tot"], float)[iy0_n, :] * 1e-6

        amp_d = float(np.nanmax(sxx_d) - np.nanmin(sxx_d))
        amp_n = float(np.nanmax(sxx_n) - np.nanmin(sxx_n))
        if amp_n > 0:
            fac = amp_d / amp_n
            h = float(disk.h)
            sig_1h = 1.0 / h
            sig_10h = 10.0 / h
            print("[line y=0 factor]")
            print(f"  factor direct/no_coupling = {fac:.6e}")
            print(f"  1/h                       = {sig_1h:.6e}")
            print(f"  10/h                      = {sig_10h:.6e}")
            print(f"  factor / (1/h)            = {fac/sig_1h:.6e}")
            print(f"  factor / (10/h)           = {fac/sig_10h:.6e}")


    # ---- next cell ----

    # -----------------------
    # (5) Matrix-level diagnostics for direct mode (no solver changes)
    # -----------------------
    # Run AFTER cell (4). This rebuilds the direct linearized blocks and reports
    # rank/conditioning-style metrics to identify block scaling issues.

    import numpy as _np

    from fracture_utils.Usolver.build_geometry import vertex_degrees
    from fracture_utils.Usolver.build_dipolar_polyline import build_polylines_full
    from fracture_utils.Usolver.build_polar_polyline import build_polylines_half_branches
    from fracture_utils.Usolver.parametrization_preprocess import normalize_representation, convert_polylines_to_csplines
    from fracture_utils.Usolver.build_discretize import (
        discretize_polylines,
        allocate_unknowns,
        assemble_operator,
        assemble_boundary_traction_operator,
        assemble_bem_boundary_to_crack_traction_operator,
    )
    from fracture_utils.Usolver.constraints import build_constraints_full, build_constraints_half

    if "direct" not in case_data:
        raise RuntimeError("direct case is not available. Run cell (3) first.")

    res_direct = case_data["direct"]["res"]
    calc_d = res_direct.calc
    network = calc_d.network
    material = calc_d.material
    applied = calc_d.applied

    # Match run_network_growth_uncoupled defaults + direct-mode overrides
    solver_kwargs_dbg = dict(
        ne_half=100,
        representation="cheb_quad",
        node_distribution="tip_dense",
        solver_option="parametrized_crack",
        parametrization="polyline",
        nq_col=12,
        nq_stress=16,
        crack_mode="half",
        junction_model="core",
    )

    # Normalize representation settings
    rep_in, dist_in, collocation_mode = normalize_representation(
        solver_kwargs_dbg["representation"],
        solver_kwargs_dbg["node_distribution"],
    )
    crack_mode = str(solver_kwargs_dbg["crack_mode"]).lower().strip()
    junction_model = str(solver_kwargs_dbg.get("junction_model", "strict")).lower().strip()

    # Build polyline structures
    deg = vertex_degrees(network)
    if crack_mode == "full":
        polylines, edge_to_polyline = build_polylines_full(network)
    else:
        polylines, edge_to_polyline = build_polylines_half_branches(network, deg)

    polylines = convert_polylines_to_csplines(
        network,
        polylines,
        crack_mode=crack_mode,
        parametrization=str(solver_kwargs_dbg["parametrization"]),
        branch_param=None,
    )

    poly_panels = discretize_polylines(
        network,
        polylines,
        ne_half=int(solver_kwargs_dbg["ne_half"]),
        node_distribution=dist_in,
        collocation_mode=collocation_mode,
        representation=rep_in,
        nq_stress=int(solver_kwargs_dbg["nq_stress"]),
        tip_min_nodes=6,
        endpoint_min_nodes=6,
        kink_side_nodes=2,
        refine_junction_endpoints=True,
        refine_kinks=True,
        tip_cluster="power",
        tip_cluster_power=2.0,
        other_min_panels=1,
    )

    offsets, junction_dof, branch_end_dof, nunk = allocate_unknowns(
        poly_panels,
        deg,
        crack_mode=crack_mode,
        junction_model=junction_model,
    )

    K, rhs = assemble_operator(
        material=material,
        applied=applied,
        poly_panels=poly_panels,
        offsets=offsets,
        nunk=int(nunk),
        nq_col=int(solver_kwargs_dbg["nq_col"]),
    )

    if crack_mode == "full":
        C = build_constraints_full(
            poly_panels=poly_panels,
            offsets=offsets,
            nunk=nunk,
            use_tip_singular=(rep_in == "singular"),
        )
    else:
        C, _P = build_constraints_half(
            network=network,
            deg=deg,
            poly_panels=poly_panels,
            offsets=offsets,
            junction_dof=junction_dof,
            branch_end_dof=branch_end_dof,
            nunk=nunk,
            junction_model=junction_model,
            theta_min_flat=5.0 * _np.pi / 180.0,
            soft_eta=1.0,
        )

    # Augmented direct payload and coupling block
    aug_dbg = bem_block.build_augmented_payload(
        d_mode=DIRECT_D_MODE,
        gauss_n=4,
    )
    Xb = _np.asarray(aug_dbg["boundary_xy"], float)
    Nb = _np.asarray(aug_dbg["boundary_n"], float)
    C_bem = _np.asarray(aug_dbg["C_bem"], float)
    C_bc = _np.asarray(aug_dbg["C_bc"], float)
    d_bc = _np.asarray(aug_dbg["d"], float).reshape(-1)

    M = assemble_boundary_traction_operator(
        material=material,
        poly_panels=poly_panels,
        offsets=offsets,
        nunk=int(nunk),
        boundary_xy=Xb,
        boundary_n=Nb,
    )

    Nbc = assemble_bem_boundary_to_crack_traction_operator(
        material=material,
        poly_panels=poly_panels,
        boundary_x1=_np.asarray(aug_dbg["boundary_x1"], float),
        boundary_y1=_np.asarray(aug_dbg["boundary_y1"], float),
        boundary_x2=_np.asarray(aug_dbg["boundary_x2"], float),
        boundary_y2=_np.asarray(aug_dbg["boundary_y2"], float),
        gauss_n=4,
    )

    ny = int(C_bem.shape[1])
    n_total = int(nunk) + ny

    Kz = _np.hstack([K, Nbc])
    C0 = _np.hstack([C, _np.zeros((C.shape[0], ny), float)]) if C.size else _np.zeros((0, n_total), float)
    C1 = _np.hstack([_np.zeros((C_bem.shape[0], int(nunk)), float), C_bem])
    C2 = _np.hstack([M, C_bc])
    c2_rank_proj = -1
    if "DIRECT_C2_PROJECT_TAU" in globals() and DIRECT_C2_PROJECT_TAU is not None:
        tau = float(DIRECT_C2_PROJECT_TAU)
        if _np.isfinite(tau) and tau > 0.0:
            U_m, s_m, _Vt_m = _np.linalg.svd(M, full_matrices=False)
            if s_m.size > 0 and _np.isfinite(s_m[0]) and s_m[0] > 0.0:
                c2_rank_proj = int(_np.sum(s_m > tau * float(s_m[0])))
                if c2_rank_proj > 0:
                    Pm = U_m[:, :c2_rank_proj].T
                    C2 = Pm @ C2
    print(f"[matrix diag] C2 projected rank used = {c2_rank_proj}")
    C_all = _np.vstack([C0, C1, C2])


    def _rank_and_cond(A, name, tol_rel=1e-12):
        A = _np.asarray(A, float)
        if A.size == 0:
            print(f"[{name}] empty")
            return
        s = _np.linalg.svd(A, compute_uv=False)
        smax = float(s[0]) if s.size else 0.0
        if s.size == 0 or (not _np.isfinite(smax)) or smax <= 0.0:
            print(f"[{name}] invalid singular spectrum")
            return
        tol = float(tol_rel) * smax
        rank = int(_np.sum(s > tol))
        spos = s[s > tol]
        smin_pos = float(spos[-1]) if spos.size else _np.nan
        cond_eff = float(smax / smin_pos) if (spos.size and smin_pos > 0) else _np.inf
        print(f"[{name}] shape={A.shape}, rank={rank}, smax={smax:.3e}, smin_pos={smin_pos:.3e}, cond_eff={cond_eff:.3e}")


    def _nrm(A):
        return float(_np.linalg.norm(_np.asarray(A, float)))

    print("[matrix diag] dimensions")
    print(f"  nunk (crack unknowns)    = {int(nunk)}")
    print(f"  ny   (BEM unknowns)      = {int(ny)}")
    print(f"  n_total                  = {int(n_total)}")
    print(f"  K shape                  = {K.shape}")
    print(f"  M shape                  = {M.shape}")
    print(f"  C shape                  = {C.shape}")
    print(f"  C_bem shape              = {C_bem.shape}")
    print(f"  C_bc shape               = {C_bc.shape}")
    print(f"  C_all shape              = {C_all.shape}")

    print("\n[matrix diag] operator norms")
    print(f"  ||K||_F      = {_nrm(K):.6e}")
    print(f"  ||M||_F      = {_nrm(M):.6e}")
    print(f"  ||Nbc||_F    = {_nrm(Nbc):.6e}")
    print(f"  ||C_bem||_F  = {_nrm(C_bem):.6e}")
    print(f"  ||C_bc||_F   = {_nrm(C_bc):.6e}")
    print(f"  ||d||_2      = {_nrm(d_bc):.6e}")

    print("\n[matrix diag] rank / condition indicators")
    _rank_and_cond(K, "K")
    _rank_and_cond(M, "M")
    _rank_and_cond(C, "C")
    _rank_and_cond(C_bem, "C_bem")
    _rank_and_cond(C_all, "C_all")
    _rank_and_cond(_np.vstack([Kz, C_all]), "[Kz; C_all]")

    # Optional: compare block scales against K (helps detect under/over-weighted constraints)
    nK = _nrm(K)
    print("\n[matrix diag] block scale ratios (vs ||K||_F)")
    if nK > 0:
        print(f"  ||M||/||K||      = {_nrm(M)/nK:.6e}")
        print(f"  ||C_bem||/||K||  = {_nrm(C_bem)/nK:.6e}")
        print(f"  ||C_bc||/||K||   = {_nrm(C_bc)/nK:.6e}")
    else:
        print("  ||K|| is zero or invalid")


    # ---- next cell ----

    # -----------------------
    # (6) Observability diagnostics: q, K residual, and M-spectrum energy
    # -----------------------
    # Run AFTER cell (5).

    import numpy as _np

    if "direct" not in case_data:
        raise RuntimeError("Run cell (3) first to populate case_data.")

    # Reuse matrices assembled in cell (5): K, rhs, M, nunk, offsets, poly_panels
    needed = ["K", "rhs", "M", "nunk", "offsets", "poly_panels"]
    for _nm in needed:
        if _nm not in globals():
            raise RuntimeError(f"Missing '{_nm}' from cell (5). Run cell (5) first.")


    def _extract_q_from_res(res, nunk_local, offsets_local, poly_panels_local):
        """
        Reconstruct crack DOF vector q from polyline_solutions.
        For half/core extra jump DOFs are not explicitly needed for M (zero columns),
        so they are left as zero.
        """
        q = _np.zeros((int(nunk_local),), float)
        sol = getattr(res, "sol", {})
        plist = sol.get("polyline_solutions", []) if isinstance(sol, dict) else []

        for pid, pp in enumerate(poly_panels_local):
            if pid >= len(plist):
                continue
            psol = plist[pid]
            off = int(offsets_local[pid])
            Np = int(pp["Np"])

            bIh = _np.asarray(psol.get("bI_hat", []), float).reshape(-1)
            bIIh = _np.asarray(psol.get("bII_hat", []), float).reshape(-1)
            if bIh.size != Np or bIIh.size != Np:
                continue

            for k in range(Np):
                q[off + 2 * k + 0] = float(bIh[k])
                q[off + 2 * k + 1] = float(bIIh[k])

        return q


    def _nrm(v):
        return float(_np.linalg.norm(_np.asarray(v, float).reshape(-1)))


    # SVD of M for weak/strong mode decomposition
    s = _np.linalg.svd(M, compute_uv=False)
    if s.size == 0:
        raise RuntimeError("Empty singular spectrum for M.")

    smax = float(s[0])
    if (not _np.isfinite(smax)) or smax <= 0.0:
        raise RuntimeError("Invalid M singular spectrum.")

    # Weak modes: sigma/smax <= tau
    for tau in (1e-6, 1e-8, 1e-10):
        _rank = int(_np.sum(s > tau * smax))
        print(f"[M spectrum] tau={tau:.0e}: effective rank={_rank}/{M.shape[1]}")

    U, S, Vt = _np.linalg.svd(M, full_matrices=False)
    V = Vt.T


    def _energy_split_in_M_basis(q, tau=1e-8):
        alpha = V.T @ q
        rel = S / max(S[0], 1e-300)
        weak = rel <= float(tau)
        strong = ~weak
        e_tot = float(_np.dot(alpha, alpha))
        e_w = float(_np.dot(alpha[weak], alpha[weak])) if _np.any(weak) else 0.0
        e_s = float(_np.dot(alpha[strong], alpha[strong])) if _np.any(strong) else 0.0
        fw = e_w / e_tot if e_tot > 0 else _np.nan
        fs = e_s / e_tot if e_tot > 0 else _np.nan
        return fw, fs


    print("\n[observability diag]")
    print("mode         ||q||         ||Kq-rhs||      ||Mq||        ||Mq||/||rhs||   E_weak(tau=1e-8)")

    rhs_n = _nrm(rhs)
    store = {}
    for mode in ("no_coupling", "iterative", "direct"):
        if mode not in case_data:
            continue
        q = _extract_q_from_res(case_data[mode]["res"], nunk, offsets, poly_panels)

        rK = K @ q - rhs
        # If monolithic Nbc is available and direct coupling stores y_bc,
        # use crack-equilibrium residual Kq + Nbc y - rhs for direct mode.
        if ('Nbc' in globals()) and mode == 'direct':
            cpl = getattr(case_data[mode]['res'], 'sol', {}).get('coupling', {})
            ybc = _np.asarray(cpl.get('y_bc', []), float).reshape(-1) if isinstance(cpl, dict) else _np.zeros((0,), float)
            if ybc.size == Nbc.shape[1]:
                rK = K @ q + Nbc @ ybc - rhs
        Mq = M @ q

        n_q = _nrm(q)
        n_rK = _nrm(rK)
        n_Mq = _nrm(Mq)
        ratio = n_Mq / rhs_n if rhs_n > 0 else _np.nan
        fw, fs = _energy_split_in_M_basis(q, tau=1e-8)

        store[mode] = dict(q=q, n_q=n_q, n_rK=n_rK, n_Mq=n_Mq, ratio=ratio, fw=fw)
        print(f"{mode:<12} {n_q:>11.3e}  {n_rK:>11.3e}  {n_Mq:>11.3e}  {ratio:>14.3e}  {fw:>14.3e}")

    print("\n[relative ratios vs no_coupling]")
    if "no_coupling" in store:
        b = store["no_coupling"]
        for mode in ("iterative", "direct"):
            if mode not in store:
                continue
            r_q = store[mode]["n_q"] / max(b["n_q"], 1e-300)
            r_m = store[mode]["n_Mq"] / max(b["n_Mq"], 1e-300)
            r_rk = store[mode]["n_rK"] / max(b["n_rK"], 1e-300)
            print(f"{mode:<12} ||q|| ratio={r_q:.3e}, ||Mq|| ratio={r_m:.3e}, ||Kq-rhs|| ratio={r_rk:.3e}")


    # ---- next cell ----

    # -----------------------
    # (8) Finite-value diagnostics for K/rhs/M assembly
    # -----------------------
    # Run AFTER cell (5). This does not modify solver logic.

    import numpy as _np

    needed = ["K", "rhs", "M", "poly_panels"]
    for _nm in needed:
        if _nm not in globals():
            raise RuntimeError(f"Missing '{_nm}'. Run cell (5) first.")


    def _finite_report(name, A):
        A = _np.asarray(A)
        total = A.size
        n_nan = int(_np.isnan(A).sum())
        n_inf = int(_np.isinf(A).sum())
        n_bad = n_nan + n_inf
        print(f"[{name}] shape={A.shape}, total={total}, bad={n_bad}, nan={n_nan}, inf={n_inf}")
        return n_bad

    print("[finite diag] overall")
    bad_rhs = _finite_report("rhs", rhs)
    bad_K = _finite_report("K", K)
    bad_M = _finite_report("M", M)

    # K/rhs row-level diagnostics (2*ncoll rows)
    row_bad = _np.any(~_np.isfinite(K), axis=1) | (~_np.isfinite(rhs))
    idx_bad = _np.where(row_bad)[0]
    print(f"[finite diag] bad K/rhs rows: {idx_bad.size}")

    if idx_bad.size > 0:
        # Build row -> (polyline id, local collocation id, component) mapping
        ncol_per_poly = [int(len(pp["x_col"])) for pp in poly_panels]
        ncol_tot = int(sum(ncol_per_poly))
        cum = _np.cumsum([0] + ncol_per_poly)

        def _row_info(r):
            comp = "tn" if r < ncol_tot else "ts"
            j = int(r if r < ncol_tot else r - ncol_tot)
            pid = int(_np.searchsorted(cum, j, side="right") - 1)
            ic = int(j - cum[pid]) if (0 <= pid < len(ncol_per_poly)) else -1
            return pid, ic, comp

        print("[finite diag] first bad rows (up to 20):")
        for r in idx_bad[:20]:
            pid, ic, comp = _row_info(int(r))
            rv = rhs[int(r)]
            print(f"  row={int(r):4d} comp={comp} pid={pid} ic={ic} rhs={rv}")

        # Count bad rows per polyline/component
        counts = {}
        for r in idx_bad:
            pid, ic, comp = _row_info(int(r))
            key = (pid, comp)
            counts[key] = counts.get(key, 0) + 1
        print("[finite diag] bad-row counts by (pid, comp):")
        for (pid, comp), c in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0][0], kv[0][1])):
            print(f"  pid={pid:3d}, comp={comp}: {c}")

    # M row/col diagnostics
    M_bad_rows = _np.where(_np.any(~_np.isfinite(M), axis=1))[0]
    M_bad_cols = _np.where(_np.any(~_np.isfinite(M), axis=0))[0]
    print(f"[finite diag] M bad rows: {M_bad_rows.size}, M bad cols: {M_bad_cols.size}")
    if M_bad_rows.size:
        print("  first bad M rows:", M_bad_rows[:20])
    if M_bad_cols.size:
        print("  first bad M cols:", M_bad_cols[:20])

    # If finite, print robust rhs norms
    if bad_rhs == 0:
        print(f"[finite diag] ||rhs||_2 = {float(_np.linalg.norm(rhs)):.6e}")
        print(f"[finite diag] max|rhs| = {float(_np.nanmax(_np.abs(rhs))):.6e}")
    else:
        rhs_f = _np.asarray(rhs, float)
        rhs_f = rhs_f[_np.isfinite(rhs_f)]
        if rhs_f.size:
            print(f"[finite diag] finite-part ||rhs||_2 = {float(_np.linalg.norm(rhs_f)):.6e}")
            print(f"[finite diag] finite-part max|rhs| = {float(_np.max(_np.abs(rhs_f))):.6e}")


    # ---- next cell ----

    # -----------------------
    # (9) BEM-only equivalence check: KKT block vs standalone BEM (q=0)
    # -----------------------
    # Purpose: verify KKT BEM block reproduces standalone BEM external-load field.

    import numpy as _np
    import matplotlib.pyplot as _plt

    from fracture_utils.Usolver.KKT import solve_kkt_lsq_eq
    from fracture_utils.Ubem.bem_solver import BEMSolver2D
    from fracture_utils.Ubem.bem_stress_field import stress_on_grid

    # Build augmented boundary data with full external traction target
    aug_bem = bem_block.build_augmented_payload(d_mode="external_total", gauss_n=4)
    C_bem = _np.asarray(aug_bem["C_bem"], float)
    C_bc = _np.asarray(aug_bem["C_bc"], float)
    d = _np.asarray(aug_bem["d"], float).reshape(-1)

    x1 = _np.asarray(aug_bem["boundary_x1"], float).reshape(-1)
    y1 = _np.asarray(aug_bem["boundary_y1"], float).reshape(-1)
    x2 = _np.asarray(aug_bem["boundary_x2"], float).reshape(-1)
    y2 = _np.asarray(aug_bem["boundary_y2"], float).reshape(-1)
    nbd = int(x1.size)
    ny = int(C_bem.shape[1])

    # BEM-only constrained solve (q disabled):
    #   C_bem y = 0
    #   C_bc  y = d
    K0 = _np.zeros((0, ny), float)
    rhs0 = _np.zeros((0,), float)
    Ceq = _np.vstack([C_bem, C_bc])
    deq = _np.concatenate([_np.zeros((C_bem.shape[0],), float), d])

    # Tiny ridge on y for numerical stability (min-norm selection)
    rd = _np.full((ny,), 1e-14, float)
    y = solve_kkt_lsq_eq(K=K0, rhs=rhs0, C=Ceq, d=deq, ridge=0.0, ridge_diag=rd)

    u = y[:2*nbd]
    t = y[2*nbd:]
    ux = u[0::2]
    uy = u[1::2]
    tx = t[0::2]
    ty = t[1::2]

    # Reconstruct stress field from solved boundary unknowns
    solver = BEMSolver2D(E=float(disk.E), nu=float(disk.nu), h=1.0, plane_strain=True)
    for i in range(nbd):
        solver.add_element(float(x1[i]), float(y1[i]), float(x2[i]), float(y2[i]), True, 0.0, 0.0)
    solver.u_x = _np.asarray(ux, float).copy()
    solver.u_y = _np.asarray(uy, float).copy()
    solver.t_x = _np.asarray(tx, float).copy()
    solver.t_y = _np.asarray(ty, float).copy()

    xs = _np.load(bem_dir / "xs.npy")
    ys = _np.load(bem_dir / "ys.npy")
    Sxx_ref = _np.load(bem_dir / "Sxx.npy")
    Syy_ref = _np.load(bem_dir / "Syy.npy")
    Sxy_ref = _np.load(bem_dir / "Sxy.npy")

    Sxx_kkt, Syy_kkt, Sxy_kkt = stress_on_grid(solver, xs, ys, inside=None, fill=_np.nan)

    mask = _np.isfinite(Sxx_ref) & _np.isfinite(Syy_ref) & _np.isfinite(Sxy_ref)
    Sxx_kkt_m = _np.where(mask, Sxx_kkt, _np.nan)
    Syy_kkt_m = _np.where(mask, Syy_kkt, _np.nan)
    Sxy_kkt_m = _np.where(mask, Sxy_kkt, _np.nan)


    def _rel_l2(A, B, m):
        a = _np.asarray(A, float)[m]
        b = _np.asarray(B, float)[m]
        n = float(_np.linalg.norm(a - b))
        dnm = float(_np.linalg.norm(b))
        return (n / dnm) if dnm > 0 else _np.nan

    print("[bem-only equivalence: KKT vs standalone]")
    print(f"  n_boundary          = {nbd}")
    print(f"  ||C_bem y||         = {float(_np.linalg.norm(C_bem @ y)):.6e}")
    print(f"  ||C_bc y - d||      = {float(_np.linalg.norm(C_bc @ y - d)):.6e}")
    print(f"  rel L2 Sxx          = {_rel_l2(Sxx_kkt_m, Sxx_ref, mask):.6e}")
    print(f"  rel L2 Syy          = {_rel_l2(Syy_kkt_m, Syy_ref, mask):.6e}")
    print(f"  rel L2 Sxy          = {_rel_l2(Sxy_kkt_m, Sxy_ref, mask):.6e}")
    print(f"  max|Sxx_kkt(masked)| [MPa] = {float(_np.nanmax(_np.abs(Sxx_kkt_m))*1e-6):.6f}")
    print(f"  max|Sxx_ref|         [MPa] = {float(_np.nanmax(_np.abs(Sxx_ref))*1e-6):.6f}")

    # Plot fields on the same disk mask as standalone BEM
    Xg, Yg = _np.meshgrid(xs * 1e3, ys * 1e3, indexing="xy")


    def _plot_component_pair(A_ref, A_kkt, name):
        A_ref_mpa = A_ref * 1e-6
        A_kkt_mpa = A_kkt * 1e-6
        v = _np.nanpercentile(_np.abs(A_ref_mpa[mask]), 99.0)
        if not _np.isfinite(v) or v <= 0:
            v = _np.nanmax(_np.abs(A_ref_mpa))
        levels = _np.linspace(-v, v, 31)

        fig, ax = _plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
        cf0 = ax[0].contourf(Xg, Yg, A_ref_mpa, levels=levels, cmap="jet", extend="both")
        ax[0].contour(Xg, Yg, A_ref_mpa, levels=15, colors="k", linewidths=0.4, alpha=0.5)
        ax[0].set_title(f"Standalone BEM {name}")

        cf1 = ax[1].contourf(Xg, Yg, A_kkt_mpa, levels=levels, cmap="jet", extend="both")
        ax[1].contour(Xg, Yg, A_kkt_mpa, levels=15, colors="k", linewidths=0.4, alpha=0.5)
        ax[1].set_title(f"KKT BEM-only {name}")

        for a in ax:
            a.set_aspect("equal", "box")
            a.set_xlabel("x [mm]")
            a.set_ylabel("y [mm]")

        cbar = fig.colorbar(cf1, ax=ax.ravel().tolist(), shrink=0.95)
        cbar.set_label("Stress [MPa]")
        _plt.show()

    _plot_component_pair(Sxx_ref, Sxx_kkt_m, "Sxx")
    _plot_component_pair(Syy_ref, Syy_kkt_m, "Syy")
    _plot_component_pair(Sxy_ref, Sxy_kkt_m, "Sxy")

    return locals()


### Case `disk_compression_2` (from `disk_compression_2.ipynb`)


In [ ]:
def run_disk_compression_2():
    """Case extracted from disk_compression_2.ipynb."""
    from pathlib import Path
    import os, sys
    import numpy as np

    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    from fracture_utils.Ubem.brazilian_disk_bem import BrazilianDiskParams, ensure_bem_field
    from fracture_utils.Ubem.disk_network_propagation import CrackGrowthParams, run_network_growth_uncoupled
    from fracture_utils.Ubem.disk_crack_plotting import PlotParams, make_standard_plot_hook, plot_total_field
    from fracture_utils.Ubem.coupling_blocks_v2 import BEMBlockV2, compose_solver_kwargs_v2
    from fracture_utils.Ubem.disk_iterative_coupling import (
        compute_crack_boundary_tractions_direct,
        solve_bem_with_extra_boundary_tractions,
    )
    from fracture_utils.Ubem.boundary_conditions import build_boundary

    # -----------------------
    # Output + mode selection
    # -----------------------
    bem_dir = repo_root / "output" / "BEM_Brazilian_disk_iterative"
    bem_dir.mkdir(parents=True, exist_ok=True)

    # Choose one:
    #  - "no_coupling"
    #  - "iterative"  (outer-cycle updates only)
    #  - "direct"     (outer-cycle updates only)
    COUPLING_METHOD = "iterative"
    OUTER_CYCLES = 2

    # -----------------------
    # Network (YOU control)
    # -----------------------
    mm = 1e-3
    vertices = np.array([
        [0, -2.0*mm, -3.0*mm],
        [1,  2.0*mm,  3.0*mm],
    ], float)
    connectivity = np.array([[0, 1]], int)

    # -----------------------
    # (1) BEM baseline
    # -----------------------
    SKIP_BEM_SOLVE = True
    disk = BrazilianDiskParams(R=25.4e-3/2, P_total=3.8e3, E=231.52e9, nu=0.3, n_elem=60, n_grid=120)

    bem_block = BEMBlockV2(disk=disk, out_dir=bem_dir)
    ensure_bem_field(
        disk,
        bem_dir,
        recompute=(not SKIP_BEM_SOLVE),
        show=True,
        save_contours=True,
    )

    boundary_mesh = build_boundary({"type": "circle", "R": disk.R, "n_boundary": disk.n_elem, "center": (0.0, 0.0)})

    # -----------------------
    # Plot options
    # -----------------------
    plot_params = PlotParams(
        cmap="jet",
        match_limits_to_crack=True,
        robust=True,
        robust_pct=97.0,
        symmetric=True,
        vmax_factor=2.0,
        n_levels=60,
        n_line_levels=15,
        dpi=300,
    )

    hook = make_standard_plot_hook(
        bem_dir=bem_dir,
        combined_out_dir=bem_dir,
        plot_params=plot_params,
        show_initial=True,
        show_cycles=True,
        show_final=True,
    )


    def _network_to_arrays(net):
        V = np.array([[int(v.id), float(v.x), float(v.y)] for v in net.vertices], float)
        C = np.array([[int(e.v0), int(e.v1)] for e in net.edges], int) if len(net.edges) else np.zeros((0,2), int)
        return V, C


    def _save_total_as_bem_arrays(Sxx_tot, Syy_tot, Sxy_tot):
        xs = np.load(bem_dir / "xs.npy")
        ys = np.load(bem_dir / "ys.npy")
        np.save(bem_dir / "xs.npy", xs)
        np.save(bem_dir / "ys.npy", ys)
        np.save(bem_dir / "Sxx.npy", np.asarray(Sxx_tot, float))
        np.save(bem_dir / "Syy.npy", np.asarray(Syy_tot, float))
        np.save(bem_dir / "Sxy.npy", np.asarray(Sxy_tot, float))


    # One-way growth kwargs (used for propagation increments)
    one_way_kwargs = compose_solver_kwargs_v2("one_way", base_solver_kwargs=None)

    # Direct kwargs (used only at outer-cycle coupling checkpoints)
    aug_payload = bem_block.build_augmented_payload(d_mode="correction_zero", gauss_n=4)
    direct_kwargs = compose_solver_kwargs_v2(
        "full_kkt",
        base_solver_kwargs={
            "augmented_enforce_crack_equilibrium": True,
            "augmented_equilibrate": True,
            "augmented_equilibrate_cols": False,
            "augmented_physical_scaling": True,
            "augmented_scale_traction_rows": True,
        },
        augmented_payload=aug_payload,
        augmented_ridge_q=1e-4,
        augmented_ridge_y=1e-8,
        augmented_keep_bem_applied=True,
    )

    V_curr = vertices.copy()
    C_curr = connectivity.copy()
    res_last = None

    for cyc in range(1, int(OUTER_CYCLES) + 1):
        print(f"\n[outer-cycle] {cyc}/{OUTER_CYCLES} method={COUPLING_METHOD}")

        # Growth for ONE outer cycle using one-way applied field from current bem_dir arrays
        grow = CrackGrowthParams(max_cycles=5, L_limit_mm=20.0, vertex_high=18, solver_kwargs=one_way_kwargs)
        out_growth = bem_dir / f"outer_{cyc:02d}_growth"
        out_growth.mkdir(parents=True, exist_ok=True)

        res_growth = run_network_growth_uncoupled(
            bem_dir=bem_dir,
            out_dir=out_growth,
            vertices=V_curr,
            connectivity=C_curr,
            params=grow,
            plot_hook=hook,
        )

        # Outer-cycle coupling update (only here)
        if COUPLING_METHOD == "iterative":
            tx_cr, ty_cr = compute_crack_boundary_tractions_direct(res=res_growth, boundary_mesh=boundary_mesh)
            solve_bem_with_extra_boundary_tractions(
                disk_params=disk,
                bem_dir=bem_dir,
                tx_extra=-tx_cr,
                ty_extra=-ty_cr,
                show=False,
            )
            res_last = res_growth

        elif COUPLING_METHOD == "direct":
            # Solve direct KKT on current network (no additional growth), then update bem_dir arrays
            grow_direct = CrackGrowthParams(max_cycles=0, L_limit_mm=20.0, vertex_high=18, solver_kwargs=direct_kwargs)
            out_direct = bem_dir / f"outer_{cyc:02d}_direct"
            out_direct.mkdir(parents=True, exist_ok=True)

            Vg, Cg = _network_to_arrays(res_growth.calc.network)
            res_direct = run_network_growth_uncoupled(
                bem_dir=bem_dir,
                out_dir=out_direct,
                vertices=Vg,
                connectivity=Cg,
                params=grow_direct,
                plot_hook=None,
            )

            Sxx_tot, Syy_tot, Sxy_tot = plot_total_field(
                bem_dir=bem_dir,
                res=res_direct,
                out_dir=out_direct,
                tag=f"outer_{cyc:02d}_direct",
                params=plot_params,
                show=True,
            )
            _save_total_as_bem_arrays(Sxx_tot, Syy_tot, Sxy_tot)
            res_last = res_direct

        else:
            # no_coupling
            res_last = res_growth

        # advance network for next outer cycle from growth result
        V_curr, C_curr = _network_to_arrays(res_growth.calc.network)

    print("\n[done] outer-cycle run completed")

    return locals()


### Case `disk_compression_inclined` (from `disk_compression_inclined.ipynb`)


In [ ]:
def run_disk_compression_inclined():
    """Case extracted from disk_compression_inclined.ipynb."""
    from pathlib import Path
    import os, sys
    import numpy as np

    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    from fracture_utils.Ubem.brazilian_disk_bem import (BrazilianDiskParams, ensure_bem_field)
    from fracture_utils.Ubem.disk_network_propagation import (CrackGrowthParams, run_network_growth_uncoupled)
    from fracture_utils.Ubem.disk_crack_plotting import PlotParams, make_standard_plot_hook

    bem_dir   = repo_root / "output" / "BEM_Brazilian_disk"
    # crack_dir = repo_root / "output" / "KinkedCrackPropagation"
    # combined_dir = bem_dir / "combined_uncoupled"

    # -----------------------
    # Network (YOU control)
    # -----------------------
    mm = 1e-3
    vertices = np.array([
        [0, -2.0*mm, -3.0*mm],
        [1,  2.0*mm,  3.0*mm],
    ], float)

    connectivity = np.array([[0, 1]], int)

    # -----------------------
    # (1) BEM (fast reuse option)
    # -----------------------
    SKIP_BEM_SOLVE = True   # <-- set False to recompute
    disk = BrazilianDiskParams(R=25.4e-3/2, P_total=3.8e3, E=231.52e9, nu=0.3, n_elem=60, n_grid=120)

    ensure_bem_field(
        disk,
        bem_dir,
        recompute=(not SKIP_BEM_SOLVE),
        show=True,              # only used if recompute=True
        save_contours=True,     # only used if recompute=True
    )

    # -----------------------
    # (2) Propagation + TOTAL-only plotting
    # -----------------------

    plot_params = PlotParams(
        cmap="jet",
        match_limits_to_crack=True,   
        robust=True,
        robust_pct=97.0,
        symmetric=True,
        vmax_factor=2.0,
        n_levels=60,
        n_line_levels=15,
        dpi=300,
    )

    hook = make_standard_plot_hook(
        bem_dir=bem_dir,
        combined_out_dir=bem_dir,
        plot_params=plot_params,
        show_initial=True,
        show_cycles=True,
        show_final=True,
    )


    grow = CrackGrowthParams(max_cycles=5, L_limit_mm=20.0, vertex_high=18)

    res_final = run_network_growth_uncoupled(
        bem_dir=bem_dir,
        out_dir=bem_dir,
        vertices=vertices,
        connectivity=connectivity,
        params=grow,
        plot_hook=hook,
    )

    return locals()


### Case `disk_energetics` (from `disk_energetics.ipynb`)


In [ ]:
def run_disk_energetics():
    """Case extracted from disk_energetics.ipynb."""
    from pathlib import Path
    import os, sys
    import json
    import shutil
    import subprocess
    from datetime import datetime
    from dataclasses import fields, MISSING
    import numpy as np
    import matplotlib.pyplot as plt

    # -----------------------
    # Repo import path
    # -----------------------
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    from fracture_utils.Ubem.brazilian_disk_bem import BrazilianDiskParams
    from fracture_utils.Ubem.boundary_conditions import build_boundary
    from fracture_utils.Ubem.bem_solver import BEMSolver2D
    from fracture_utils.Ubem.bem_stress_field import circle_inside
    from fracture_utils.Ubem.bem_stress_plotter import (
        ContourOpts,
        eval_and_plot_stress_components_contours_separate,
    )
    from fracture_utils.Ubem.direct_kkt_coupling import build_bem_dense_operators
    from fracture_utils.Ubem.crack_energetics import compute_disk_boundary_compliance_from_solver
    from fracture_utils.Ubem.disk_iterative_coupling import (
        compute_crack_boundary_tractions_direct,
        solve_bem_with_extra_boundary_tractions,
    )
    from fracture_utils.Ubem.disk_crack_plotting import PlotParams, make_standard_plot_hook

    import importlib
    import fracture_utils.Ubem.disk_network_propagation as dnp
    importlib.reload(dnp)
    CrackGrowthParams = dnp.CrackGrowthParams
    run_network_growth_uncoupled = dnp.run_network_growth_uncoupled


    # -----------------------
    # User config
    # -----------------------
    mm = 1e-3

    # Displacement-controlled loading (borrowed from BEM_displacement.ipynb)
    U_Y_TOP_IMPOSED = -1e-5
    U_Y_BOTTOM_IMPOSED = 0.0

    # Disk + material params (same family as previous notebooks)
    disk = BrazilianDiskParams(
        R=25.4e-3 / 2,
        P_total=3.8e3,     # kept for compatibility (not used in imposed_uy solve)
        E=231.52e9,
        nu=0.3,
        n_elem=90,
        
        gauss_n=6,
        plane_strain=True,
        arc_half_angle_deg=15.0,
        h=6.35e-3,
        n_grid=120,
        pad_frac=0.03,
    )

    # Crack network + growth setup (borrowed from disk_compression_inclined.ipynb)
    Kc_demo = 0.0e6
    vertices = np.array([
        [0, -1.0*mm, -6.0*mm],
        [1, -0.5*mm,  5.0*mm],
        [2,  1.0*mm, -3.0*mm],
        [3,  1.5*mm,  2.5*mm],
        [4, -2.0*mm, -2.5*mm],
        [5,  4.0*mm, -1.0*mm],
        [6, -6.0*mm,  3.0*mm],
        [7,  2.0*mm,  1.5*mm],
    ], float)
    connectivity = np.array([
        [0, 1],
        [2, 3],
        [4, 5],
        [6, 7],
    ], int)

    def _git_short_hash(default: str = "nogit") -> str:
        try:
            out = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=repo_root)
            return out.decode("utf-8", errors="ignore").strip() or default
        except Exception:
            return default


    def _run_id() -> str:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        return f"{ts}_{_git_short_hash()}"


    def _dataclass_defaults(dc_type):
        d = {}
        for f in fields(dc_type):
            if f.default is not MISSING:
                d[f.name] = f.default
            elif f.default_factory is not MISSING:  # type: ignore[attr-defined]
                d[f.name] = f.default_factory()      # type: ignore[misc]
            else:
                d[f.name] = None
        return d


    def _fmt(v):
        if isinstance(v, np.ndarray):
            return np.array2string(v, separator=", ")
        if isinstance(v, (list, tuple, dict)):
            try:
                return json.dumps(v, ensure_ascii=False)
            except Exception:
                return str(v)
        return str(v)


    def _write_provenance(run_dir: Path, run_id: str, table_rows):
        prov = run_dir / f"provenance_{run_id}.md"
        lines = []
        lines.append(f"# Provenance - {run_id}\n")
        lines.append("\n")
        lines.append(f"- Generated: {datetime.now().isoformat()}\n")
        lines.append(f"- Repo root: `{repo_root}`\n")
        lines.append(f"- Notebook: `VCM_3_4/notebooks/disk_energetics.ipynb`\n")
        lines.append("\n")
        lines.append("## Run Configuration\n")
        lines.append("\n")
        lines.append("| Group | Parameter | Default | Effective | Source |\n")
        lines.append("|---|---|---|---|---|\n")
        for g, p, d, e, s in table_rows:
            lines.append(f"| {g} | `{p}` | `{_fmt(d)}` | `{_fmt(e)}` | {s} |\n")
        prov.write_text("".join(lines), encoding="utf-8")


    def _write_provenance_json(run_dir: Path, run_id: str, table_rows):
        prov_json = run_dir / f"provenance_{run_id}.json"
        rows = []
        for g, p, d, e, s in table_rows:
            rows.append({
                "group": g,
                "parameter": p,
                "default": d,
                "effective": e,
                "source": s,
            })
        payload = {
            "run_id": run_id,
            "generated_at": datetime.now().isoformat(),
            "repo_root": str(repo_root),
            "notebook": "VCM_3_4/notebooks/disk_energetics.ipynb",
            "rows": rows,
        }

        def _json_default(obj):
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            if isinstance(obj, (np.floating, np.integer)):
                return obj.item()
            return str(obj)

        prov_json.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=_json_default), encoding="utf-8")


    run_id = _run_id()
    run_dir = repo_root / "output" / "NetworkSimulations" / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    print(f"[run_id] {run_id}")
    print(f"[run_dir] {run_dir}")

    OUT_DIR = run_dir
    BEM_DIR = run_dir
    RUN_DIR = run_dir
    BEM_DIR.mkdir(parents=True, exist_ok=True)
    RUN_DIR.mkdir(parents=True, exist_ok=True)


    # -----------------------
    # Displacement-controlled BEM solve
    # -----------------------
    def _build_disk_mesh(params: BrazilianDiskParams):
        return build_boundary({"type": "circle", "R": float(params.R), "n_boundary": int(params.n_elem), "center": (0.0, 0.0)})


    def _arc_masks(mesh, params: BrazilianDiskParams):
        theta_deg = np.asarray(mesh.theta_deg, dtype=float)
        arc_half = float(params.arc_half_angle_deg)
        top = (theta_deg >= 90.0 - arc_half) & (theta_deg <= 90.0 + arc_half)
        bot = (theta_deg >= -90.0 - arc_half) & (theta_deg <= -90.0 + arc_half)
        return top, bot


    def solve_disk_imposed_uy(params: BrazilianDiskParams, uy_top: float, uy_bottom: float = 0.0):
        mesh = _build_disk_mesh(params)
        top, bot = _arc_masks(mesh, params)
        n = int(mesh.n_seg)

        solver = BEMSolver2D(E=float(params.E), nu=float(params.nu), h=float(params.h), plane_strain=bool(params.plane_strain))
        for i in range(n):
            solver.add_element(
                float(mesh.x1[i]), float(mesh.y1[i]),
                float(mesh.x2[i]), float(mesh.y2[i]),
                True, 0.0, 0.0,
            )

        CplusH, G = build_bem_dense_operators(
            segments=solver.segs,
            E=float(params.E),
            nu=float(params.nu),
            plane_strain=bool(params.plane_strain),
            gauss_n=int(params.gauss_n),
        )

        C_bem = np.hstack([CplusH, -G])
        C_bc = np.zeros((2 * n, 4 * n), float)
        d = np.zeros((2 * n,), float)

        for i in range(n):
            r0 = 2 * i
            r1 = 2 * i + 1

            # keep u_x free by enforcing t_x = 0
            C_bc[r0, 2 * n + 2 * i + 0] = 1.0

            if top[i]:
                C_bc[r1, 2 * i + 1] = 1.0
                d[r1] = float(uy_top)
            elif bot[i]:
                C_bc[r1, 2 * i + 1] = 1.0
                d[r1] = float(uy_bottom)
            else:
                C_bc[r1, 2 * n + 2 * i + 1] = 1.0

        A = np.vstack([C_bem, C_bc])
        b = np.concatenate([np.zeros((2 * n,), float), d])
        y, *_ = np.linalg.lstsq(A, b, rcond=None)

        u = y[:2 * n]
        t = y[2 * n:]
        solver.u_x = np.asarray(u[0::2], float).copy()
        solver.u_y = np.asarray(u[1::2], float).copy()
        solver.t_x = np.asarray(t[0::2], float).copy()
        solver.t_y = np.asarray(t[1::2], float).copy()
        solver._colloc_xy = np.column_stack([np.asarray(mesh.xm, float), np.asarray(mesh.ym, float)])

        return solver, mesh


    def save_bem_field_arrays_and_plots(solver, params: BrazilianDiskParams, out_dir: Path, basename: str = "brazilian_disk"):
        bbox = (-params.R, params.R, -params.R, params.R)
        inside = circle_inside(float(params.R), center=(0.0, 0.0), pad=float(params.pad_frac) * float(params.R))

        opts_common = dict(
            dpi=300,
            show=True,
            robust=True,
            robust_pct=97.0,
            n_levels=30,
            n_line_levels=20,
            x_scale=1e3,
            y_scale=1e3,
            x_label="x [mm]",
            y_label="y [mm]",
            value_scale=1e-6,
            cbar_label="Stress [MPa]",
            cmap="jet",
        )

        xs, ys, Sxx, Syy, Sxy, _ = eval_and_plot_stress_components_contours_separate(
            solver,
            bbox=bbox,
            out_dir=out_dir,
            basename=basename,
            n=int(params.n_grid),
            inside=inside,
            normalize_by=None,
            opts_xx=ContourOpts(**opts_common, title=r"$\sigma_{xx}$", symmetric=True),
            opts_yy=ContourOpts(**opts_common, title=r"$\sigma_{yy}$", symmetric=True),
            opts_xy=ContourOpts(**opts_common, title=r"$\sigma_{xy}$", symmetric=True),
        )

        # Save with canonical names expected by run_network_growth_uncoupled
        np.save(out_dir / "xs.npy", xs)
        np.save(out_dir / "ys.npy", ys)
        np.save(out_dir / "Sxx.npy", Sxx)
        np.save(out_dir / "Syy.npy", Syy)
        np.save(out_dir / "Sxy.npy", Sxy)

        return xs, ys, Sxx, Syy, Sxy


    solver_bem, mesh_bem = solve_disk_imposed_uy(
        disk,
        uy_top=float(U_Y_TOP_IMPOSED),
        uy_bottom=float(U_Y_BOTTOM_IMPOSED),
    )
    _ = save_bem_field_arrays_and_plots(solver_bem, disk, BEM_DIR)

    comp_ref = compute_disk_boundary_compliance_from_solver(
        solver=solver_bem,
        mesh=mesh_bem,
        h=float(disk.h),
        arc_half_angle_deg=float(disk.arc_half_angle_deg),
    )
    C_REL_REF = float(comp_ref.get("C_rel_m_per_N", np.nan))
    print("[bem] reference C_rel_m_per_N =", C_REL_REF)

    # Mutable compliance state updated after each outer-cycle BEM update.
    _compliance_state = {"C_rel": float(C_REL_REF)}


    def compliance_callback(**kwargs):
        return float(_compliance_state["C_rel"])


    # -----------------------
    # Crack growth run + metrics save
    # -----------------------
    plot_params = PlotParams(
        cmap="jet",
        match_limits_to_crack=True,
        robust=True,
        robust_pct=97.0,
        symmetric=True,
        vmax_factor=2.0,
        n_levels=60,
        n_line_levels=15,
        dpi=300,
    )

    hook = make_standard_plot_hook(
        bem_dir=BEM_DIR,
        combined_out_dir=RUN_DIR,
        plot_params=plot_params,
        show_initial=True,
        show_cycles=True,
        show_final=True,
    )

    # Mode switch
    #   "no_coupling" : single growth run using fixed BEM field
    #   "iterative"   : outer-loop traction feedback update of BEM
    COUPLING_METHOD = "iterative"
    OUTER_CYCLES = 8
    n_inner_max_for_index = 8


    def _network_to_arrays(net):
        V = np.array([[int(v.id), float(v.x), float(v.y)] for v in net.vertices], float)
        C = np.array([[int(e.v0), int(e.v1)] for e in net.edges], int) if len(net.edges) else np.zeros((0, 2), int)
        return V, C


    def _cycle_dir_sort_key(p: Path):
        name = p.name
        idx = -1
        if name.startswith("cycle_"):
            parts = name.split("_")
            if len(parts) > 1 and parts[1].isdigit():
                idx = int(parts[1])

        phase_rank = 0
        if "terminated" in name:
            phase_rank = 4
        elif "post_simplify" in name:
            phase_rank = 3
        elif "pre_simplify" in name:
            phase_rank = 2
        elif "initial" in name:
            phase_rank = 1
        return (idx, phase_rank, name)


    def _latest_cycle_plot_dir(outer_dir: Path) -> Path:
        candidates = []
        for p in outer_dir.glob("cycle_*"):
            if p.is_dir() and (p / "network_graph.png").exists() and (p / "deformed_network.png").exists():
                candidates.append(p)
        if not candidates:
            return outer_dir
        return sorted(candidates, key=_cycle_dir_sort_key)[-1]


    def _save_outer_boundary_snapshot_to_cycle_dir(*, source_bem_dir: Path, outer_dir: Path, outer_cycle: int):
        dst = _latest_cycle_plot_dir(outer_dir)
        file_map = [
            ("xs.npy", "outer_boundary_xs.npy"),
            ("ys.npy", "outer_boundary_ys.npy"),
            ("Sxx.npy", "outer_boundary_Sxx.npy"),
            ("Syy.npy", "outer_boundary_Syy.npy"),
            ("Sxy.npy", "outer_boundary_Sxy.npy"),
            ("brazilian_disk_iter_sxx_contour.png", "outer_boundary_sxx_contour.png"),
            ("brazilian_disk_iter_syy_contour.png", "outer_boundary_syy_contour.png"),
            ("brazilian_disk_iter_sxy_contour.png", "outer_boundary_sxy_contour.png"),
        ]

        copied = []
        for src_name, dst_name in file_map:
            src = Path(source_bem_dir) / src_name
            if src.exists():
                shutil.copy2(src, dst / dst_name)
                copied.append(dst_name)

        manifest = {
            "outer_cycle": int(outer_cycle),
            "saved_at": datetime.now().isoformat(),
            "source_bem_dir": str(source_bem_dir),
            "destination_dir": str(dst),
            "copied_files": copied,
        }
        (dst / f"outer_boundary_snapshot_outer_{int(outer_cycle):02d}.json").write_text(
            json.dumps(manifest, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        print(f"[outer-cycle] saved outer-boundary snapshot to {dst}")


    base_growth_kwargs = dict(
        max_cycles=4,
        L_limit_mm=20.0,
        vertex_high=25,
        max_inner_cycles_per_outer=6,
        enable_inner_cycle_metrics_save=True,
        domain_area_m2=float(np.pi * disk.R * disk.R),
        disk_radius_m=float(disk.R),
        arc_half_angle_deg=float(disk.arc_half_angle_deg),
        boundary_band_frac=0.05,
        loading_axis=(0.0, 1.0),
        control_mode="displacement",
        disp_delta_m=float(abs(U_Y_TOP_IMPOSED - U_Y_BOTTOM_IMPOSED)),
        disk_thickness_m=float(disk.h),
        inner_cycle_max_for_index=int(n_inner_max_for_index),
        compliance_evaluator=compliance_callback,
        check_tip_outside_disk=True,
        tip_outside_tolerance_m=0.0,
        # Match VCM_3_0 kinked-propagation strategy: iterative intersections + conservative simplification.
        intersection_detect_mode="iterative",
        intersection_verbose=False,
        simplify_each_step=False,
        simplification_config={
            "max_angle_deviation": 5.0,
            "min_edge_length": 0.50 * mm,
            "merge_at_degree2": True,
            "remove_degree2_nodes": False,
            "merge_vertex_tolerance": 1.0 * mm,
            "max_merged_edge_length": 20.0 * mm,
            "preserve_tips": True,
            "preserve_junctions": True,
        },
        solver_kwargs={
            "ne_half": 40,
            "Kc_demo": Kc_demo,
            "refine_kinks": False,
            "refine_junction_endpoints": False,
            "endpoint_min_nodes": 4,
            "kink_side_nodes": 1,
            "tip_cluster": "power",
            "tip_cluster_power": 2.5,
            "other_min_panels": 4,
        },
    )

    V_curr = vertices.copy()
    C_curr = connectivity.copy()
    all_rows = []
    res_final = None

    if COUPLING_METHOD == "no_coupling":
        grow = CrackGrowthParams(**base_growth_kwargs)
        res_final = run_network_growth_uncoupled(
            bem_dir=BEM_DIR,
            out_dir=RUN_DIR,
            vertices=V_curr,
            connectivity=C_curr,
            params=grow,
            plot_hook=hook,
        )

        m = np.load(RUN_DIR / "inner_cycle_metrics.npz", allow_pickle=True)
        nloc = int(np.asarray(m["total_cycle_number"]).size)
        for i in range(nloc):
            row = {k: np.asarray(m[k])[i] for k in m.files}
            all_rows.append(row)

    elif COUPLING_METHOD == "iterative":
        boundary_mesh = build_boundary({"type": "circle", "R": float(disk.R), "n_boundary": int(disk.n_elem), "center": (0.0, 0.0)})

        for oc in range(1, int(OUTER_CYCLES) + 1):
            print(f"\n[outer-cycle] {oc}/{OUTER_CYCLES} mode=iterative")
            outer_dir = RUN_DIR / f"outer_{oc:02d}"
            outer_dir.mkdir(parents=True, exist_ok=True)

            grow = CrackGrowthParams(**base_growth_kwargs)
            res_outer = run_network_growth_uncoupled(
                bem_dir=BEM_DIR,
                out_dir=outer_dir,
                vertices=V_curr,
                connectivity=C_curr,
                params=grow,
                plot_hook=hook,
            )
            res_final = res_outer

            # Concatenate metrics with continuous total_cycle_number over outer loops
            m = np.load(outer_dir / "inner_cycle_metrics.npz", allow_pickle=True)
            nloc = int(np.asarray(m["total_cycle_number"]).size)
            cycle_offset = (oc - 1) * int(n_inner_max_for_index)
            for i in range(nloc):
                row = {k: np.asarray(m[k])[i] for k in m.files}
                row["total_cycle_number"] = int(row["total_cycle_number"]) + int(cycle_offset)
                row["outer_loop_index"] = int(oc)
                all_rows.append(row)

            # Iterative coupling update: crack-induced boundary tractions -> BEM
            tx_cr, ty_cr = compute_crack_boundary_tractions_direct(res=res_outer, boundary_mesh=boundary_mesh)
            solver_iter, mesh_iter, _ = solve_bem_with_extra_boundary_tractions(
                disk_params=disk,
                bem_dir=BEM_DIR,
                tx_extra=-tx_cr,
                ty_extra=-ty_cr,
                show=False,
            )
            _save_outer_boundary_snapshot_to_cycle_dir(
                source_bem_dir=BEM_DIR,
                outer_dir=outer_dir,
                outer_cycle=oc,
            )

            comp_iter = compute_disk_boundary_compliance_from_solver(
                solver=solver_iter,
                mesh=mesh_iter,
                h=float(disk.h),
                arc_half_angle_deg=float(disk.arc_half_angle_deg),
            )
            _compliance_state["C_rel"] = float(comp_iter.get("C_rel_m_per_N", np.nan))
            print(f"[outer-cycle] updated C_rel_m_per_N = {_compliance_state['C_rel']:.6e}")

            # Next outer cycle starts from updated network
            V_curr, C_curr = _network_to_arrays(res_outer.calc.network)

    else:
        raise ValueError("COUPLING_METHOD must be 'no_coupling' or 'iterative'.")


    # Save combined metrics to canonical location for plotting
    if not all_rows:
        raise RuntimeError("No inner-cycle metrics collected.")

    all_keys = sorted({k for r in all_rows for k in r.keys()})
    combined = {}
    for k in all_keys:
        vals = [r.get(k, np.nan) for r in all_rows]
        arr = np.asarray(vals)
        if arr.dtype.kind in ("i", "u", "f", "b"):
            combined[k] = arr
        else:
            combined[k] = arr.astype(str)

    np.savez(RUN_DIR / "inner_cycle_metrics.npz", **combined)


    # -----------------------
    # Load + plot saved inner-cycle metrics vs total cycle number
    # -----------------------
    metrics_npz = RUN_DIR / "inner_cycle_metrics.npz"
    if not metrics_npz.exists():
        raise FileNotFoundError(f"Missing metrics file: {metrics_npz}")

    m = np.load(metrics_npz, allow_pickle=True)
    x = np.asarray(m["total_cycle_number"], int)

    # Print all per-inner-cycle metrics (including K_eff) in table form
    exclude_nonmetrics = {"control_mode"}
    print("\n[inner-cycle metrics]\n")
    metric_cols = [k for k in m.files if k not in exclude_nonmetrics and np.asarray(m[k]).ndim == 1 and np.asarray(m[k]).size == x.size]
    metric_cols = sorted(metric_cols)

    # Keep identifiers first for readability
    front = [k for k in ["total_cycle_number", "outer_loop_index", "outer_cycle", "inner_step", "global_step"] if k in metric_cols]
    rest = [k for k in metric_cols if k not in front]
    cols = front + rest

    print(" | ".join(cols))
    print("-|-".join(["---"] * len(cols)))
    for i in range(x.size):
        vals = []
        for k in cols:
            v = np.asarray(m[k])[i]
            arr_k = np.asarray(m[k])
            if np.issubdtype(arr_k.dtype, np.integer):
                vals.append(f"{int(v)}")
            elif np.issubdtype(arr_k.dtype, np.floating):
                vals.append("nan" if not np.isfinite(v) else f"{float(v):.6e}")
            else:
                vals.append(str(v))
        print(" | ".join(vals))

    # Plot ALL numeric metrics versus total cycle number
    exclude = {
        "total_cycle_number",
        "cycle_index",
        "outer_cycle",
        "inner_step",
        "global_step",
        "control_mode",
    }
    metric_keys = []
    for k in m.files:
        if k in exclude:
            continue
        arr = np.asarray(m[k])
        if arr.ndim == 1 and arr.size == x.size and np.issubdtype(arr.dtype, np.number):
            metric_keys.append(k)

    metric_keys = sorted(metric_keys)
    if not metric_keys:
        raise RuntimeError("No numeric metric arrays found to plot.")

    n = len(metric_keys)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.6*nrows), constrained_layout=True)
    axs = np.atleast_1d(axs).ravel()

    for i, k in enumerate(metric_keys):
        ax = axs[i]
        y = np.asarray(m[k], float)
        ax.plot(x, y, "-o", ms=3)
        ax.set_xlabel("total_cycle_number")
        ax.set_ylabel(k)
        ax.set_title(k)
        ax.grid(True, alpha=0.3)

    for j in range(len(metric_keys), len(axs)):
        axs[j].set_axis_off()

    plt.show()
    print("[done] disk energetics workflow complete")




    # ---- next cell ----

    # Extract alpha, Q, Keffe_mean vs total cycle number and plot/save them
    from pathlib import Path
    import re
    import numpy as np
    import matplotlib.pyplot as plt

    # Resolve run directory and metrics path with fallback if RUN_DIR is not defined
    if "RUN_DIR" in globals():
        run_dir = Path(RUN_DIR)
    else:
        cwd = Path.cwd().resolve()
        repo_root_guess = cwd.parent if cwd.name == "notebooks" else cwd
        base = repo_root_guess / "output" / "NetworkSimulations"
        candidates = sorted([p for p in base.glob("*") if p.is_dir()])
        if not candidates:
            raise FileNotFoundError(f"No run directories found in {base}")
        run_dir = candidates[-1]

    metrics_npz = run_dir / "inner_cycle_metrics.npz"


    def _rows_to_metrics(rows):
        keys = sorted({k for r in rows for k in r.keys()})
        out = {}
        for k in keys:
            vals = [r.get(k, np.nan) for r in rows]
            arr = np.asarray(vals)
            if arr.dtype.kind in ("i", "u", "f", "b"):
                out[k] = arr
            else:
                out[k] = arr.astype(str)
        return out


    def _build_from_outer_dirs(base_dir):
        rows = []
        running_total = 0
        outer_dirs = sorted([p for p in base_dir.glob("outer_*") if p.is_dir()])

        for od in outer_dirs:
            p = od / "inner_cycle_metrics.npz"
            if not p.exists():
                continue

            mloc = np.load(p, allow_pickle=True)
            if "total_cycle_number" not in mloc.files:
                continue

            nloc = int(np.asarray(mloc["total_cycle_number"]).size)
            if nloc == 0:
                continue

            local_total = np.asarray(mloc["total_cycle_number"], int)
            local_total = local_total - int(np.min(local_total)) + 1

            mo = re.search(r"outer_(\d+)", od.name)
            oc = int(mo.group(1)) if mo else np.nan

            for i in range(nloc):
                row = {k: np.asarray(mloc[k])[i] for k in mloc.files}
                row["total_cycle_number"] = int(local_total[i]) + int(running_total)
                if "outer_loop_index" not in row:
                    row["outer_loop_index"] = int(oc) if np.isfinite(oc) else np.nan
                rows.append(row)

            running_total = int(max(r["total_cycle_number"] for r in rows))

        return _rows_to_metrics(rows) if rows else None


    m = None
    source_label = None

    # Prefer reconstruction from per-outer files because it is robust to interrupted runs.
    m_outer = _build_from_outer_dirs(run_dir)
    if m_outer is not None and "total_cycle_number" in m_outer:
        m = m_outer
        source_label = "outer_* metrics"
    elif metrics_npz.exists():
        mz = np.load(metrics_npz, allow_pickle=True)
        m = {k: np.asarray(mz[k]) for k in mz.files}
        source_label = str(metrics_npz)
    else:
        raise FileNotFoundError(
            f"No metrics found in {run_dir}. Run the growth workflow cell first."
        )

    x_all = np.asarray(m["total_cycle_number"], int)
    mask = (x_all >= 1) & (x_all <= 75)
    if not np.any(mask):
        raise RuntimeError("No cycles in [1, 75] found for plotting.")

    x = x_all[mask]


    def _pick_metric(candidates, label):
        for k in candidates:
            if k in m:
                return np.asarray(m[k], float), k
        raise KeyError(f"Could not find {label}. Tried {candidates}. Available keys: {list(m.keys())}")


    def _pick_keffe_mean():
        explicit = [
            "Keffe_mean",
            "K_effe_mean",
            "K_eff_mean",
            "Keff_mean",
            "K_eff_mean_MPa_sqrt_m",
        ]
        for k in explicit:
            if k in m:
                return np.asarray(m[k], float), k

        for k in m.keys():
            kl = str(k).lower()
            if ("k_eff" in kl or "keff" in kl) and "mean" in kl:
                arr = np.asarray(m[k])
                if arr.ndim == 1 and arr.size == x_all.size and np.issubdtype(arr.dtype, np.number):
                    return np.asarray(arr, float), k

        raise KeyError(
            "Could not find Keffe_mean-like metric. "
            f"Available keys: {list(m.keys())}"
        )


    def _add_stable_crack_annotation(ax, y):
        x_line = 26
        ax.axvline(x_line, color="red", linestyle="--", linewidth=1.5)

        y = np.asarray(y, float)
        y_finite = y[np.isfinite(y)]
        if y_finite.size == 0:
            y_mid = 0.0
        else:
            y_mid = 0.5 * (float(np.min(y_finite)) + float(np.max(y_finite)))

        x0 = x_line + 1.0
        x1 = x_line + 3.0
        ax.annotate(
            "",
            xy=(x1, y_mid),
            xytext=(x0, y_mid),
            arrowprops=dict(arrowstyle="->", color="black", linewidth=3.0),
        )
        ax.text(x1 + 0.3, y_mid, "Stable Crack", color="black", va="center", ha="left")


    alpha_all, alpha_key = _pick_metric(["alpha", "alpha_mean"], "alpha")
    Q_all, Q_key = _pick_metric(["Q", "Q_mean", "Q_total"], "Q")
    Keffe_all, Keffe_key = _pick_keffe_mean()

    alpha = alpha_all[mask]
    Q = Q_all[mask]
    Keffe = Keffe_all[mask]

    # Save directory for plots
    plots_dir = run_dir / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)

    # 1) Damage Metric alpha
    fig1, ax1 = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    ax1.plot(x, alpha, "-o", ms=3)
    ax1.set_xlabel("Growth Cycle Number")
    ax1.set_ylabel(r"Damage Metric, $\alpha$, m$^{-1}$")
    ax1.set_xlim(1, 75)
    ax1.grid(True, alpha=0.3)
    _add_stable_crack_annotation(ax1, alpha)
    fig1.savefig(plots_dir / "damage_metric_alpha_vs_growth_cycle.png", dpi=300)

    # 2) Orientation Factor Q
    fig2, ax2 = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    ax2.plot(x, Q, "-o", ms=3)
    ax2.set_xlabel("Growth Cycle Number")
    ax2.set_ylabel("Orientation Factor, Q")
    ax2.set_xlim(1, 75)
    ax2.grid(True, alpha=0.3)
    _add_stable_crack_annotation(ax2, Q)
    fig2.savefig(plots_dir / "orientation_factor_Q_vs_growth_cycle.png", dpi=300)

    # 3) Crack Tip SIF Keff
    fig3, ax3 = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    ax3.plot(x, Keffe, "-o", ms=3)
    ax3.set_xlabel("Growth Cycle Number")
    ax3.set_ylabel(r"Crack Tip SIF, K$_{eff}$, MPa$\sqrt{m}$")
    ax3.set_xlim(1, 75)
    ax3.axhline(1.00, color="black", linestyle="-.", linewidth=1.5)
    ax3.text(27.0, 1.03, "Fracture Toughness", color="black", va="bottom", ha="left")
    ax3.grid(True, alpha=0.3)
    _add_stable_crack_annotation(ax3, Keffe)
    fig3.savefig(plots_dir / "crack_tip_sif_keff_vs_growth_cycle.png", dpi=300)

    plt.show()
    print(f"[metrics source] {source_label}")
    print(f"[cycles plotted] min={int(np.min(x))}, max={int(np.max(x))}, count={int(x.size)}")
    print(f"[selected keys] alpha={alpha_key}, Q={Q_key}, Keffe_mean={Keffe_key}")
    print(f"[saved plots] {plots_dir}")

    return locals()


### Case `disk_experiments` (from `disk_experiments.ipynb`)


In [ ]:
def run_disk_experiments():
    """# Disk Experiments

Configurable experiment runner for Brazilian-disk + crack-network coupling.

This notebook:
- exposes run inputs in one place,
- writes all outputs to `output/disk_experiments/<run_id>`,
- writes a provenance file containing run configuration."""

    from pathlib import Path
    from datetime import datetime
    import os
    import sys
    import json
    import subprocess
    from dataclasses import asdict, fields, MISSING

    import numpy as np

    # -----------------------
    # Repo setup
    # -----------------------
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    from fracture_utils.Ubem.brazilian_disk_bem import BrazilianDiskParams, ensure_bem_field
    from fracture_utils.Ubem.disk_network_propagation import CrackGrowthParams, run_network_growth_uncoupled
    from fracture_utils.Ubem.disk_crack_plotting import PlotParams, make_standard_plot_hook, plot_total_field
    from fracture_utils.Ubem.coupling_blocks_v2 import BEMBlockV2, compose_solver_kwargs_v2
    from fracture_utils.Ubem.disk_iterative_coupling import (
        compute_crack_boundary_tractions_direct,
        solve_bem_with_extra_boundary_tractions,
    )
    from fracture_utils.Ubem.boundary_conditions import build_boundary


    # =======================
    # USER CONFIG (edit here)
    # =======================

    # Run control===
    COUPLING_METHOD = "iterative"   # "no_coupling" | "iterative" | "direct"
    OUTER_CYCLES = 4
    SKIP_BEM_SOLVE = True

    # Initial network geometry
    mm = 1e-3
    # VERTICES = np.array([
    #     [0, 0.0 * mm, -3.0 * mm],
    #     [1, 0.0 * mm,  3.0 * mm],
    # ], float)
    # CONNECTIVITY = np.array([[0, 1]], int)
    VERTICES = np.array([
        [0,   0.0*mm,  0.0*mm],   
        [1,  1.0*mm,  0.0*mm],   
        [2,  2.0*mm,  1*mm],   
        [3,  4.0*mm,  2*mm],   
        [4,  10.0*mm, 2.0*mm],
        [5,  12.0*mm, -1.0*mm],
        [6,  9.0*mm, -2*mm],
        [7,  7.0*mm, -4.0*mm],
        [8,  7.0*mm, -6.0*mm],
        [9, 8.0*mm, -8.0*mm]
    ], dtype=float)

    CONNECTIVITY = np.array([
        [0, 1],   # e0
        [1, 2],   # e1
        [2, 3],
        [3, 4],
        [4, 5],
        [5, 6],
        [6, 7],
        [7, 8],
        [7, 9],
    ], dtype=int)

    # Disk/BEM params (start from module defaults, set explicit current values)
    DISK_PARAMS = dict(
        R=25.4e-3 / 2,
        P_total=3.8e3,
        E=231.52e9,
        nu=0.3,
        n_elem=120,
        gauss_n=6,
        plane_strain=True,
        arc_half_angle_deg=10.0,
        h=6.35e-3,
        n_grid=120,
        pad_frac=0.03,
    )

    # BEM baseline compute/load behavior
    BEM_FIELD_CONFIG = dict(
        show=True,
        save_contours=True,
    )

    # Plot config
    PLOT_PARAMS = dict(
        cmap="jet",
        n_bands=40,
        vmax_factor=2.0,
        dpi=300,
        match_limits_to_crack=True,
        robust=True,
        robust_pct=97.0,
        symmetric=True,
        n_levels=60,
        n_line_levels=15,
        fixed_vlim_mpa=None,
        augmented_correction_stride=4,
    )

    HOOK_CONFIG = dict(
        show_initial=True,
        show_cycles=True,
        show_final=True,
        only_post_simplify=True,
    )

    # Solver kwargs exposed (defaults from run_network_growth_uncoupled)
    SOLVER_KWARGS_DEFAULTS = dict(
        ne_half=60,
        representation="cheb_quad",
        node_distribution="tip_dense",
        solver_option="parametrized_crack",
        parametrization="polyline",
        nq_col=9,
        nq_stress=12,
        crack_mode="half",
        junction_model="core",
    )

    # One-way run solver kwargs (editable)
    ONE_WAY_SOLVER_KWARGS = dict(SOLVER_KWARGS_DEFAULTS)

    # Growth / propagation params (including newly used simplification_config)
    GROWTH_PARAMS = dict(
        material_E=200e9,
        material_nu=0.30,
        plane_stress=False,
        f0=0.03,
        f_fixed=0.03,
        step_mode="fixed_step",
        simultaneous_tip_growth=True,
        Kc_demo=1e5,
        rmax_frac=0.25,
        min_pts=8,
        max_cycles=8,
        vertex_high=16,
        L_limit_mm=20.0,
        simplification_config=None,   # e.g. {"max_angle_deviation": 2.0, ...}
    )

    # Direct KKT payload + solver knobs (used only when COUPLING_METHOD == "direct")
    AUGMENTED_CONFIG = dict(
        d_mode="correction_zero",
        payload_gauss_n=4,
        augmented_enforce_crack_equilibrium=True,
        augmented_equilibrate=True,
        augmented_equilibrate_cols=False,
        augmented_physical_scaling=True,
        augmented_scale_traction_rows=True,
        augmented_ridge_q=1e-4,
        augmented_ridge_y=1e-8,
        augmented_keep_bem_applied=True,
    )

    # Iterative BEM update controls
    ITERATIVE_UPDATE_CONFIG = dict(
        show=False,
        gauss_n=None,   # None => disk.gauss_n
    )


    # =======================
    # Utilities
    # =======================

    def _git_short_hash(default="nogit"):
        try:
            out = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=repo_root)
            return out.decode("utf-8", errors="ignore").strip() or default
        except Exception:
            return default


    def _run_id():
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        return f"{ts}_{_git_short_hash()}"


    def _network_to_arrays(net):
        V = np.array([[int(v.id), float(v.x), float(v.y)] for v in net.vertices], float)
        C = np.array([[int(e.v0), int(e.v1)] for e in net.edges], int) if len(net.edges) else np.zeros((0, 2), int)
        return V, C


    def _save_total_as_bem_arrays(bem_dir: Path, Sxx_tot, Syy_tot, Sxy_tot):
        xs = np.load(bem_dir / "xs.npy")
        ys = np.load(bem_dir / "ys.npy")
        np.save(bem_dir / "xs.npy", xs)
        np.save(bem_dir / "ys.npy", ys)
        np.save(bem_dir / "Sxx.npy", np.asarray(Sxx_tot, float))
        np.save(bem_dir / "Syy.npy", np.asarray(Syy_tot, float))
        np.save(bem_dir / "Sxy.npy", np.asarray(Sxy_tot, float))


    def _dataclass_defaults(dc_type):
        d = {}
        for f in fields(dc_type):
            if f.default is not MISSING:
                d[f.name] = f.default
            elif f.default_factory is not MISSING:  # type: ignore[attr-defined]
                d[f.name] = f.default_factory()      # type: ignore[misc]
            else:
                d[f.name] = None
        return d


    def _fmt(v):
        if isinstance(v, np.ndarray):
            return np.array2string(v, separator=", ")
        if isinstance(v, (list, tuple, dict)):
            try:
                return json.dumps(v, ensure_ascii=False)
            except Exception:
                return str(v)
        return str(v)


    def _write_provenance(run_dir: Path, run_id: str, table_rows):
        prov = run_dir / f"provenance_{run_id}.md"
        lines = []
        lines.append(f"# Provenance - {run_id}\n")
        lines.append("")
        lines.append(f"- Generated: {datetime.now().isoformat()}\n")
        lines.append(f"- Repo root: `{repo_root}`\n")
        lines.append(f"- Notebook: `VCM_3_2/notebooks/disk_experiments.ipynb`\n")
        lines.append("")
        lines.append("## Run Configuration\n")
        lines.append("")
        lines.append("| Group | Parameter | Default | Effective | Source |\n")
        lines.append("|---|---|---|---|---|\n")
        for g, p, d, e, s in table_rows:
            lines.append(f"| {g} | `{p}` | `{_fmt(d)}` | `{_fmt(e)}` | {s} |\n")
        prov.write_text("".join(lines), encoding="utf-8")


    def _write_provenance_json(run_dir: Path, run_id: str, table_rows):
        prov_json = run_dir / f"provenance_{run_id}.json"
        rows = []
        for g, p, d, e, s in table_rows:
            rows.append({
                "group": g,
                "parameter": p,
                "default": d,
                "effective": e,
                "source": s,
            })
        payload = {
            "run_id": run_id,
            "generated_at": datetime.now().isoformat(),
            "repo_root": str(repo_root),
            "notebook": "VCM_3_2/notebooks/disk_experiments.ipynb",
            "rows": rows,
        }

        def _json_default(obj):
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            if isinstance(obj, (np.floating, np.integer)):
                return obj.item()
            return str(obj)

        prov_json.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=_json_default), encoding="utf-8")


    # =======================
    # Build run directories
    # =======================
    run_id = _run_id()
    run_dir = repo_root / "output" / "disk_experiments" / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    print(f"[run_id] {run_id}")
    print(f"[run_dir] {run_dir}")


    # =======================
    # Instantiate config objects
    # =======================
    disk = BrazilianDiskParams(**DISK_PARAMS)
    plot_params = PlotParams(**PLOT_PARAMS)

    boundary_mesh = build_boundary({
        "type": "circle",
        "R": disk.R,
        "n_boundary": disk.n_elem,
        "center": (0.0, 0.0),
    })

    bem_block = BEMBlockV2(disk=disk, out_dir=run_dir)
    one_way_kwargs = compose_solver_kwargs_v2("one_way", base_solver_kwargs=ONE_WAY_SOLVER_KWARGS)

    aug_payload = bem_block.build_augmented_payload(
        d_mode=AUGMENTED_CONFIG["d_mode"],
        gauss_n=AUGMENTED_CONFIG["payload_gauss_n"],
    )
    direct_kwargs = compose_solver_kwargs_v2(
        "full_kkt",
        base_solver_kwargs={
            "augmented_enforce_crack_equilibrium": AUGMENTED_CONFIG["augmented_enforce_crack_equilibrium"],
            "augmented_equilibrate": AUGMENTED_CONFIG["augmented_equilibrate"],
            "augmented_equilibrate_cols": AUGMENTED_CONFIG["augmented_equilibrate_cols"],
            "augmented_physical_scaling": AUGMENTED_CONFIG["augmented_physical_scaling"],
            "augmented_scale_traction_rows": AUGMENTED_CONFIG["augmented_scale_traction_rows"],
        },
        augmented_payload=aug_payload,
        augmented_ridge_q=AUGMENTED_CONFIG["augmented_ridge_q"],
        augmented_ridge_y=AUGMENTED_CONFIG["augmented_ridge_y"],
        augmented_keep_bem_applied=AUGMENTED_CONFIG["augmented_keep_bem_applied"],
    )

    hook = make_standard_plot_hook(
        bem_dir=run_dir,
        combined_out_dir=run_dir,
        plot_params=plot_params,
        show_initial=HOOK_CONFIG["show_initial"],
        show_cycles=HOOK_CONFIG["show_cycles"],
        show_final=HOOK_CONFIG["show_final"],
        only_post_simplify=HOOK_CONFIG["only_post_simplify"],
    )


    # =======================
    # Provenance table rows
    # =======================
    rows = []

    # Run control
    rows += [
        ("run", "COUPLING_METHOD", "iterative", COUPLING_METHOD, "notebook"),
        ("run", "OUTER_CYCLES", 5, OUTER_CYCLES, "notebook"),
        ("run", "SKIP_BEM_SOLVE", True, SKIP_BEM_SOLVE, "notebook"),
    ]

    # Network
    rows += [
        ("network", "mm", 1e-3, mm, "notebook"),
        ("network", "vertices", "(current in source)", VERTICES, "notebook"),
        ("network", "connectivity", "(current in source)", CONNECTIVITY, "notebook"),
    ]

    # Disk params defaults vs effective
    disk_defaults = _dataclass_defaults(BrazilianDiskParams)
    for k, v in asdict(disk).items():
        rows.append(("disk", k, disk_defaults.get(k), v, "notebook/default"))

    # BEM field controls
    rows += [
        ("bem_field", "recompute", False, (not SKIP_BEM_SOLVE), "derived"),
        ("bem_field", "show", True, BEM_FIELD_CONFIG["show"], "notebook"),
        ("bem_field", "save_contours", True, BEM_FIELD_CONFIG["save_contours"], "notebook"),
    ]

    # Plot params defaults vs effective
    plot_defaults = PlotParams().__dict__.copy()
    for k, v in plot_params.__dict__.items():
        rows.append(("plot", k, plot_defaults.get(k), v, "notebook/default"))
    for k, v in HOOK_CONFIG.items():
        hook_default = dict(show_initial=True, show_cycles=False, show_final=True, only_post_simplify=True).get(k)
        rows.append(("plot_hook", k, hook_default, v, "notebook/default"))

    # Solver kwargs
    for k, v in SOLVER_KWARGS_DEFAULTS.items():
        rows.append(("solver_kwargs", k, v, ONE_WAY_SOLVER_KWARGS.get(k, v), "notebook/default"))

    # Growth params
    growth_defaults = _dataclass_defaults(CrackGrowthParams)
    for k, v in GROWTH_PARAMS.items():
        rows.append(("growth", k, growth_defaults.get(k), v, "notebook/default"))

    # Direct / augmented
    rows += [
        ("augmented", "d_mode", "correction_zero", AUGMENTED_CONFIG["d_mode"], "notebook/default"),
        ("augmented", "payload_gauss_n", 4, AUGMENTED_CONFIG["payload_gauss_n"], "notebook/default"),
        ("augmented", "augmented_ridge_q", 1e-4, AUGMENTED_CONFIG["augmented_ridge_q"], "notebook/default"),
        ("augmented", "augmented_ridge_y", 1e-10, AUGMENTED_CONFIG["augmented_ridge_y"], "notebook/default"),
        ("augmented", "augmented_keep_bem_applied", True, AUGMENTED_CONFIG["augmented_keep_bem_applied"], "notebook/default"),
    ]

    # Iterative update
    rows += [
        ("iterative_update", "show", False, ITERATIVE_UPDATE_CONFIG["show"], "notebook/default"),
        ("iterative_update", "gauss_n", "disk.gauss_n", ITERATIVE_UPDATE_CONFIG["gauss_n"], "notebook/default"),
    ]

    _write_provenance(run_dir, run_id, rows)
    _write_provenance_json(run_dir, run_id, rows)
    print(f"[provenance] {run_dir / ('provenance_' + run_id + '.md')}")
    print(f"[provenance-json] {run_dir / ('provenance_' + run_id + '.json')}")


    # =======================
    # Baseline BEM field
    # =======================
    ensure_bem_field(
        disk,
        run_dir,
        recompute=(not SKIP_BEM_SOLVE),
        show=bool(BEM_FIELD_CONFIG["show"]),
        save_contours=bool(BEM_FIELD_CONFIG["save_contours"]),
    )


    # =======================
    # Outer-cycle run
    # =======================
    V_curr = VERTICES.copy()
    C_curr = CONNECTIVITY.copy()
    res_last = None

    for cyc in range(1, int(OUTER_CYCLES) + 1):
        print(f"\n[outer-cycle] {cyc}/{OUTER_CYCLES} method={COUPLING_METHOD}")

        grow = CrackGrowthParams(
            material_E=GROWTH_PARAMS["material_E"],
            material_nu=GROWTH_PARAMS["material_nu"],
            plane_stress=GROWTH_PARAMS["plane_stress"],
            solver_kwargs=one_way_kwargs,
            f0=GROWTH_PARAMS["f0"],
            f_fixed=GROWTH_PARAMS["f_fixed"],
            step_mode=GROWTH_PARAMS["step_mode"],
            simultaneous_tip_growth=GROWTH_PARAMS["simultaneous_tip_growth"],
            Kc_demo=GROWTH_PARAMS["Kc_demo"],
            rmax_frac=GROWTH_PARAMS["rmax_frac"],
            min_pts=GROWTH_PARAMS["min_pts"],
            max_cycles=GROWTH_PARAMS["max_cycles"],
            vertex_high=GROWTH_PARAMS["vertex_high"],
            L_limit_mm=GROWTH_PARAMS["L_limit_mm"],
            simplification_config=GROWTH_PARAMS["simplification_config"],
        )

        out_growth = run_dir / f"outer_{cyc:02d}_growth"
        out_growth.mkdir(parents=True, exist_ok=True)

        res_growth = run_network_growth_uncoupled(
            bem_dir=run_dir,
            out_dir=out_growth,
            vertices=V_curr,
            connectivity=C_curr,
            params=grow,
            plot_hook=hook,
        )

        if COUPLING_METHOD == "iterative":
            tx_cr, ty_cr = compute_crack_boundary_tractions_direct(res=res_growth, boundary_mesh=boundary_mesh)
            solve_bem_with_extra_boundary_tractions(
                disk_params=disk,
                bem_dir=run_dir,
                tx_extra=-tx_cr,
                ty_extra=-ty_cr,
                gauss_n=ITERATIVE_UPDATE_CONFIG["gauss_n"],
                show=bool(ITERATIVE_UPDATE_CONFIG["show"]),
            )
            res_last = res_growth

        elif COUPLING_METHOD == "direct":
            grow_direct = CrackGrowthParams(
                material_E=GROWTH_PARAMS["material_E"],
                material_nu=GROWTH_PARAMS["material_nu"],
                plane_stress=GROWTH_PARAMS["plane_stress"],
                solver_kwargs=direct_kwargs,
                f0=GROWTH_PARAMS["f0"],
                f_fixed=GROWTH_PARAMS["f_fixed"],
                step_mode=GROWTH_PARAMS["step_mode"],
                simultaneous_tip_growth=GROWTH_PARAMS["simultaneous_tip_growth"],
                Kc_demo=GROWTH_PARAMS["Kc_demo"],
                rmax_frac=GROWTH_PARAMS["rmax_frac"],
                min_pts=GROWTH_PARAMS["min_pts"],
                max_cycles=0,
                vertex_high=GROWTH_PARAMS["vertex_high"],
                L_limit_mm=GROWTH_PARAMS["L_limit_mm"],
                simplification_config=GROWTH_PARAMS["simplification_config"],
            )

            out_direct = run_dir / f"outer_{cyc:02d}_direct"
            out_direct.mkdir(parents=True, exist_ok=True)

            Vg, Cg = _network_to_arrays(res_growth.calc.network)
            res_direct = run_network_growth_uncoupled(
                bem_dir=run_dir,
                out_dir=out_direct,
                vertices=Vg,
                connectivity=Cg,
                params=grow_direct,
                plot_hook=None,
            )

            Sxx_tot, Syy_tot, Sxy_tot = plot_total_field(
                bem_dir=run_dir,
                res=res_direct,
                out_dir=out_direct,
                tag=f"outer_{cyc:02d}_direct",
                params=plot_params,
                show=True,
            )
            _save_total_as_bem_arrays(run_dir, Sxx_tot, Syy_tot, Sxy_tot)
            res_last = res_direct

        else:
            # no_coupling
            res_last = res_growth

        V_curr, C_curr = _network_to_arrays(res_growth.calc.network)

    print("\n[done] outer-cycle run completed")
    print(f"[output] {run_dir}")


    # ---- next cell ----

    ## Least squares fits to K_I and K_II data in compression experiments,
    ## annotated with three propagation regimes and a crack-arrest indicator.

    import os
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt

    # Digitized data: y-position (mm) vs SIF (MPa sqrt(m))
    ki = np.array([
        [2.96997115830967, 2.26123418082751],
        [3.16265495749845, 2.16184830336284],
        [3.42439900461923, 2.06238632645141],
        [3.65855700229058, 1.96846438926389],
        [3.94864809332835, 1.97365437153272],
        [4.27987093533829, 1.93472189457183],
        [4.61185477181580, 2.00598141651510],
        [4.88824796243731, 2.02771541850891],
        [5.35812399643854, 2.06576514188742],
        [5.70391988250245, 2.13700944394134],
        [6.25674431346883, 2.18598704787416],
        [6.12349418219729, 2.89136803975435],
        [6.67647081205718, 2.96238404346800],
        [7.13124110587715, 2.81312258859878],
        [7.64285768642461, 2.89520345187091],
        [7.65784927743574, 3.06598583028301],
        [8.19564406766762, 2.93867145585852],
        [8.87224424878430, 2.91037768155426],
        [8.87722876254689, 3.63213527437656],
        [9.49998858508298, 3.80775757760241],
        [10.2042889648192, 3.79045256340986],
        [10.8784539636397, 3.40954439261227],
        [11.0784433097171, 2.36800170462761],
        [11.8003607113776, 0.901641465066549],
        [12.4098031307312, -2.85109621253053],
        [12.3917675618498, -3.46264658655932],
    ])

    kii = np.array([
        [3.32926606831747, 0.0832458032743578],
        [3.58270489895781, 0.0208719880749469],
        [3.92133615945356, 0.0387847263716013],
        [4.13354675992024, 0.130247872547848],
        [4.23163283918159, 0.0388539118074381],
        [4.82452057637757, 0.110335046197003],
        [5.26117502028391, 0.0279351921153762],
        [5.65586220777015, -0.00542162233557428],
        [6.12176322228022, 0.0593422351927445],
        [6.02276515318284, 0.0214160371840272],
        [6.61709949494632, 0.298025699244621],
        [7.58851018598302, 0.0440616882504263],
        [7.06567270257181, -0.0942934594604793],
        [7.57360386935274, -0.0696540099249649],
        [8.12435138653902, 0.0263439270911302],
        [8.15183687332146, -0.0762140471593089],
        [8.81513651544407, -0.0203247941733281],
        [8.81463334863798, -0.0916738472762952],
        [9.47860912115627, 0.0600906958167973],
        [9.47961545476844, 0.202788802022731],
        [10.1417986955400, 0.100372343436504],
        [10.8779789047316, 0.490726006805331],
        [11.6537677759398, 0.497587944123327],
        [12.3629027693043, -0.947069996792312],
        [12.4129993144352, -1.84337989722818],
    ])

    def polyfit_with_bounds(x, y, degree=3):
        coeff = np.polyfit(x, y, degree)
        model = np.poly1d(coeff)
        y_fit = model(x)
        residual = y - y_fit
        lower_shift = residual.min()
        upper_shift = residual.max()
        return model, lower_shift, upper_shift

    ki_x, ki_y = ki[:, 0], ki[:, 1]
    kii_x, kii_y = kii[:, 0], kii[:, 1]

    ki_model, ki_low, ki_up = polyfit_with_bounds(ki_x, ki_y, degree=3)
    kii_model, kii_low, kii_up = polyfit_with_bounds(kii_x, kii_y, degree=3)

    x_min = min(ki_x.min(), kii_x.min())
    x_max = max(ki_x.max(), kii_x.max())
    x_plot = np.linspace(x_min, x_max, 500)

    ki_fit = ki_model(x_plot)
    ki_lower = ki_fit + ki_low
    ki_upper = ki_fit + ki_up

    kii_fit = kii_model(x_plot)
    kii_lower = kii_fit + kii_low
    kii_upper = kii_fit + kii_up

    # Locate where the K_I least-squares fit first crosses zero (crack-arrest indicator)
    sign_change_idx = np.where(np.diff(np.sign(ki_fit)) < 0)[0]
    y_arrest = float(x_plot[sign_change_idx[-1]]) if sign_change_idx.size else 12.0

    # Regime boundaries (mm) along the loading axis
    y_a_b = 8.0    # tensile-dominated  -> geometric stress-concentration band
    y_b_c = 11.0   # geometric          -> compressive contact / arrest zone

    fig, ax = plt.subplots(figsize=(10, 6))

    # --- Regime backgrounds (drawn first so data sits on top)
    ax.axvspan(x_min, y_a_b, color="#cfe9ff", alpha=0.35, zorder=0,
               label="(a) tensile regime")
    ax.axvspan(y_a_b, y_b_c, color="#fff1c2", alpha=0.45, zorder=0,
               label="(b) geometric stress-concentration")
    ax.axvspan(y_b_c, x_max, color="#f9c8c2", alpha=0.45, zorder=0,
               label="(c) compressive / arrest")

    # --- K_I
    ax.scatter(ki_x, ki_y, s=35, color="tab:blue", alpha=0.9, zorder=3, label=r"$K_I$ data")
    ax.plot(x_plot, ki_fit, color="tab:blue", lw=2.2, zorder=4, label=r"$K_I$ least-squares fit")
    ax.plot(x_plot, ki_lower, color="tab:blue", lw=1.0, alpha=0.55, zorder=2)
    ax.plot(x_plot, ki_upper, color="tab:blue", lw=1.0, alpha=0.55, zorder=2)
    ax.fill_between(x_plot, ki_lower, ki_upper, color="lightgreen", alpha=0.18, zorder=1)

    # --- K_II
    ax.scatter(kii_x, kii_y, s=35, color="tab:orange", alpha=0.9, zorder=3, label=r"$K_{II}$ data")
    ax.plot(x_plot, kii_fit, color="tab:orange", lw=2.2, zorder=4, label=r"$K_{II}$ least-squares fit")
    ax.plot(x_plot, kii_lower, color="tab:orange", lw=1.0, alpha=0.55, zorder=2)
    ax.plot(x_plot, kii_upper, color="tab:orange", lw=1.0, alpha=0.55, zorder=2)
    ax.fill_between(x_plot, kii_lower, kii_upper, color="lightgreen", alpha=0.18, zorder=1)

    # --- Zero line and crack-arrest indicator
    ax.axhline(0.0, color="0.35", lw=0.8, zorder=2)
    ax.axvline(y_arrest, color="firebrick", lw=1.6, ls="--", zorder=4)
    y_top = max(ki_y.max(), kii_y.max()) + 0.5
    ax.annotate(
        "crack-arrest indicator\n(" + r"$K_I$ first crosses $0$)",
        xy=(y_arrest, 0.0),
        xytext=(y_arrest - 1.6, y_top - 0.5),
        arrowprops=dict(arrowstyle="->", color="firebrick", lw=1.2),
        color="firebrick", fontsize=14, ha="center",
    )

    # --- Regime labels at top of axes
    ymin, ymax = ax.get_ylim()
    y_label = ymax - 0.05 * (ymax - ymin)
    ax.text(0.5 * (x_min + y_a_b), y_label, "(a)", ha="center", va="top", fontsize=14, color="0.25")
    ax.text(0.5 * (y_a_b + y_b_c), y_label, "(b)", ha="center", va="top", fontsize=14, color="0.25")
    ax.text(0.5 * (y_b_c + x_max), y_label, "(c)", ha="center", va="top", fontsize=14, color="0.25")

    ax.set_xlabel("Crack-tip position along loading axis (mm)", fontsize=16)
    ax.set_ylabel(r"Stress intensity factor (SIF), MPa$\sqrt{\mathrm{m}}$", fontsize=16)
    ax.tick_params(axis="both", which="major", labelsize=14)
    ax.grid(alpha=0.25)
    ax.legend(ncol=2, frameon=True, fontsize=11, loc="lower left")

    plt.tight_layout()

    # Save the annotated figure to disk so it can be included in the manuscript
    out_path = Path(os.getcwd()).resolve() / "SIF_vs_position_regimes.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {out_path}  |  K_I sign change at y = {y_arrest:.2f} mm")

    plt.show()

    return locals()


### Case `function_diagnostics` (from `function_diagnostics.ipynb`)


In [ ]:
def run_function_diagnostics():
    """Case extracted from function_diagnostics.ipynb."""
    """
    Quick Parameter Options Finder
    ==============================

    Simple functions to find valid options for function parameters.
    Copy this into your Jupyter notebook!
    """
    import os, sys
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt

    # Add repo root (one level above notebooks/) to sys.path
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
        
    """
    Example Jupyter Notebook Cell - FunctionLocator Usage
    =====================================================

    Copy this into your Jupyter notebook to use FunctionLocator.
    """

    # ============================================================================
    # CELL 1: Import and Setup
    # ============================================================================

    from preamble import CrackNetworkV4
    from fracture_utils.Udiagnostics.function_locator import FunctionLocator, analyze

    # Import your solver
    from fracture_utils.Usolver.parametrization import DCENetworkStaticV4
    from fracture_utils.Usolver.material import Material, AppliedStress

    # ----------------------------
    # Build initial network
    # ----------------------------
    mm = 1e-3
    vertices = np.array([
        [0,   -5.0*mm,  -3.0*mm],
        [1,    5.0*mm,   3.0*mm],
    ], dtype=float)

    connectivity = np.array([[0, 1]], dtype=int)

    net = CrackNetworkV4.from_vertices_connectivity(
        vertices=vertices,
        connectivity=connectivity,
        Nv_max=4,
        validate=True
    )

    material = Material(E=200e9, nu=0.30, plane_stress=False)
    applied  = AppliedStress(sigma_xx=100e6, sigma_yy=0.0e6, sigma_xy=0.0)


    calc = DCENetworkStaticV4(material, net, applied)



    # ---- next cell ----


    # ============================================================================
    # CELL 2: Quick Analysis - Show All Parameters
    # ============================================================================

    # Create FunctionLocator instance
    loc = FunctionLocator(calc.solve)

    # Show all parameters with types and defaults
    loc.show_params()



    # ---- next cell ----


    # ============================================================================
    # CELL 3: Find Valid Options for Specific Parameters
    # ============================================================================

    # Find options for 'parametrization'
    loc.what_options('parametrization')

    # Find options for other parameters
    loc.what_options('representation')
    loc.what_options('crack_mode')
    loc.what_options('node_distribution')
    loc.what_options('solver_option')



    # ---- next cell ----


    # ============================================================================
    # CELL 4: Get Options Programmatically (No Print)
    # ============================================================================

    # Get options as a list without printing
    param_options = loc.what_options('parametrization', verbose=False)
    print(f"Available parametrizations: {param_options}")

    crack_modes = loc.what_options('crack_mode', verbose=False)
    print(f"Available crack modes: {crack_modes}")

    junction_models = loc.what_options('junction_model', verbose=False)
    print(f"Available junction models: {junction_models}")



    # ---- next cell ----


    # ============================================================================
    # CELL 5: Find All String Parameters and Their Options
    # ============================================================================

    # Automatically find all string parameters and their possible values
    all_options = loc.find_all_string_params()

    print("All string parameters with detected options:")
    print("=" * 60)
    for param, values in all_options.items():
        print(f"\n{param}:")
        for val in values:
            print(f"  • {val}")



    # ---- next cell ----


    # ============================================================================
    # CELL 6: Show Only String Parameters
    # ============================================================================

    # Filter to show only string parameters
    string_params = loc.show_params(filter_type='str')



    # ---- next cell ----


    # ============================================================================
    # CELL 7: View Source Code
    # ============================================================================

    # Show full source code
    loc.show_source()

    # Or show only first 30 lines
    # loc.show_source(lines=(1, 30))



    # ---- next cell ----


    # ============================================================================
    # CELL 8: Search in Source Code
    # ============================================================================

    # Search for specific keyword with context
    loc.search_source('parametrization', context=3)
    loc.search_source('crack_mode', context=2)



    # ---- next cell ----


    # ============================================================================
    # CELL 9: Quick Analysis Function
    # ============================================================================

    # Quick way to analyze any function
    analyze(calc.solve)  # Show all params

    # Or analyze specific parameter
    analyze(calc.solve, 'parametrization')



    # ---- next cell ----


    # ============================================================================
    # CELL 10: Find Classes in Your Package
    # ============================================================================

    # Find where a class is defined
    FunctionLocator.find_class('Material', 'fracture_utils')
    FunctionLocator.find_class('DCENetworkStaticV4', 'fracture_utils')
    FunctionLocator.find_class('CrackNetworkGenerator', 'fracture_utils')
    FunctionLocator.find_class('network_ops', 'fracture_utils')




    # ---- next cell ----


    # ============================================================================
    # CELL 11: Use with Different Functions
    # ============================================================================

    # You can reuse the same locator object
    loc.set_function(some_other_function)
    loc.show_params()

    # Or create a new one
    loc2 = FunctionLocator(Material.__init__)
    loc2.show_params()



    # ---- next cell ----


    # ============================================================================
    # CELL 12: Build a Configuration Dictionary
    # ============================================================================

    # Use found options to build valid configuration
    all_options = loc.find_all_string_params()

    # Create a config with first valid option for each parameter
    solver_config = {}
    for param, values in all_options.items():
        if values:
            solver_config[param] = values[0]  # Use first option

    print("Solver configuration with valid options:")
    print(solver_config)

    # Use it in your solve call
    # sol = calc.solve(
    #     ne_half=20,
    #     **solver_config  # Unpack the config
    # )



    # ---- next cell ----


    # ============================================================================
    # CELL 13: Save Options to File (Optional)
    # ============================================================================

    import json

    # Get all options
    all_options = loc.find_all_string_params()

    # Save to JSON
    with open('solver_options.json', 'w') as f:
        json.dump(all_options, f, indent=2)

    print("Saved options to: solver_options.json")

    # Load later
    with open('solver_options.json', 'r') as f:
        loaded_options = json.load(f)
        print(loaded_options)



    # ---- next cell ----


    # ============================================================================
    # CELL 14: Interactive Parameter Explorer
    # ============================================================================

    def explore_solver(solver_func):
        """Interactive exploration of solver parameters"""
        loc = FunctionLocator(solver_func)
        
        print("="*60)
        print("SOLVER PARAMETER EXPLORER")
        print("="*60)
        
        # Get all string parameters
        sig = loc.get_signature()
        string_params = [
            name for name, param in sig.parameters.items()
            if str(param.annotation).replace("'", "") == 'str'
            and name not in ['self', '_ignored']
        ]
        
        if not string_params:
            print("No string parameters found")
            return
        
        print(f"\nFound {len(string_params)} string parameter(s):")
        for i, param in enumerate(string_params, 1):
            print(f"{i}. {param}")
        
        print("\n" + "="*60)
        
        # Show options for each
        for param in string_params:
            options = loc.what_options(param, verbose=False)
            if options:
                print(f"\n{param}:")
                print(f"  Options: {', '.join(options)}")

    # Use it
    explore_solver(calc.solve)



    # ---- next cell ----


    # ============================================================================
    # CELL 15: Cheat Sheet
    # ============================================================================

    def cheat_sheet():
        """Print quick reference"""
        print("""
        FUNCTION LOCATOR - QUICK REFERENCE
        ==================================
        
        # Create locator
        loc = FunctionLocator(calc.solve)
        
        # Show all parameters
        loc.show_params()
        
        # Find valid options
        loc.what_options('parametrization')
        
        # Get options as list (no print)
        opts = loc.what_options('crack_mode', verbose=False)
        
        # Find all string params + options
        all_opts = loc.find_all_string_params()
        
        # View source code
        loc.show_source()
        loc.show_source(lines=(1, 30))
        
        # Search in source
        loc.search_source('parametrization')
        
        # Quick analysis
        analyze(calc.solve)
        analyze(calc.solve, 'parametrization')
        
        # Find class
        FunctionLocator.find_class('Material', 'fracture_utils')
        
        # Switch function
        loc.set_function(other_func)
        """)

    cheat_sheet()
    return locals()


### Case `graph_net` (from `graph_net.ipynb`)


In [ ]:
def run_graph_net():
    """# Crack Network Generator - Usage Guide

## Installation

Required packages:
```bash
pip install numpy networkx matplotlib scipy
```

## Quick Start

```python
from crack_network_generator import CrackNetworkGenerator
import matplotlib.pyplot as plt

mm = 1e-3

# Create generator
generator = CrackNetworkGenerator(
    domain_size=(20*mm, 20*mm),
    seed=42  # For reproducibility
)

# Generate network
generator.generate_network(
    n_junctions=5,
    n_tips=20,
    edge_length_mean=3*mm,
    edge_length_std=1*mm,
    allow_intersections=False,
    min_junction_angle_deg=45.0,
    min_edge_distance=1.0*mm
)

# Visualize
fig, ax = generator.visualize(node_size=80, edge_width=1.5)
plt.show()

# Export to arrays
vertices, connectivity = generator.to_arrays()
```

## All Parameters

### Basic Network Structure
- `n_junctions`: Number of junction vertices (degree ≥ 3)
- `n_tips`: Number of tip vertices (degree = 1)
- `max_junction_order`: Maximum junction degree (3-6 typical)

### Edge Properties
- `edge_length_mean`: Mean edge length (m)
- `edge_length_std`: Std deviation of edge length (m)
- `min_edge_length`: Minimum edge length (m)
- `max_edge_length`: Maximum edge length (m)

### Spatial Distribution
- `spatial_distribution`: 'uniform', 'clustered', or 'grid'
- `connection_strategy`: 'nearest_neighbor', 'delaunay', or 'random'

### Geometric Constraints
- `allow_intersections`: If False, prevents edge crossings (default: False)
- `min_junction_angle_deg`: Minimum angle between edges at junctions (degrees, default: 30)
- `min_edge_distance`: Minimum distance between parallel edges (m, default: 0)

### Connected Crack Paths (NEW!)
- `n_crack_paths`: Number of polyline cracks (default: 0)
- `segments_per_path_mean`: Average segments per path
- `segments_per_path_std`: Std deviation of segments per path
- `path_angle_change_mean_deg`: Systematic curvature (0° = straight)
- `path_angle_change_std_deg`: Random wiggle (10-45° typical)

## Example Use Cases

### 1. Junction Network with Constraints
```python
generator.generate_network(
    n_junctions=10,
    n_tips=30,
    edge_length_mean=4*mm,
    edge_length_std=1.5*mm,
    max_junction_order=4,
    
    # Prevent crowding
    allow_intersections=False,
    min_junction_angle_deg=40.0,
    min_edge_distance=1.0*mm,
    
    spatial_distribution='uniform',
    connection_strategy='nearest_neighbor'
)
```

### 2. Long Straight Propagating Cracks
```python
generator.generate_network(
    n_junctions=0,  # No junctions
    n_tips=0,       # No isolated tips
    
    edge_length_mean=2*mm,
    edge_length_std=0.5*mm,
    
    # Create long paths
    n_crack_paths=8,
    segments_per_path_mean=20,
    segments_per_path_std=5,
    path_angle_change_mean_deg=0.0,   # Straight
    path_angle_change_std_deg=10.0,   # Small wiggle
    
    allow_intersections=False,
    min_edge_distance=1.5*mm
)
```

### 3. Curved/Tortuous Cracks
```python
generator.generate_network(
    n_junctions=0,
    n_tips=0,
    
    edge_length_mean=1.5*mm,
    edge_length_std=0.3*mm,
    
    n_crack_paths=10,
    segments_per_path_mean=15,
    segments_per_path_std=5,
    path_angle_change_mean_deg=5.0,   # Slight curve
    path_angle_change_std_deg=25.0,   # High variation
    
    allow_intersections=True,  # Allow coalescence
    spatial_distribution='clustered'
)
```

### 4. Mixed Network (Junctions + Long Cracks)
```python
generator.generate_network(
    # Junction network
    n_junctions=5,
    n_tips=20,
    max_junction_order=4,
    
    edge_length_mean=3*mm,
    edge_length_std=1*mm,
    
    # Add long cracks
    n_crack_paths=8,
    segments_per_path_mean=10,
    segments_per_path_std=3,
    path_angle_change_mean_deg=0.0,
    path_angle_change_std_deg=20.0,
    
    allow_intersections=False,
    min_junction_angle_deg=45.0,
    min_edge_distance=0.8*mm
)
```

## Visualization Options

### Standard Visualization
```python
fig, ax = generator.visualize(
    figsize=(12, 10),
    show_labels=False,        # Hide vertex IDs
    show_edge_labels=False,   # Hide edge lengths
    node_size=100,            # Circle size
    edge_width=1.5,           # Line thickness
    edge_color='black'        # Line color
)
plt.show()
```

### Path Coloring (Shows Connected Components)
```python
fig, ax = generator.visualize_with_paths(figsize=(14, 14))
plt.show()
```

## Export to Arrays

```python
# Get arrays in your format
vertices, connectivity = generator.to_arrays()

# vertices: (n_vertices, 3) array with [id, x, y]
# connectivity: (n_edges, 3) array with [edge_id, v0, v1]

# Use with your CrackNetworkV4
from your_module import CrackNetworkV4

net = CrackNetworkV4.from_vertices_connectivity(
    vertices=vertices,
    connectivity=connectivity,
    Nv_max=3,
    degrees="auto",
    validate=True
)
```

## Statistics

```python
generator.print_statistics()
# Prints:
# - Vertex counts by type (tips, kinks, junctions)
# - Edge statistics (count, length distribution)
# - Degree distribution
# - Connected components
```

## Typical Parameter Ranges

| Parameter | Conservative | Moderate | Aggressive |
|-----------|-------------|----------|------------|
| `min_junction_angle_deg` | 60° | 40° | 30° |
| `min_edge_distance` | 2mm | 1mm | 0.5mm |
| `segments_per_path_mean` | 5 | 10 | 20 |
| `path_angle_change_std_deg` | 10° | 25° | 45° |

## Tips

1. **Start simple**: Begin with just junctions/tips, then add paths
2. **Constraints trade-off**: Stricter constraints may result in fewer edges
3. **Seed for reproducibility**: Always set a seed if you need to regenerate
4. **Check statistics**: Use `print_statistics()` to verify network properties
5. **Path angles**: 
   - Straight: `mean=0°, std=10°`
   - Curved: `mean=5°, std=20°`
   - Jagged: `mean=0°, std=40°`

## Troubleshooting

**Problem**: Not enough edges created
- **Solution**: Relax constraints (lower min_angle, lower min_distance)
- Or increase `max_angle_attempts` in the code

**Problem**: Edges too close together
- **Solution**: Increase `min_edge_distance`

**Problem**: Sharp angles at junctions
- **Solution**: Increase `min_junction_angle_deg`

**Problem**: Paths terminate early
- **Solution**: Increase domain size or reduce segment length"""
    import sys
    import os
    import matplotlib.pyplot as plt

    # Get the path to fracture_utils
    notebook_dir = os.getcwd()  # Current notebook directory
    fracture_utils_path = os.path.join(notebook_dir, '..', 'fracture_utils')
    sys.path.insert(0, os.path.abspath(fracture_utils_path))

    # Now import from Ugenerator
    from Ugenerator import CrackNetworkGenerator as gen, NetworkAnalyzer, NetworkVisualizer


    # ============================================================================
    # USAGE EXAMPLE
    # ============================================================================

    # Create generator
    mm = 1e-3
    generator = gen(
        domain_size=(100*mm, 100*mm),
        seed=42  # For reproducibility
    )

    # Generate network
    generator.generate_network(
        n_junctions=0,           # Number of junction vertices
        n_tips=20,                # Number of tip vertices
        edge_length_mean=40*mm,   # Mean edge length
        edge_length_std=5*mm,    # Std dev of edge length
        max_junction_order=3,    # Max junction degree (3-4 typical)
        spatial_distribution='uniform',  # 'uniform', 'clustered', 'grid'
        connection_strategy='nearest_neighbor',  # 'nearest_neighbor', 'delaunay', 'random'
        min_edge_length=5*mm,
        max_edge_length=20*mm
    )

    # Print statistics
    generator.print_statistics()

    # Visualize
    fig, ax = generator.visualize(
        figsize=(12, 10),
        show_labels=False,
        show_edge_labels=False,
        node_size=10,
        edge_color='blue',
        edge_width=2.0
    )
    plt.show()

    # Export to your format
       
        # Export arrays for further analysis
    vertices, connectivity, vertex_types = generator.to_arrays()
    print(f"\nExported arrays:")
    print(f"  Vertices shape: {vertices.shape}")
    print(f"  Connectivity shape: {connectivity.shape}")
    print(f"  Vertex types shape: {vertex_types.shape}")
    print(f"  Vertex types: {set(vertex_types)}")

    # ---- next cell ----

    import sys
    import os
    import matplotlib.pyplot as plt

    # Get the path to fracture_utils
    notebook_dir = os.getcwd()  # Current notebook directory
    fracture_utils_path = os.path.join(notebook_dir, '..', 'fracture_utils')
    sys.path.insert(0, os.path.abspath(fracture_utils_path))

    # Now import from Ugenerator
    from Ugenerator import CrackNetworkGenerator, NetworkAnalyzer, NetworkVisualizer

    # ============================================================================
    # USAGE EXAMPLE
    # ============================================================================

    if __name__ == "__main__":
        # Create generator
        mm = 1e-3
        generator = CrackNetworkGenerator(
            domain_size=(20*mm, 20*mm),
            seed=42  # For reproducibility
        )
        
        # Generate network WITHOUT intersections
        generator.generate_network(
            n_junctions=10,           # Number of junction vertices
            n_tips=100,               # Number of tip vertices
            edge_length_mean=5*mm,   # Mean edge length
            edge_length_std=2*mm,    # Std dev of edge length
            max_junction_order=3,    # Max junction degree (3-4 typical)
            spatial_distribution='uniform',  # 'uniform', 'clustered', 'grid'
            connection_strategy='nearest_neighbor',  # 'nearest_neighbor', 'delaunay', 'random'
            min_edge_length=1*mm,
            max_edge_length=3*mm,
            allow_intersections=False  # ← Prevent edge crossings
        )
        
        # Print statistics
        generator.print_statistics()
        
        # Visualize
        fig, ax = generator.visualize(
            figsize=(12, 10),
            show_labels=False,
            show_edge_labels=False,
            node_size=40,
            edge_width=1.5,
            edge_color='black'
        )
        plt.show()
        
        # Export arrays for further analysis
    vertices, connectivity, vertex_types = generator.to_arrays()
    print(f"\nExported arrays:")
    print(f"  Vertices shape: {vertices.shape}")
    print(f"  Connectivity shape: {connectivity.shape}")
    print(f"  Vertex types shape: {vertex_types.shape}")
    print(f"  Vertex types: {set(vertex_types)}")

    # ---- next cell ----

    import sys
    import os
    import matplotlib.pyplot as plt

    # Get the path to fracture_utils
    notebook_dir = os.getcwd()  # Current notebook directory
    fracture_utils_path = os.path.join(notebook_dir, '..', 'fracture_utils')
    sys.path.insert(0, os.path.abspath(fracture_utils_path))

    # Now import from Ugenerator
    from Ugenerator import CrackNetworkGenerator, NetworkAnalyzer, NetworkVisualizer
    mm = 1e-3

    generator = CrackNetworkGenerator(
        domain_size=(20*mm, 20*mm),
        seed=42
    )

    generator.generate_network(
        n_junctions=0,              # No junctions
        n_tips=0,                   # No isolated tips
        edge_length_mean=2*mm,
        edge_length_std=0.5*mm,
        min_edge_length=1*mm,
        max_edge_length=3*mm,
        
        # Connected crack paths
        n_crack_paths=5,                    # 5 long cracks
        segments_per_path_mean=10,          # Average 10 segments each
        segments_per_path_std=3,            # ±3 segments variation
        path_angle_change_mean_deg=0.0,     # Straight (no systematic curvature)
        path_angle_change_std_deg=10.0,     # Small random wiggles (±10°)
        
        allow_intersections=True,
        detect_intersections=True,
        min_edge_distance=1.0*mm
    )

    fig, ax = generator.visualize(node_size=60, edge_width=2.0)
    plt.show()

    # ---- next cell ----

    import numpy as np
    import matplotlib.pyplot as plt
    from pathlib import Path
    import os, sys, importlib

    # Ensure project root is on path (so "utils" is importable)
    nb_dir = os.path.abspath(os.getcwd())
    root   = os.path.abspath(os.path.join(nb_dir, ".."))
    if root not in sys.path:
        sys.path.insert(0, root)

    import graph_utils.crack_network_generator  as gen
    importlib.reload(gen)

    generator = CrackNetworkGenerator(
        domain_size=(20*mm, 20*mm),
        seed=123
    )

    generator.generate_network(
        n_junctions=0,
        n_tips=0,
        edge_length_mean=1.5*mm,
        edge_length_std=0.3*mm,
        min_edge_length=1*mm,
        max_edge_length=2*mm,
        
        # Curved paths
        n_crack_paths=8,
        segments_per_path_mean=15,          # Longer paths
        segments_per_path_std=5,
        path_angle_change_mean_deg=5.0,     # Slight systematic curve
        path_angle_change_std_deg=20.0,     # Moderate wiggle
        
        allow_intersections=False,
        min_edge_distance=0.5*mm
    )

    fig, ax = generator.visualize(node_size=40, edge_width=1.5)
    plt.show()

    # ---- next cell ----

    import sys
    import os
    import matplotlib.pyplot as plt

    # Get the path to fracture_utils
    notebook_dir = os.getcwd()  # Current notebook directory
    fracture_utils_path = os.path.join(notebook_dir, '..', 'fracture_utils')
    sys.path.insert(0, os.path.abspath(fracture_utils_path))

    # Now import from Ugenerator
    from Ugenerator import CrackNetworkGenerator, NetworkAnalyzer, NetworkVisualizer

    generator = CrackNetworkGenerator(
        domain_size=(30*mm, 30*mm),
        seed=456
    )

    generator.generate_network(
        # Junction network
        n_junctions=5,
        n_tips=20,
        max_junction_order=4,
        
        # Edge properties
        edge_length_mean=3*mm,
        edge_length_std=1*mm,
        min_edge_length=1.5*mm,
        max_edge_length=5*mm,
        
        # Add long connected cracks
        n_crack_paths=10,                   # 10 polyline cracks
        segments_per_path_mean=8,           # 8 segments each on average
        segments_per_path_std=3,
        path_angle_change_mean_deg=0.0,     # No systematic curvature
        path_angle_change_std_deg=25.0,     # Random direction changes
        
        # Constraints
        allow_intersections=False,
        min_junction_angle_deg=40.0,
        min_edge_distance=1.0*mm,
        
        spatial_distribution='uniform',
        connection_strategy='nearest_neighbor'
    )

    fig, ax = generator.visualize(
        figsize=(14, 14),
        node_size=80,
        edge_width=2.0,
        show_labels=False
    )
    plt.show()

    # ---- next cell ----

    import sys
    import os
    import matplotlib.pyplot as plt

    # Get the path to fracture_utils
    notebook_dir = os.getcwd()  # Current notebook directory
    fracture_utils_path = os.path.join(notebook_dir, '..', 'fracture_utils')
    sys.path.insert(0, os.path.abspath(fracture_utils_path))

    # Now import from Ugenerator
    from fracture_utils.Ugenerator import CrackNetworkGenerator, NetworkAnalyzer, NetworkVisualizer
    # Simulate fatigue cracks with gradual growth
    generator = CrackNetworkGenerator(
        domain_size=(25*mm, 25*mm),
        seed=789
    )

    generator.generate_network(
        n_junctions=0,
        n_tips=0,
        
        # Variable segment lengths (crack slows down)
        edge_length_mean=2*mm,
        edge_length_std=1*mm,
        min_edge_length=0.5*mm,
        max_edge_length=4*mm,
        
        # Long propagating cracks
        n_crack_paths=12,
        segments_per_path_mean=20,          # Very long cracks
        segments_per_path_std=8,
        path_angle_change_mean_deg=2.0,     # Slight systematic turn
        path_angle_change_std_deg=15.0,     # Moderate randomness
        
        allow_intersections=False,           # Allow crack coalescence
        min_edge_distance=0.0,              # Allow proximity
        
        spatial_distribution='clustered'    # Clustered nucleation
    )

    fig, ax = generator.visualize(
        figsize=(12, 12),
        node_size=50,
        edge_width=1.5,
        show_labels=False,
        edge_color='darkred'
    )
    ax.set_title('Simulated Fatigue Crack Network', fontsize=16, weight='bold')
    plt.show()

    # ---- next cell ----

    import sys
    import os
    import matplotlib.pyplot as plt
    import numpy as np
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
        
    # Get the path to fracture_utils
    notebook_dir = os.getcwd()  # Current notebook directory
    fracture_utils_path = os.path.join(notebook_dir, '..', 'fracture_utils')
    sys.path.insert(0, os.path.abspath(fracture_utils_path))

    # Import from Ugenerator
    from fracture_utils.Ugenerator import CrackNetworkGenerator, NetworkAnalyzer, NetworkVisualizer

    # Import simplifier
    from fracture_utils.Ugenerator.crack_network_simplifier import CrackNetworkSimplifier, SimplificationConfig
    out_dir = Path(repo_root) / "output" / "NetworkSimplification"
    out_dir.mkdir(parents=True, exist_ok=True)

    mm = 1e-3

    # ============================================================================
    # STEP 1: Generate Complex Network
    # ============================================================================

    print("="*70)
    print("GENERATING COMPLEX FATIGUE CRACK NETWORK")
    print("="*70)

    generator = CrackNetworkGenerator(
        domain_size=(25*mm, 25*mm),
        seed=789
    )

    generator.generate_network(
        n_junctions=0,
        n_tips=0,
        
        # Variable segment lengths (crack slows down)
        edge_length_mean=2*mm,
        edge_length_std=1*mm,
        min_edge_length=0.1*mm,
        max_edge_length=4*mm,
        
        # Long propagating cracks
        n_crack_paths=12,
        segments_per_path_mean=20,          # Very long cracks
        segments_per_path_std=8,
        path_angle_change_mean_deg=2.0,     # Slight systematic turn
        path_angle_change_std_deg=15.0,     # Moderate randomness
        
        allow_intersections=False,          # Prevent coalescence for now
        min_edge_distance=0.0,              # Allow proximity
        
        spatial_distribution='clustered'    # Clustered nucleation
    )

    # Show statistics
    generator.print_statistics()

    # ============================================================================
    # STEP 2: Export Original Network
    # ============================================================================

    print("\n" + "="*70)
    print("EXPORTING NETWORK DATA")
    print("="*70)

    # Export to arrays - get all 3 values
    original_vertices, original_connectivity, vertex_types = generator.to_arrays()

    print(f"\n✓ Exported network:")
    print(f"  Vertices shape: {original_vertices.shape}")
    print(f"  Connectivity shape: {original_connectivity.shape}")
    print(f"  Vertices: {len(original_vertices)}")
    print(f"  Edges: {len(original_connectivity)}")

    # ============================================================================
    # STEP 3: Configure Simplification
    # ============================================================================

    print("\n" + "="*70)
    print("CONFIGURING SIMPLIFICATION")
    print("="*70)

    # Create configuration
    config = SimplificationConfig(
        max_angle_deviation=10.0,      # Merge edges if angle deviation < 10°
        min_edge_length=0.3*mm,       # Remove edges shorter than 0.3mm
        merge_at_degree2=True,        # Merge colinear edges at internal nodes
        remove_degree2_nodes=False,    # Remove degree-2 internal nodes
        preserve_tips=True,           # Don't remove crack tips
        preserve_junctions=True,      # Don't modify junctions
        max_merged_edge_length=20*mm, # Don't merge if result > 20mm
        merge_vertex_tolerance=1e-6   # Merge vertices closer than this
    )

    print(f"\nSimplification settings:")
    print(f"  Max angle deviation: {config.max_angle_deviation}°")
    print(f"  Min edge length: {config.min_edge_length/mm:.2f} mm")
    print(f"  Merge at degree-2 nodes: {config.merge_at_degree2}")
    print(f"  Remove degree-2 nodes: {config.remove_degree2_nodes}")

    # ============================================================================
    # STEP 4: Simplify Network
    # ============================================================================

    print("\n" + "="*70)
    print("SIMPLIFYING NETWORK")
    print("="*70)

    # Create simplifier
    simplifier = CrackNetworkSimplifier(config=config)

    # Load original network
    # Note: original_connectivity might have shape (n, 3) with [edge_id, v0, v1]
    # If so, we need to extract just [v0, v1]
    if original_connectivity.shape[1] == 3:
        # Has edge_id column - extract just v0, v1
        connectivity_for_simplifier = original_connectivity[:, 1:3]
    else:
        # Already in [v0, v1] format
        connectivity_for_simplifier = original_connectivity

    simplifier.load_from_arrays(
        original_vertices, 
        connectivity_for_simplifier,
        domain_size=(25*mm, 25*mm)
    )

    # Get statistics before simplification
    stats_before = simplifier.get_statistics()
    print(f"\nBefore simplification:")
    print(f"  Vertices: {stats_before['n_vertices']}")
    print(f"  Edges: {stats_before['n_edges']}")
    print(f"  Tips: {stats_before['n_tips']}")
    print(f"  Junctions: {stats_before['n_junctions']}")
    print(f"  Internal nodes: {stats_before['n_internal']}")

    # Simplify!
    simplification_stats = simplifier.simplify(verbose=True)

    # Get statistics after simplification
    stats_after = simplifier.get_statistics()

    # Export simplified network
    simplified_vertices, simplified_connectivity = simplifier.to_arrays()

    print(f"\n✓ Simplification complete!")
    print(f"  Original: {len(original_vertices)} vertices, {len(connectivity_for_simplifier)} edges")
    print(f"  Simplified: {len(simplified_vertices)} vertices, {len(simplified_connectivity)} edges")
    print(f"  Reduction: {len(original_vertices) - len(simplified_vertices)} vertices "
          f"({100*(len(original_vertices) - len(simplified_vertices))/len(original_vertices):.1f}%), "
          f"{len(connectivity_for_simplifier) - len(simplified_connectivity)} edges "
          f"({100*(len(connectivity_for_simplifier) - len(simplified_connectivity))/len(connectivity_for_simplifier):.1f}%)")

    # ============================================================================
    # STEP 5: Visualize Comparison
    # ============================================================================

    print("\n" + "="*70)
    print("CREATING VISUALIZATION")
    print("="*70)

    fig = plt.figure(figsize=(18, 8))

    # Left: Original network
    ax1 = plt.subplot(1, 2, 1)
    ax1.set_title('Original Network\n(Before Simplification)', 
                  fontsize=14, fontweight='bold')

    # Plot edges - handle both formats
    for row in connectivity_for_simplifier:
        v0, v1 = int(row[0]), int(row[1])
        pos0 = original_vertices[v0, 1:]
        pos1 = original_vertices[v1, 1:]
        ax1.plot([pos0[0]/mm, pos1[0]/mm], 
                [pos0[1]/mm, pos1[1]/mm], 
                'darkred', linewidth=1.5, alpha=0.7)

    # Plot vertices
    ax1.scatter(original_vertices[:, 1]/mm, original_vertices[:, 2]/mm, 
               c='red', s=30, zorder=5, alpha=0.6, edgecolors='darkred', linewidths=0.5)

    ax1.set_xlabel('x [mm]', fontsize=11)
    ax1.set_ylabel('y [mm]', fontsize=11)
    ax1.grid(True, alpha=0.3, linestyle='--')
    ax1.axis('equal')

    # Add info box
    info_text = (f'Vertices: {len(original_vertices)}\n'
                 f'Edges: {len(connectivity_for_simplifier)}\n'
                 f'Tips: {stats_before["n_tips"]}\n'
                 f'Junctions: {stats_before["n_junctions"]}\n'
                 f'Internal: {stats_before["n_internal"]}')
    ax1.text(0.02, 0.98, info_text,
            transform=ax1.transAxes, verticalalignment='top',
            fontsize=10, family='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    # Right: Simplified network
    ax2 = plt.subplot(1, 2, 2)
    ax2.set_title('Simplified Network\n(After Simplification)', 
                  fontsize=14, fontweight='bold')

    # Plot edges
    for v0, v1 in simplified_connectivity:
        idx_v0 = int(v0)
        idx_v1 = int(v1)
        pos0 = simplified_vertices[idx_v0, 1:]
        pos1 = simplified_vertices[idx_v1, 1:]
        ax2.plot([pos0[0]/mm, pos1[0]/mm], 
                [pos0[1]/mm, pos1[1]/mm], 
                'darkgreen', linewidth=2, alpha=0.8)

    # Plot vertices
    ax2.scatter(simplified_vertices[:, 1]/mm, simplified_vertices[:, 2]/mm, 
               c='green', s=40, zorder=5, edgecolors='darkgreen', linewidths=0.5)

    ax2.set_xlabel('x [mm]', fontsize=11)
    ax2.set_ylabel('y [mm]', fontsize=11)
    ax2.grid(True, alpha=0.3, linestyle='--')
    ax2.axis('equal')

    # Add info box
    reduction_v = len(original_vertices) - len(simplified_vertices)
    reduction_e = len(connectivity_for_simplifier) - len(simplified_connectivity)
    info_text = (f'Vertices: {len(simplified_vertices)}\n'
                 f'Edges: {len(simplified_connectivity)}\n'
                 f'Tips: {stats_after["n_tips"]}\n'
                 f'Junctions: {stats_after["n_junctions"]}\n'
                 f'Internal: {stats_after["n_internal"]}\n\n'
                 f'Reduced by:\n'
                 f'  {reduction_v} vertices ({100*reduction_v/len(original_vertices):.1f}%)\n'
                 f'  {reduction_e} edges ({100*reduction_e/len(connectivity_for_simplifier):.1f}%)')
    ax2.text(0.02, 0.98, info_text,
            transform=ax2.transAxes, verticalalignment='top',
            fontsize=10, family='monospace',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

    plt.tight_layout()
    plt.savefig(out_dir / 'network_simplification_fatigue_cracks.png', dpi=150, bbox_inches='tight')
    print(f"\n✓ Saved comparison plot: {out_dir / 'network_simplification_fatigue_cracks.png'}")
    plt.show()

    # ============================================================================
    # STEP 6: Detailed Comparison Stats
    # ============================================================================

    print("\n" + "="*70)
    print("DETAILED STATISTICS")
    print("="*70)

    print("\nSimplification Summary:")
    print(f"  Vertices merged: {simplification_stats['vertices_merged']}")
    print(f"  Edges removed (too small): {simplification_stats['edges_removed']}")
    print(f"  Edge pairs merged (colinear): {simplification_stats['edges_merged']}")
    print(f"  Degree-2 nodes removed: {simplification_stats['degree2_removed']}")

    print("\nEdge Length Statistics:")

    # Calculate edge lengths for both networks
    def edge_lengths(vertices, connectivity):
        lengths = []
        for row in connectivity:
            v0, v1 = int(row[0]), int(row[1])
            pos0 = vertices[v0, 1:]
            pos1 = vertices[v1, 1:]
            length = np.linalg.norm(pos1 - pos0)
            lengths.append(length)
        return np.array(lengths)

    original_lengths = edge_lengths(original_vertices, connectivity_for_simplifier)
    simplified_lengths = edge_lengths(simplified_vertices, simplified_connectivity)

    print(f"\nOriginal network:")
    print(f"  Mean edge length: {np.mean(original_lengths)/mm:.3f} mm")
    print(f"  Min edge length: {np.min(original_lengths)/mm:.3f} mm")
    print(f"  Max edge length: {np.max(original_lengths)/mm:.3f} mm")
    print(f"  Total crack length: {np.sum(original_lengths)/mm:.2f} mm")

    print(f"\nSimplified network:")
    print(f"  Mean edge length: {np.mean(simplified_lengths)/mm:.3f} mm")
    print(f"  Min edge length: {np.min(simplified_lengths)/mm:.3f} mm")
    print(f"  Max edge length: {np.max(simplified_lengths)/mm:.3f} mm")
    print(f"  Total crack length: {np.sum(simplified_lengths)/mm:.2f} mm")

    print("\n" + "="*70)
    print("SIMPLIFICATION COMPLETE!")
    print("="*70)

    # ---- next cell ----

    import sys
    import os
    import matplotlib.pyplot as plt

    # Get the path to fracture_utils
    notebook_dir = os.getcwd()  # Current notebook directory
    fracture_utils_path = os.path.join(notebook_dir, '..', 'fracture_utils')
    sys.path.insert(0, os.path.abspath(fracture_utils_path))

    # Now import from Ugenerator
    from Ugenerator import CrackNetworkGenerator, NetworkAnalyzer, NetworkVisualizer

    mm=1e-3
    # Generate exactly 3 non-intersecting meandering cracks, each with 3 edges
    generator = CrackNetworkGenerator(
        domain_size=(25*mm, 25*mm),
        seed=4  # Change seed for different patterns
    )

    generator.generate_network(
        n_junctions=0,
        n_tips=0,
        
        # Edge properties for meandering
        edge_length_mean=3*mm,
        edge_length_std=0.5*mm,
        min_edge_length=2*mm,
        max_edge_length=6*mm,
        
        # Exactly 3 cracks, each with 3 segments
        n_crack_paths=3,
        segments_per_path_mean=3,
        segments_per_path_std=1,  # Some variation in segments per path
        
        # Small random angles for meandering
        path_angle_change_mean_deg=0.0,     # No systematic turn
        path_angle_change_std_deg=30.0,     # Small random angles (±30°)
        
        # Non-intersecting
        allow_intersections=False,
        min_edge_distance=1.0*mm,  # Keep cracks separated
        
        spatial_distribution='uniform'
    )

    # Print statistics
    generator.print_statistics()

    # Visualize with path coloring to see the 3 separate cracks
    fig, ax = generator.visualize_with_paths(figsize=(12, 12))
    ax.set_title('Three Meandering Cracks (3 edges each, non-intersecting)', 
                 fontsize=16, weight='bold')
    plt.show()

    # Also show standard visualization
    fig2, ax2 = generator.visualize(
        figsize=(12, 12),
        node_size=100,
        edge_width=2.0,
        show_labels=True,
        show_edge_labels=True,
        edge_color='black'
    )
    plt.show()

    # Export arrays for further analysis
    vertices, connectivity, vertex_types = generator.to_arrays()
    print(f"\nExported arrays:")
    print(f"  Vertices shape: {vertices.shape}")
    print(f"  Connectivity shape: {connectivity.shape}")
    print(f"  Vertex types shape: {vertex_types.shape}")
    print(f"  Vertex types: {set(vertex_types)}")

    # ---- next cell ----


    import sys
    import os
    import matplotlib.pyplot as plt

    # Get the path to fracture_utils
    notebook_dir = os.getcwd()  # Current notebook directory
    fracture_utils_path = os.path.join(notebook_dir, '..', 'fracture_utils')
    sys.path.insert(0, os.path.abspath(fracture_utils_path))

    # Now import from Ugenerator
    from Ugenerator import CrackNetworkGenerator, NetworkAnalyzer, NetworkVisualizer

    # ----------------------------
    # Problem definition
    # ----------------------------
    mm = 1e-3
    generator = CrackNetworkGenerator(
        domain_size=(25*mm, 25*mm),
        seed=7  # Change seed for different patterns
    )

    generator.generate_network(
        n_junctions=3,
        n_tips=6,
        
        # Edge properties for meandering
        edge_length_mean=5*mm,
        edge_length_std=2*mm,
        min_edge_length=2*mm,
        max_edge_length=10*mm,

        # Exactly 4 cracks, each with 4 segments
        n_crack_paths=2,
        segments_per_path_mean=4,
        segments_per_path_std=2,  # Some variation in segments per path
        
        # Small random angles for meandering
        path_angle_change_mean_deg=0.0,     # No systematic turn
        path_angle_change_std_deg=10.0,     # Small random angles (±10°)

        # Non-intersecting
        allow_intersections=True,
        detect_intersections=True,
        min_edge_distance=1.0*mm,  # Keep cracks separated
        
        spatial_distribution='uniform'
    )


    # Print statistics
    generator.print_statistics()

    fig2, ax2 = generator.visualize(
        figsize=(12, 12),
        node_size=20,
        edge_width=2.0,
        show_labels=True,
        show_edge_labels=True,
        edge_color='black'
    )
    plt.show()
    vertices, connectivity, vertex_types = generator.to_arrays()




    # ---- next cell ----

    import sys
    import os
    import matplotlib.pyplot as plt

    # Get the path to fracture_utils
    notebook_dir = os.getcwd()  # Current notebook directory
    fracture_utils_path = os.path.join(notebook_dir, '..', 'fracture_utils')
    sys.path.insert(0, os.path.abspath(fracture_utils_path))

    # Now import from Ugenerator
    from Ugenerator import CrackNetworkGenerator, NetworkAnalyzer, NetworkVisualizer

    mm = 1e-3
    generator = CrackNetworkGenerator(domain_size=(25*mm, 25*mm), seed=42)

    generator.generate_network(
        n_junctions=5,
        n_tips=20,
        allow_intersections=True,
        detect_intersections=True,
    )

    # Use modular components
    analyzer = NetworkAnalyzer(generator.G)
    analyzer.print_statistics()

    visualizer = NetworkVisualizer(generator.G, (25*mm, 25*mm))
    fig, ax = visualizer.visualize()
    plt.show()

    # ---- next cell ----

    """
    Example: Using the Boundary Module

    This demonstrates how to:
    1. Create boundaries (circle, rectangle, polygon)
    2. Generate cracks within boundaries
    3. Detect and trim crack-boundary intersections
    4. Visualize bounded crack networks
    """
    from pathlib import Path
    import os, sys

    # --- Ensure repo root on sys.path
    nb_dir = os.path.abspath(os.getcwd())
    root   = os.path.abspath(os.path.join(nb_dir, ".."))
    if root not in sys.path:
        sys.path.insert(0, root)

    # --- Preamble re-exports
    enable_autoreload()


    # import sys
    # import os
    # import matplotlib.pyplot as plt
    # import numpy as np

    # # Get the path to fracture_utils
    # notebook_dir = os.getcwd()  # Current notebook directory
    # fracture_utils_path = os.path.join(notebook_dir, '..', 'fracture_utils')
    # sys.path.insert(0, os.path.abspath(fracture_utils_path))

    # # Now import from Ugenerator
    # from Ugenerator import CrackNetworkGenerator, NetworkAnalyzer,                                   NetworkVisualizer, NetworkVisualizer
    # from Ugenerator.boundary import (Boundary, BoundaryManager)

    out_dir = Path(r"/Users/ghoni/Documents/GitHub/Fracture/VCM_1_3/output/BoundaryExamples")
    out_dir.mkdir(parents=True, exist_ok=True)




    mm = 1e-3

    print("="*60)
    print("Boundary Module Examples")
    print("="*60)

    # =============================================================================
    # Example 1: Circular Boundary (Like your BEM disk)
    # =============================================================================
    print("\n1. Circular Boundary Example")
    print("-"*60)

    # Create circular boundary
    radius = 10*mm
    boundary_circle = Boundary.create_circle(center=np.array([0, 0]), radius=radius)

    # Generate crack network (some cracks will extend outside)
    generator = CrackNetworkGenerator(
        domain_size=(25*mm, 25*mm),  # Larger than boundary
        seed=42
    )

    generator.generate_network(
        n_junctions=3,
        n_tips=10,
        n_crack_paths=5,
        segments_per_path_mean=4,
        edge_length_mean=4*mm,
        edge_length_std=1*mm,
        path_angle_change_std_deg=25.0,
        allow_intersections=True,
        detect_intersections=False,
        spatial_distribution='uniform'
    )

    print(f"Generated {generator.G.number_of_nodes()} nodes, {generator.G.number_of_edges()} edges")

    # Trim cracks at boundary
    boundary_manager = BoundaryManager(boundary_circle)
    generator.G = boundary_manager.trim_cracks_at_boundary(generator.G, verbose=True)

    # Add boundary to graph for visualization
    generator.G = boundary_manager.add_boundary_to_graph(generator.G, n_points_per_segment=100)

    # Visualize
    visualizer = NetworkVisualizer(generator.G, (25*mm, 25*mm))

    fig, ax = visualizer.visualize(
        figsize=(10, 10),
        show_labels=False,
        show_edge_labels=False,
        node_size=60,
        edge_width=1.5
    )
    ax.set_title('Crack Network in Circular Boundary', fontsize=14, weight='bold')

    # Highlight boundary edges in red
    pos = {n: generator.G.nodes[n]['pos'] for n in generator.G.nodes()}
    pos_mm = {k: (v[0]*1e3, v[1]*1e3) for k, v in pos.items()}

    boundary_edges = [(u, v) for u, v, d in generator.G.edges(data=True) 
                      if d.get('type') == 'boundary']
    import networkx as nx
    nx.draw_networkx_edges(generator.G, pos_mm, edgelist=boundary_edges,
                           edge_color='red', width=2.0, ax=ax)

    plt.savefig(str(out_dir) + "/example_circular_boundary.png", dpi=150, bbox_inches="tight")
    print("Saved: example_circular_boundary.png")
    plt.close()

    # =============================================================================
    # Example 2: Rectangular Boundary
    # =============================================================================
    print("\n2. Rectangular Boundary Example")
    print("-"*60)

    # Create rectangular boundary
    boundary_rect = Boundary.create_rectangle(
        xmin=-10*mm, xmax=10*mm,
        ymin=-8*mm, ymax=8*mm
    )

    # Generate cracks
    generator2 = CrackNetworkGenerator(domain_size=(25*mm, 25*mm), seed=123)
    generator2.generate_network(
        n_junctions=4,
        n_tips=15,
        edge_length_mean=3*mm,
        edge_length_std=1*mm,
        allow_intersections=False,
        spatial_distribution='uniform'
    )

    # Trim and add boundary
    boundary_manager2 = BoundaryManager(boundary_rect)
    generator2.G = boundary_manager2.trim_cracks_at_boundary(generator2.G, verbose=True)
    generator2.G = boundary_manager2.add_boundary_to_graph(generator2.G)

    # Visualize
    visualizer2 = NetworkVisualizer(generator2.G, (25*mm, 25*mm))
    fig, ax = visualizer2.visualize(
        figsize=(10, 10),
        show_labels=False,
        show_edge_labels=False,
        node_size=60
    )
    ax.set_title('Crack Network in Rectangular Boundary', fontsize=14, weight='bold')
    plt.savefig(str(out_dir) + "/example_rectangular_boundary.png", dpi=150, bbox_inches="tight")
    print("Saved: example_rectangular_boundary.png")
    plt.close()

    # =============================================================================
    # Example 3: Custom Polygon Boundary
    # =============================================================================
    print("\n3. Custom Polygon Boundary Example")
    print("-"*60)

    # Create hexagonal boundary
    n_sides = 6
    angles = np.linspace(0, 2*np.pi, n_sides, endpoint=False)
    radius = 10*mm
    hex_vertices = [radius * np.array([np.cos(a), np.sin(a)]) for a in angles]

    boundary_hex = Boundary.create_polygon(hex_vertices)

    # Generate cracks with paths
    generator3 = CrackNetworkGenerator(domain_size=(25*mm, 25*mm), seed=456)
    generator3.generate_network(
        n_junctions=0,
        n_tips=0,
        n_crack_paths=8,
        segments_per_path_mean=6,
        edge_length_mean=3*mm,
        edge_length_std=1*mm,
        path_angle_change_std_deg=20.0,
        allow_intersections=True,
        spatial_distribution='uniform'
    )

    # Trim and add boundary
    boundary_manager3 = BoundaryManager(boundary_hex)
    generator3.G = boundary_manager3.trim_cracks_at_boundary(generator3.G, verbose=True)
    generator3.G = boundary_manager3.add_boundary_to_graph(generator3.G)

    # Visualize with paths
    visualizer3 = NetworkVisualizer(generator3.G, (25*mm, 25*mm))
    fig, ax = visualizer3.visualize_with_paths(figsize=(10, 10))
    ax.set_title('Crack Paths in Hexagonal Boundary', fontsize=14, weight='bold')
    plt.savefig(str(out_dir) + "/example_hexagon_boundary.png", dpi=150, bbox_inches="tight")
    print("Saved: example_hexagon_boundary.png")
    plt.close()

    # =============================================================================
    # Example 4: Preventing Generation Outside Boundary
    # =============================================================================
    print("\n4. Generate Cracks Only Inside Boundary")
    print("-"*60)

    # You can also filter positions during generation
    # This is more efficient than trimming after

    boundary_disk = Boundary.create_circle(center=np.array([0, 0]), radius=12*mm)
    boundary_manager4 = BoundaryManager(boundary_disk)

    # Generate positions
    generator4 = CrackNetworkGenerator(domain_size=(25*mm, 25*mm), seed=789)

    # Manually generate positions inside boundary
    n_attempts = 100
    junction_positions = []
    for _ in range(n_attempts):
        pos = generator4._generate_positions(1, 'uniform')[0]
        if boundary_disk.is_point_inside(pos):
            junction_positions.append(pos)
            if len(junction_positions) >= 5:
                break

    junction_positions = np.array(junction_positions)
    print(f"Generated {len(junction_positions)} junction positions inside boundary")

    # Now use these positions... (would need to modify generator to accept positions)
    # For now, we can demonstrate the concept:

    # Generate network and trim
    generator4.generate_network(
        n_junctions=5,
        n_tips=12,
        edge_length_mean=3*mm,
        edge_length_std=1*mm,
        allow_intersections=False,
        spatial_distribution='uniform'
    )

    generator4.G = boundary_manager4.trim_cracks_at_boundary(generator4.G, verbose=True)
    generator4.G = boundary_manager4.add_boundary_to_graph(generator4.G)

    visualizer4 = NetworkVisualizer(generator4.G, (25*mm, 25*mm))
    fig, ax = visualizer4.visualize(figsize=(10, 10), show_labels=False, node_size=50)
    ax.set_title('Cracks Trimmed at Circular Boundary', fontsize=14, weight='bold')
    plt.savefig(str(out_dir) + "/example_trimmed_boundary.png", dpi=150, bbox_inches="tight")
    print("Saved: example_trimmed_boundary.png")
    plt.close()

    # =============================================================================
    # Example 5: Point Inside/Outside Testing
    # =============================================================================
    print("\n5. Inside/Outside Testing")
    print("-"*60)

    boundary_test = Boundary.create_circle(center=np.array([0, 0]), radius=10*mm)

    test_points = [
        np.array([0, 0]),
        np.array([5*mm, 0]),
        np.array([15*mm, 0]),
        np.array([7*mm, 7*mm]),
    ]

    for i, point in enumerate(test_points):
        inside = boundary_test.is_point_inside(point)
        status = "INSIDE" if inside else "OUTSIDE"
        print(f"  Point {i+1} at ({point[0]*1e3:.1f}, {point[1]*1e3:.1f}) mm: {status}")

    print("\n" + "="*60)
    print("Examples complete!")
    print("="*60)


    # ---- next cell ----

    """
    Example: CAD Import for Boundary Definitions

    Demonstrates importing boundaries from various CAD formats:
    - DXF files (AutoCAD)
    - SVG files (vector graphics)
    - Text files (simple formats)
    - Numpy arrays (programmatic)
    - STL files (2D projection)

    Also shows manual creation methods.
    """

    import numpy as np
    import matplotlib.pyplot as plt
    from Ugenerator import (
        CrackNetworkGenerator,
        Boundary,
        BoundaryManager,
        CADImporter,
        CADExporter,
        NetworkVisualizer
    )

    mm = 1e-3

    print("="*70)
    print("CAD Import Examples for Boundary Definitions")
    print("="*70)

    # =============================================================================
    # Method 1: Manual Creation (Built-in Shapes)
    # =============================================================================
    print("\n1. Manual Creation - Built-in Shapes")
    print("-"*70)

    # Circle (like your BEM disk)
    boundary_circle = Boundary.create_circle(
        center=np.array([0, 0]), 
        radius=10*mm
    )
    print("✓ Created circular boundary (radius = 10 mm)")

    # Rectangle
    boundary_rect = Boundary.create_rectangle(
        xmin=-10*mm, xmax=10*mm,
        ymin=-8*mm, ymax=8*mm
    )
    print("✓ Created rectangular boundary (20mm × 16mm)")

    # Regular polygon (hexagon)
    n_sides = 6
    angles = np.linspace(0, 2*np.pi, n_sides, endpoint=False)
    radius = 10*mm
    vertices = [radius * np.array([np.cos(a), np.sin(a)]) for a in angles]
    boundary_hex = Boundary.create_polygon(vertices)
    print(f"✓ Created hexagonal boundary ({n_sides} sides)")

    # =============================================================================
    # Method 2: From Numpy Arrays
    # =============================================================================
    print("\n2. From Numpy Arrays")
    print("-"*70)

    # Define vertices manually
    vertices = np.array([
        [-10, -10],
        [10, -10],
        [10, 10],
        [-10, 10]
    ]) * mm

    # Option A: Sequential connectivity (assumes closed loop)
    boundary_from_vertices = CADImporter.from_numpy_arrays(vertices)
    print("✓ Created boundary from vertex array (sequential connectivity)")

    # Option B: Explicit connectivity
    connectivity = np.array([
        [0, 1],  # Edge 0: vertex 0 to vertex 1
        [1, 2],  # Edge 1: vertex 1 to vertex 2
        [2, 3],  # Edge 2: vertex 2 to vertex 3
        [3, 0]   # Edge 3: vertex 3 to vertex 0
    ])
    boundary_from_arrays = CADImporter.from_numpy_arrays(vertices, connectivity)
    print("✓ Created boundary from vertex + connectivity arrays")

    # =============================================================================
    # Method 3: From Text File
    # =============================================================================
    print("\n3. From Text File")
    print("-"*70)

    # Create example text file with vertices
    example_vertices = np.array([
        [0, 0],
        [15, 0],
        [15, 10],
        [0, 10]
    ]) * mm

    # Save to file
    np.savetxt('boundary_vertices.txt', example_vertices, 
              header='x y', comments='', fmt='%.6e')
    print("✓ Created example text file: boundary_vertices.txt")

    # Import from text file
    boundary_from_text = CADImporter.from_text_file(
        'boundary_vertices.txt', 
        format='vertices'
    )
    print("✓ Imported boundary from text file")

    # =============================================================================
    # Method 4: From DXF File (if ezdxf available)
    # =============================================================================
    print("\n4. From DXF File (AutoCAD format)")
    print("-"*70)

    try:
        import ezdxf
        
        # Create example DXF file
        doc = ezdxf.new('R2010')
        msp = doc.modelspace()
        
        # Add a rectangle to DXF
        rect_points = [
            (-12*mm*1e3, -12*mm*1e3),  # DXF typically uses mm
            (12*mm*1e3, -12*mm*1e3),
            (12*mm*1e3, 12*mm*1e3),
            (-12*mm*1e3, 12*mm*1e3),
            (-12*mm*1e3, -12*mm*1e3)  # Close
        ]
        msp.add_lwpolyline(rect_points, dxfattribs={'layer': 'BOUNDARY'})
        
        # Add a circle
        msp.add_circle((0, 0), 5*mm*1e3, dxfattribs={'layer': 'BOUNDARY'})
        
        doc.saveas('example_boundary.dxf')
        print("✓ Created example DXF file: example_boundary.dxf")
        
        # Import from DXF
        boundary_from_dxf = CADImporter.from_dxf(
            'example_boundary.dxf',
            layer='BOUNDARY',
            scale=mm  # Convert from mm to meters
        )
        print("✓ Imported boundary from DXF file")
        
    except ImportError:
        print("⚠ ezdxf not installed. Install with: pip install ezdxf")
        print("  Skipping DXF import example")

    # =============================================================================
    # Method 5: From SVG File (if svgpathtools available)
    # =============================================================================
    print("\n5. From SVG File (vector graphics)")
    print("-"*70)

    try:
        from svgpathtools import Path, Line, wsvg
        
        # Create simple SVG with a rectangle
        path = Path(
            Line(complex(-10, -10), complex(10, -10)),
            Line(complex(10, -10), complex(10, 10)),
            Line(complex(10, 10), complex(-10, 10)),
            Line(complex(-10, 10), complex(-10, -10))
        )
        
        wsvg([path], filename='example_boundary.svg')
        print("✓ Created example SVG file: example_boundary.svg")
        
        # Import from SVG
        boundary_from_svg = CADImporter.from_svg(
            'example_boundary.svg',
            scale=mm
        )
        print("✓ Imported boundary from SVG file")
        
    except ImportError:
        print("⚠ svgpathtools not installed. Install with: pip install svgpathtools")
        print("  Skipping SVG import example")

    # =============================================================================
    # Method 6: Complex Shape from Points
    # =============================================================================
    print("\n6. Complex Custom Shape")
    print("-"*70)

    # Create a gear-like shape
    n_teeth = 8
    inner_radius = 8*mm
    outer_radius = 10*mm
    complex_vertices = []

    for i in range(2 * n_teeth):
        angle = i * np.pi / n_teeth
        if i % 2 == 0:
            r = outer_radius
        else:
            r = inner_radius
        complex_vertices.append(r * np.array([np.cos(angle), np.sin(angle)]))

    boundary_gear = Boundary.create_polygon(complex_vertices)
    print(f"✓ Created gear-like boundary with {n_teeth} teeth")

    # =============================================================================
    # Use Case: Generate Cracks in Imported Boundary
    # =============================================================================
    print("\n7. Generate Cracks in Imported Boundary")
    print("-"*70)

    # Use the circular boundary
    boundary = boundary_circle

    # Generate crack network
    generator = CrackNetworkGenerator(
        domain_size=(25*mm, 25*mm),
        seed=42
    )

    generator.generate_network(
        n_junctions=4,
        n_tips=15,
        n_crack_paths=3,
        segments_per_path_mean=5,
        edge_length_mean=4*mm,
        edge_length_std=1*mm,
        path_angle_change_std_deg=25.0,
        allow_intersections=True,
        spatial_distribution='uniform'
    )

    print(f"Generated {generator.G.number_of_nodes()} nodes, "
          f"{generator.G.number_of_edges()} edges")

    # Apply boundary
    boundary_manager = BoundaryManager(boundary)
    generator.G = boundary_manager.trim_cracks_at_boundary(generator.G, verbose=True)
    generator.G = boundary_manager.add_boundary_to_graph(generator.G)

    # Visualize
    visualizer = NetworkVisualizer(generator.G, (25*mm, 25*mm))
    fig, ax = visualizer.visualize(
        figsize=(10, 10),
        show_labels=False,
        show_edge_labels=False,
        node_size=60
    )
    ax.set_title('Crack Network in Imported Circular Boundary', 
                 fontsize=14, weight='bold')
    plt.savefig('cracks_in_imported_boundary.png', dpi=150, bbox_inches='tight')
    print("✓ Saved: cracks_in_imported_boundary.png")
    plt.close()

    # =============================================================================
    # Export Boundary to CAD
    # =============================================================================
    print("\n8. Export Boundary to CAD Formats")
    print("-"*70)

    try:
        import ezdxf
        # Export boundary to DXF
        CADExporter.to_dxf(boundary_circle, 'exported_boundary.dxf', layer='CRACK_BOUNDARY')
        print("✓ Exported boundary to DXF: exported_boundary.dxf")
    except ImportError:
        print("⚠ ezdxf not available for export")

    # Export to text file
    CADExporter.to_text_file(boundary_circle, 'exported_boundary.txt', n_points_per_segment=50)
    print("✓ Exported boundary to text file: exported_boundary.txt")

    # =============================================================================
    # Summary Table
    # =============================================================================
    print("\n" + "="*70)
    print("SUMMARY: Boundary Definition Methods")
    print("="*70)
    print()
    print("Method                 | Function                          | Requires")
    print("-" * 70)
    print("Built-in shapes        | Boundary.create_circle()          | Nothing")
    print("                       | Boundary.create_rectangle()       | ")
    print("                       | Boundary.create_polygon()         | ")
    print("-" * 70)
    print("Numpy arrays           | CADImporter.from_numpy_arrays()   | numpy")
    print("-" * 70)
    print("Text file              | CADImporter.from_text_file()      | numpy")
    print("-" * 70)
    print("DXF (AutoCAD)          | CADImporter.from_dxf()            | ezdxf")
    print("-" * 70)
    print("SVG (vector graphics)  | CADImporter.from_svg()            | svgpathtools")
    print("-" * 70)
    print("STL (3D mesh, 2D proj) | CADImporter.from_stl_2d()         | numpy-stl")
    print("="*70)

    print("\nInstallation commands:")
    print("  pip install ezdxf           # For DXF support")
    print("  pip install svgpathtools    # For SVG support")
    print("  pip install numpy-stl       # For STL support")
    print()
    print("="*70)
    print("Examples complete!")
    print("="*70)


    # ---- next cell ----

    """
    Example: Simplify Evolved Crack Network
    ========================================

    This example shows how to:
    1. Import your evolved crack network
    2. Simplify it (merge colinear edges, remove small features)
    3. Export it back for continued growth
    4. Use with CrackNetworkGenerator for further evolution
    """

    import numpy as np
    import matplotlib.pyplot as plt
    from fracture_utils.Ugenerator.crack_network_simplifier import CrackNetworkSimplifier, SimplificationConfig

    # Assuming you have your fracture_utils available
    # from fracture_utils import CrackNetworkGenerator

    # ============================================================================
    # STEP 1: Get Your Evolved Network Data
    # ============================================================================

    # Example: From your crack evolution simulation
    # Assuming you have vertices and connectivity from your simulation

    # Your evolved network might look like this:
    # vertices shape: (n_vertices, 3) = [id, x, y] or (n_vertices, 2) = [x, y]
    # connectivity shape: (n_edges, 2) = [v0, v1]

    # For this example, let's assume you extract it from your simulation:
    """
    # From your simulation code:
    evolved_vertices = your_network.get_vertices()  # Get current vertex positions
    evolved_connectivity = your_network.get_edges()  # Get current connectivity

    # OR if using CrackNetworkV4:
    vertices_array = []
    for v in net.vertices:
        vertices_array.append([v.id, v.x, v.y])
    evolved_vertices = np.array(vertices_array)

    connectivity_array = []
    for e in net.edges:
        connectivity_array.append([e.v0, e.v1])
    evolved_connectivity = np.array(connectivity_array)
    """

    # ============================================================================
    # EXAMPLE DATA (Replace with your actual network)
    # ============================================================================

    mm = 1e-3

    # Simulated evolved network with many small/colinear segments
    np.random.seed(42)

    # Create example evolved network (many vertices from growth simulation)
    n_vertices = 50
    vertices = np.zeros((n_vertices, 3))
    vertices[:, 0] = np.arange(n_vertices)  # IDs

    # Positions forming a meandering path with many small segments
    t = np.linspace(0, 4*np.pi, n_vertices)
    vertices[:, 1] = 50*mm * t/np.max(t) + 10*mm*np.sin(5*t)  # x
    vertices[:, 2] = 100*mm * (np.sin(t) + 0.5*np.sin(3*t))    # y

    # Connectivity: chain of edges
    connectivity = np.array([[i, i+1] for i in range(n_vertices-1)])

    print("Evolved Network (Before Simplification):")
    print(f"  Vertices: {len(vertices)}")
    print(f"  Edges: {len(connectivity)}")

    # ============================================================================
    # STEP 2: Configure Simplification
    # ============================================================================

    # Option 1: Use defaults
    # config = SimplificationConfig()

    # Option 2: Custom configuration
    config = SimplificationConfig(
        max_angle_deviation=5.0,      # Merge edges if angle deviation < 5°
        min_edge_length=0.5*mm,       # Remove edges shorter than 0.5mm
        merge_at_degree2=True,        # Merge colinear edges at internal nodes
        remove_degree2_nodes=True,    # Remove degree-2 internal nodes
        preserve_tips=True,           # Don't remove crack tips
        preserve_junctions=True,      # Don't modify junctions
        max_merged_edge_length=20*mm  # Don't merge if result > 20mm
    )

    print("\nSimplification Configuration:")
    print(f"  Max angle deviation: {config.max_angle_deviation}°")
    print(f"  Min edge length: {config.min_edge_length/mm:.2f} mm")
    print(f"  Merge at degree-2: {config.merge_at_degree2}")

    # ============================================================================
    # STEP 3: Load and Simplify Network
    # ============================================================================

    # Create simplifier
    simplifier = CrackNetworkSimplifier(config=config)

    # Load your evolved network
    simplifier.load_from_arrays(vertices, connectivity)

    # Get statistics before simplification
    stats_before = simplifier.get_statistics()
    print("\nBefore Simplification:")
    print(f"  Vertices: {stats_before['n_vertices']}")
    print(f"  Edges: {stats_before['n_edges']}")
    print(f"  Tips: {stats_before['n_tips']}")
    print(f"  Junctions: {stats_before['n_junctions']}")
    print(f"  Internal: {stats_before['n_internal']}")

    # Simplify!
    simplification_stats = simplifier.simplify(verbose=True)

    # Get statistics after simplification
    stats_after = simplifier.get_statistics()
    print("\nAfter Simplification:")
    print(f"  Vertices: {stats_after['n_vertices']}")
    print(f"  Edges: {stats_after['n_edges']}")
    print(f"  Tips: {stats_after['n_tips']}")
    print(f"  Junctions: {stats_after['n_junctions']}")
    print(f"  Internal: {stats_after['n_internal']}")

    # ============================================================================
    # STEP 4: Export Simplified Network
    # ============================================================================

    # Export to arrays
    simplified_vertices, simplified_connectivity = simplifier.to_arrays()

    print("\nSimplified Network:")
    print(f"  Vertices: {len(simplified_vertices)}")
    print(f"  Edges: {len(simplified_connectivity)}")
    print(f"  Reduction: {len(vertices) - len(simplified_vertices)} vertices, "
          f"{len(connectivity) - len(simplified_connectivity)} edges")

    # ============================================================================
    # STEP 5: Use Simplified Network for Continued Growth
    # ============================================================================

    # Option A: Convert back to your simulation format
    """
    # Load into your CrackNetworkV4 or other format
    simplified_net = CrackNetworkV4.from_vertices_connectivity(
        vertices=simplified_vertices,
        connectivity=simplified_connectivity,
        Nv_max=len(simplified_vertices),
        validate=True
    )

    # Continue growth simulation
    # ... your growth code ...
    """

    # Option B: Use with CrackNetworkGenerator for further evolution
    """
    from fracture_utils import CrackNetworkGenerator

    # Create generator
    gen = CrackNetworkGenerator(domain_size=(100*mm, 200*mm))

    # Load simplified network
    gen.load_from_arrays(simplified_vertices, simplified_connectivity)

    # Add more cracks to the simplified network
    gen.generate_network(
        n_crack_paths=5,  # Add 5 more crack paths
        segments_per_path_mean=3,
        edge_length_mean=5*mm,
        allow_intersections=True
    )

    # Export combined network
    final_vertices, final_connectivity = gen.to_arrays(return_types=False)
    """

    # ============================================================================
    # STEP 6: Visualize Before/After
    # ============================================================================

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

    # Plot original evolved network
    ax1.set_title("Before Simplification", fontsize=14, fontweight='bold')
    for v0, v1 in connectivity:
        pos0 = vertices[int(v0), 1:]
        pos1 = vertices[int(v1), 1:]
        ax1.plot([pos0[0]/mm, pos1[0]/mm], 
                 [pos0[1]/mm, pos1[1]/mm], 
                 'b-', linewidth=1, alpha=0.7)

    ax1.scatter(vertices[:, 1]/mm, vertices[:, 2]/mm, 
               c='red', s=20, zorder=5, alpha=0.5)
    ax1.set_xlabel('x [mm]')
    ax1.set_ylabel('y [mm]')
    ax1.grid(True, alpha=0.3)
    ax1.axis('equal')
    ax1.text(0.02, 0.98, f'{len(vertices)} vertices\n{len(connectivity)} edges',
             transform=ax1.transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    # Plot simplified network
    ax2.set_title("After Simplification", fontsize=14, fontweight='bold')
    for v0, v1 in simplified_connectivity:
        pos0 = simplified_vertices[int(v0), 1:]
        pos1 = simplified_vertices[int(v1), 1:]
        ax2.plot([pos0[0]/mm, pos1[0]/mm], 
                 [pos0[1]/mm, pos1[1]/mm], 
                 'g-', linewidth=2, alpha=0.7)

    ax2.scatter(simplified_vertices[:, 1]/mm, simplified_vertices[:, 2]/mm, 
               c='darkgreen', s=30, zorder=5)
    ax2.set_xlabel('x [mm]')
    ax2.set_ylabel('y [mm]')
    ax2.grid(True, alpha=0.3)
    ax2.axis('equal')
    ax2.text(0.02, 0.98, 
             f'{len(simplified_vertices)} vertices\n{len(simplified_connectivity)} edges',
             transform=ax2.transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

    plt.tight_layout()
    plt.savefig('network_simplification_comparison.png', dpi=150, bbox_inches='tight')
    print("\n✓ Saved comparison plot: network_simplification_comparison.png")

    # ============================================================================
    # STEP 7: Advanced - Iterative Simplification
    # ============================================================================

    def iterative_simplification(vertices, connectivity, 
                                 max_iterations=5,
                                 config=None):
        """
        Apply simplification iteratively until no more changes occur.
        
        Sometimes merging edges creates new opportunities for simplification.
        """
        if config is None:
            config = SimplificationConfig()
        
        print("\n" + "="*60)
        print("ITERATIVE SIMPLIFICATION")
        print("="*60)
        
        current_vertices = vertices.copy()
        current_connectivity = connectivity.copy()
        
        for iteration in range(max_iterations):
            print(f"\nIteration {iteration + 1}:")
            print(f"  Input: {len(current_vertices)} vertices, "
                  f"{len(current_connectivity)} edges")
            
            # Simplify
            simplifier = CrackNetworkSimplifier(config=config)
            simplifier.load_from_arrays(current_vertices, current_connectivity)
            stats = simplifier.simplify(verbose=False)
            
            # Export
            new_vertices, new_connectivity = simplifier.to_arrays()
            
            # Check if converged
            if (len(new_vertices) == len(current_vertices) and 
                len(new_connectivity) == len(current_connectivity)):
                print("  ✓ Converged (no more changes)")
                break
            
            current_vertices = new_vertices
            current_connectivity = new_connectivity
            
            print(f"  Output: {len(new_vertices)} vertices, "
                  f"{len(new_connectivity)} edges")
        
        return current_vertices, current_connectivity

    # Example usage
    final_vertices, final_connectivity = iterative_simplification(
        vertices, connectivity, 
        max_iterations=5,
        config=config
    )

    print("\n" + "="*60)
    print("FINAL RESULT")
    print("="*60)
    print(f"Original: {len(vertices)} vertices, {len(connectivity)} edges")
    print(f"Final: {len(final_vertices)} vertices, {len(final_connectivity)} edges")
    print(f"Total reduction: {len(vertices) - len(final_vertices)} vertices, "
          f"{len(connectivity) - len(final_connectivity)} edges")


    # ---- next cell ----

    """
    QUICK GUIDE: Simplify Your Evolved Crack Network
    =================================================

    Step-by-step guide to simplify your evolved network from the image.
    """

    import numpy as np
    from crack_network_simplifier import CrackNetworkSimplifier, SimplificationConfig

    # ============================================================================
    # YOUR CURRENT WORKFLOW (from your image)
    # ============================================================================

    """
    You have an evolved crack network that looks complex with many segments.
    From your plot, it appears you have:
    - Many vertices (v1, v2, v38, v41, v42, v44, v47, v48, etc.)
    - Many edges connecting them
    - Some very small segments
    - Some nearly-colinear segments
    """

    # ============================================================================
    # STEP 1: Extract Network from Your Simulation
    # ============================================================================

    # Method A: If using CrackNetworkV4 from Usolver
    """
    from fracture_utils.Usolver.network import CrackNetworkV4

    # Assuming 'net' is your current network
    vertices_list = []
    for v in net.vertices:
        vertices_list.append([v.id, v.x, v.y])
    vertices = np.array(vertices_list)

    connectivity_list = []
    for e in net.edges:
        connectivity_list.append([e.v0, e.v1])
    connectivity = np.array(connectivity_list)
    """

    # Method B: If you have arrays already
    """
    # Directly from your simulation
    vertices = your_network.vertices  # Shape (n, 3) or (n, 2)
    connectivity = your_network.edges  # Shape (m, 2)
    """

    # ============================================================================
    # STEP 2: Configure What to Simplify
    # ============================================================================

    mm = 1e-3

    # Create configuration
    config = SimplificationConfig(
        # Merge edges if angle deviation < 5 degrees
        max_angle_deviation=5.0,
        
        # Remove edges shorter than 0.5mm (adjust based on your scale)
        min_edge_length=0.5*mm,
        
        # Merge colinear segments at degree-2 nodes
        merge_at_degree2=True,
        
        # Remove degree-2 internal nodes (simplify polylines)
        remove_degree2_nodes=True,
        
        # Keep crack tips
        preserve_tips=True,
        
        # Keep junctions
        preserve_junctions=True,
        
        # Don't merge if result would be > 20mm
        max_merged_edge_length=20*mm
    )

    # Adjust these based on your needs:
    # - If you want MORE simplification: increase max_angle_deviation (e.g., 10°)
    # - If you want to remove more small features: increase min_edge_length
    # - If you want to preserve more detail: decrease max_angle_deviation

    # ============================================================================
    # STEP 3: Simplify
    # ============================================================================

    # Create simplifier
    simplifier = CrackNetworkSimplifier(config=config)

    # Load your network
    simplifier.load_from_arrays(vertices, connectivity)

    # Get stats before
    print("Before simplification:")
    stats = simplifier.get_statistics()
    print(f"  Vertices: {stats['n_vertices']}")
    print(f"  Edges: {stats['n_edges']}")

    # Simplify!
    simplifier.simplify(verbose=True)

    # Get stats after
    print("\nAfter simplification:")
    stats = simplifier.get_statistics()
    print(f"  Vertices: {stats['n_vertices']}")
    print(f"  Edges: {stats['n_edges']}")

    # ============================================================================
    # STEP 4: Export Simplified Network
    # ============================================================================

    # Get simplified arrays
    simplified_vertices, simplified_connectivity = simplifier.to_arrays()

    print(f"\n✓ Simplified network ready:")
    print(f"  vertices shape: {simplified_vertices.shape}")
    print(f"  connectivity shape: {simplified_connectivity.shape}")

    # ============================================================================
    # STEP 5: Convert Back to Your Format and Continue Growth
    # ============================================================================

    # Option 1: Load back into CrackNetworkV4
    """
    from fracture_utils.Usolver.network import CrackNetworkV4

    simplified_net = CrackNetworkV4.from_vertices_connectivity(
        vertices=simplified_vertices,
        connectivity=simplified_connectivity,
        Nv_max=len(simplified_vertices),
        validate=True
    )

    # Now continue your crack evolution with simplified_net
    # ... your growth algorithm ...
    """

    # Option 2: Use with CrackNetworkGenerator to add more cracks
    """
    from fracture_utils import CrackNetworkGenerator

    gen = CrackNetworkGenerator(domain_size=(200*mm, 300*mm))
    gen.load_from_arrays(simplified_vertices, simplified_connectivity)

    # Add more crack paths to the simplified network
    gen.generate_network(
        n_crack_paths=3,
        segments_per_path_mean=5,
        edge_length_mean=10*mm,
        allow_intersections=True
    )

    # Get final network
    final_vertices, final_connectivity = gen.to_arrays(return_types=False)
    """

    # ============================================================================
    # STEP 6: Visual Comparison (Optional)
    # ============================================================================

    import matplotlib.pyplot as plt

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

    # Before
    ax1.set_title("Your Evolved Network (Before)")
    for v0, v1 in connectivity:
        p0 = vertices[int(v0), 1:]
        p1 = vertices[int(v1), 1:]
        ax1.plot([p0[0]/mm, p1[0]/mm], [p0[1]/mm, p1[1]/mm], 'b-', lw=1)
    ax1.scatter(vertices[:, 1]/mm, vertices[:, 2]/mm, c='red', s=10)
    ax1.set_xlabel('x [mm]')
    ax1.set_ylabel('y [mm]')
    ax1.grid(True, alpha=0.3)
    ax1.axis('equal')

    # After
    ax2.set_title("Simplified Network (After)")
    for v0, v1 in simplified_connectivity:
        p0 = simplified_vertices[int(v0), 1:]
        p1 = simplified_vertices[int(v1), 1:]
        ax2.plot([p0[0]/mm, p1[0]/mm], [p0[1]/mm, p1[1]/mm], 'g-', lw=2)
    ax2.scatter(simplified_vertices[:, 1]/mm, simplified_vertices[:, 2]/mm, 
               c='darkgreen', s=20)
    ax2.set_xlabel('x [mm]')
    ax2.set_ylabel('y [mm]')
    ax2.grid(True, alpha=0.3)
    ax2.axis('equal')

    plt.tight_layout()
    plt.savefig('my_network_simplified.png', dpi=150)
    print("\n✓ Saved: my_network_simplified.png")

    # ============================================================================
    # KEY PARAMETERS TO ADJUST
    # ============================================================================

    print("\n" + "="*60)
    print("TUNING GUIDE")
    print("="*60)
    print("""
    Adjust these parameters based on your needs:

    1. max_angle_deviation (default: 5.0 degrees)
       - INCREASE (e.g., 10°) for MORE aggressive merging
       - DECREASE (e.g., 2°) to preserve more shape details
       
    2. min_edge_length (default: 0.5mm)
       - INCREASE to remove more small features
       - DECREASE to preserve finer details
       
    3. merge_at_degree2 (default: True)
       - True: Merge colinear edges at internal nodes
       - False: Keep all edges separate
       
    4. remove_degree2_nodes (default: True)
       - True: Simplify polylines (remove internal nodes)
       - False: Keep all vertices

    Example for AGGRESSIVE simplification:
        config = SimplificationConfig(
            max_angle_deviation=15.0,
            min_edge_length=2.0*mm,
            merge_at_degree2=True,
            remove_degree2_nodes=True
        )

    Example for CONSERVATIVE simplification:
        config = SimplificationConfig(
            max_angle_deviation=2.0,
            min_edge_length=0.1*mm,
            merge_at_degree2=False,
            remove_degree2_nodes=False
        )
    """)

    return locals()


### Case `propagation` (from `propagation.ipynb`)


In [ ]:
def run_propagation():
    """Case extracted from propagation.ipynb."""
    # ============================================================
    # Cotterell?Rice (IJF 1980) validation for circular arc crack
    # ============================================================

    import os, sys
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt

    # Add repo root (one level above notebooks/) to sys.path
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    # --- Preamble re-exports
    out_dir = Path(repo_root) / "output" / "Arc_CotterellRice_Validation"
    out_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # Problem inputs
    # -------------------------
    mm = 1e-3
    a_chord = 10.0 * mm     # HALF chord length
    E = 200e9
    nu = 0.30
    plane = "strain"
    edge_index = 0

    sig_xx = 0e6
    sig_yy = 100e6
    sig_xy = 0.0

    # alpha = half-angle; total arc angle = 2*alpha
    alpha_deg = np.linspace(1.0, 30.0, 6)
    alpha = np.deg2rad(alpha_deg)

    ne_half_values = [30, 40, 60]
    scale = 1e-6

    # solver knobs
    base_knobs = dict(
        crack_mode="half",
        representation="cheb_quad",
        node_distribution="tip_dense",
        nq_col=12,
        nq_stress=16,
        ridge=1e-10,
        r0_factor=1e-3,
        solver_option="parametrized_crack",
        junction_model="strict",
        parametrization="polyline",
    )

    # -------------------------
    # Arc geometry (TRUE circle center)
    # -------------------------
    def arc_network(alpha):
        R = a_chord / np.sin(alpha)
        y_c = R * np.cos(alpha)

        V = np.array([
            [0, -a_chord, 0.0],
            [1,  a_chord, 0.0],
            [2,  0.0,     y_c],
        ], float)

        E_conn = np.array([[0, 1]], int)

        return CrackNetworkV4.from_vertices_connectivity(
            vertices=V,
            connectivity=E_conn,
            Nv_max=4,
            validate=True,
            edge_kinds=["arc"],
            edge_ctrl_vids=[(2,)],
            vertex_roles={2: "control"},
        )

    # -------------------------
    # Solver wrapper
    # -------------------------
    def solve_K_arc(net, ne_half, al):
        material = Material(E=E, nu=nu, plane_stress=("stress" in plane))
        applied = AppliedStress(sig_xx, sig_yy, sig_xy)

        calc = DCENetworkStaticV4(material, net, applied)
        knobs = dict(base_knobs)
        knobs["ne_half"] = int(ne_half)

        sol = calc.solve(**knobs)
        res = DCEResultsNetworkV4(calc, sol)

        x, COD, CSD, extra = res.reconstruct_cod_csd_panel_midpoints(
            edge_index=edge_index,
            enforce_global_tip_zero=True,
        )

        a_arc = float(extra["a_edge"])

        mu = E / (2.0 * (1.0 + nu))
        kappa = 3.0 - 4.0 * nu

        KI, KII, _ = sif_from_cod_fit(
            x=x,
            COD=COD,
            CSD=CSD,
            a=a_arc,
            mu=mu,
            kappa=kappa,
            tip="right",
            window="fixed",
            rho_min=1e-6,
            rho_max=0.2,
            min_pts=10,
            n_fit=400,
            two_term=False,
        )

        KI_rn, KII_rn = rotate_sifs(float(KI), float(KII), -2.0 * al)
        return KI_rn, -KII_rn


    # -------------------------
    # Run sweep
    # -------------------------
    KI_ana, KII_ana = [], []
    for al in alpha:
        R = a_chord / np.sin(al)
        a_arc = R * al
        KIa, KIIa = cotterell_rice_K(al, a_arc, sig_xx, sig_yy, sig_xy)
        KI_ana.append(KIa)
        KII_ana.append(KIIa)

    KI_ana = np.asarray(KI_ana, float)
    KII_ana = np.asarray(KII_ana, float)

    num_by_ne = {}
    for ne_half in ne_half_values:
        KI_num, KII_num = [], []
        for al in alpha:
            net = arc_network(al)
            ki, kii = solve_K_arc(net, ne_half, al)
            KI_num.append(ki)
            KII_num.append(kii)

        num_by_ne[ne_half] = {
            "KI": np.asarray(KI_num, float),
            "KII": np.asarray(KII_num, float),
        }

    # -------------------------
    # Plots: KI and KII overlays
    # -------------------------
    colors = {30: "tab:blue", 40: "tab:orange", 60: "tab:green"}

    fig1, ax1 = plt.subplots()
    ax1.plot(alpha_deg, KI_ana * scale, "k--", lw=3, label="Analytical (Cotterell-Rice)")
    for ne_half in ne_half_values:
        ax1.plot(
            alpha_deg,
            num_by_ne[ne_half]["KI"] * scale,
            lw=2,
            color=colors[ne_half],
            label=f"Numerical arc ne_half={ne_half}",
        )
    ax1.set_xlabel(r"$\alpha$ [deg]  (total arc angle $=2\alpha$)")
    ax1.set_ylabel(r"$K_I$ [MPa $\sqrt{m}$]")
    ax1.set_title(r"Arc crack: $K_I$ vs $\alpha$")
    ax1.grid(True)
    ax1.legend()
    ax1.set_ylim(bottom=0)
    fig1.savefig(out_dir / "KI_vs_alpha_ne30_40_60.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig2, ax2 = plt.subplots()
    ax2.plot(alpha_deg, KII_ana * scale, "k--", lw=3, label="Analytical (Cotterell-Rice)")
    for ne_half in ne_half_values:
        ax2.plot(
            alpha_deg,
            num_by_ne[ne_half]["KII"] * scale,
            lw=2,
            color=colors[ne_half],
            label=f"Numerical arc ne_half={ne_half}",
        )
    ax2.set_xlabel(r"$\alpha$ [deg]  (total arc angle $=2\alpha$)")
    ax2.set_ylabel(r"$K_{II}$ [MPa $\sqrt{m}$]")
    ax2.set_title(r"Arc crack: $K_{II}$ vs $\alpha$")
    ax2.grid(True)
    ax2.legend()
    ax2.set_ylim(bottom=0)
    fig2.savefig(out_dir / "KII_vs_alpha_ne30_40_60.png", dpi=300, bbox_inches="tight")
    plt.show()

    # -------------------------
    # Plots: percent error overlays
    # -------------------------
    def percent_error(num, ana, eps=1e-14):
        return 100.0 * np.abs(num - ana) / np.maximum(np.abs(ana), eps)

    fig3, ax3 = plt.subplots()
    for ne_half in ne_half_values:
        err_ki = percent_error(num_by_ne[ne_half]["KI"], KI_ana)
        ax3.plot(alpha_deg, err_ki, lw=2, color=colors[ne_half], label=f"ne_half={ne_half}")
    ax3.set_xlabel(r"$\alpha$ [deg]  (total arc angle $=2\alpha$)")
    ax3.set_ylabel(r"Percent error [%]")
    ax3.set_title(r"$K_I$ percent error: numerical vs analytical")
    ax3.grid(True)
    ax3.legend()
    fig3.savefig(out_dir / "KI_percent_error_ne30_40_60.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig4, ax4 = plt.subplots()
    for ne_half in ne_half_values:
        err_kii = percent_error(num_by_ne[ne_half]["KII"], KII_ana)
        ax4.plot(alpha_deg, err_kii, lw=2, color=colors[ne_half], label=f"ne_half={ne_half}")
    ax4.set_xlabel(r"$\alpha$ [deg]  (total arc angle $=2\alpha$)")
    ax4.set_ylabel(r"Percent error [%]")
    ax4.set_title(r"$K_{II}$ percent error: numerical vs analytical")
    ax4.grid(True)
    ax4.legend()
    fig4.savefig(out_dir / "KII_percent_error_ne30_40_60.png", dpi=300, bbox_inches="tight")
    plt.show()



    # ---- next cell ----

    # ============================================================
    # Cotterell-Rice validation for circular arc crack using a POLYLINE approximation
    # ============================================================

    import os, sys
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt

    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    out_dir = Path(repo_root) / "output" / "ArcPolyline_CotterellRice_Validation"
    out_dir.mkdir(parents=True, exist_ok=True)

    mm = 1e-3
    a_chord = 10.0 * mm
    E = 200e9
    nu = 0.30
    plane = "stress"

    sig_xx = 0e6
    sig_yy = 100e6
    sig_xy = 0.0

    alpha_deg = np.linspace(1.0, 45.0, 12)
    alpha = np.deg2rad(alpha_deg)

    ne_half_values = [30, 40, 60]
    nseg_test = 25
    scale = 1e-6

    min_segs = 6
    fit_frac = 0.30
    rmax_frac = 0.30
    two_term = True

    base_knobs = dict(
        crack_mode="half",
        representation="cheb_quad",
        node_distribution="tip_dense",
        nq_col=12,
        nq_stress=16,
        ridge=1e-10,
        r0_factor=1e-3,
        solver_option="parametrized_crack",
        junction_model="strict",
        parametrization="polyline",
    )

    from fracture_utils.Uprocessor import SIF_cod as sifcod

    def solve_arc_polyline(al, *, ne_half, nseg, fit_frac, min_segs, rmax_frac, two_term):
        R = a_chord / np.sin(al)
        a_arc = R * al
        net, V = polyline_arc_network(al, a_chord, nseg=nseg)
        edge_tip = int(nseg - 1)

        KI, KII, KI_CR, KII_CR, meta = sifcod.solve_K_polyline(
            net=net,
            V=V,
            ne_half=int(ne_half),
            edge_index_use=edge_tip,
            a_fit_global=float(a_arc),
            E=E, nu=nu, plane=plane,
            sig_xx=sig_xx, sig_yy=sig_yy, sig_xy=sig_xy,
            base_knobs=base_knobs,
            fit_frac=float(fit_frac),
            min_segs=int(min_segs),
            rmin_frac=1e-6,
            rmax_frac=float(rmax_frac),
            min_pts=12,
            two_term=bool(two_term),
            theta_rot_override=float(-2.0 * al),
        )
        return float(a_arc), float(KI_CR), float(KII_CR), meta

    KI_ana, KII_ana = [], []
    for al in alpha:
        R = a_chord / np.sin(al)
        a_arc = R * al
        KIa, KIIa = cotterell_rice_K(al, a_arc, sig_xx, sig_yy, sig_xy)
        KI_ana.append(KIa)
        KII_ana.append(KIIa)

    KI_ana = np.asarray(KI_ana, float)
    KII_ana = np.asarray(KII_ana, float)

    num_by_ne = {}
    for ne_half in ne_half_values:
        KI_poly, KII_poly = [], []
        for al in alpha:
            _, KIn, KIIn, _ = solve_arc_polyline(
                al,
                ne_half=ne_half,
                nseg=nseg_test,
                fit_frac=fit_frac,
                min_segs=min_segs,
                rmax_frac=rmax_frac,
                two_term=two_term,
            )
            KI_poly.append(KIn)
            KII_poly.append(KIIn)

        num_by_ne[ne_half] = {
            "KI": np.asarray(KI_poly, float),
            "KII": -np.asarray(KII_poly, float),
        }

    colors = {30: "tab:blue", 40: "tab:orange", 60: "tab:green"}

    fig1, ax1 = plt.subplots()
    ax1.plot(alpha_deg, KI_ana * scale, "k--", lw=3, label="Analytical")
    for ne_half in ne_half_values:
        ax1.plot(alpha_deg, num_by_ne[ne_half]["KI"] * scale, lw=2, color=colors[ne_half], label=f"Numerical ne_half={ne_half}")
    ax1.set_xlabel(r"$\alpha$ [deg]")
    ax1.set_ylabel(r"$K_I$ [MPa $\sqrt{m}$]")
    ax1.set_title("K_I: analytical + numerical overlays")
    ax1.grid(True)
    ax1.legend()
    fig1.savefig(out_dir / f"K_I_vs_alpha_ne30_40_60_nseg{nseg_test}.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig2, ax2 = plt.subplots()
    ax2.plot(alpha_deg, KII_ana * scale, "k--", lw=3, label="Analytical")
    for ne_half in ne_half_values:
        ax2.plot(alpha_deg, num_by_ne[ne_half]["KII"] * scale, lw=2, color=colors[ne_half], label=f"Numerical ne_half={ne_half}")
    ax2.set_xlabel(r"$\alpha$ [deg]")
    ax2.set_ylabel(r"$K_{II}$ [MPa $\sqrt{m}$]")
    ax2.set_title("K_II: analytical + numerical overlays")
    ax2.grid(True)
    ax2.legend()
    fig2.savefig(out_dir / f"K_II_vs_alpha_ne30_40_60_nseg{nseg_test}.png", dpi=300, bbox_inches="tight")
    plt.show()

    def percent_error(num, ana, eps=1e-14):
        return 100.0 * np.abs(num - ana) / np.maximum(np.abs(ana), eps)

    fig3, ax3 = plt.subplots()
    for ne_half in ne_half_values:
        ax3.plot(alpha_deg, percent_error(num_by_ne[ne_half]["KI"], KI_ana), lw=2, color=colors[ne_half], label=f"ne_half={ne_half}")
    ax3.set_xlabel(r"$\alpha$ [deg]")
    ax3.set_ylabel("Percent error [%]")
    ax3.set_title("K_I percent error")
    ax3.grid(True)
    ax3.legend()
    fig3.savefig(out_dir / f"K_I_percent_error_ne30_40_60_nseg{nseg_test}.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig4, ax4 = plt.subplots()
    for ne_half in ne_half_values:
        ax4.plot(alpha_deg, percent_error(num_by_ne[ne_half]["KII"], KII_ana), lw=2, color=colors[ne_half], label=f"ne_half={ne_half}")
    ax4.set_xlabel(r"$\alpha$ [deg]")
    ax4.set_ylabel("Percent error [%]")
    ax4.set_title("K_II percent error")
    ax4.grid(True)
    ax4.legend()
    fig4.savefig(out_dir / f"K_II_percent_error_ne30_40_60_nseg{nseg_test}.png", dpi=300, bbox_inches="tight")
    plt.show()



    # ---- next cell ----

    ## BRANCHED CRACK VALIDATION: Lo J. Applied Mechanics (1978) 45, p797
    # ============================================================
    # Symmetric Branched Crack (robust Euclidean near-tip fit)
    # ============================================================

    import os, sys
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt

    # Add repo root (one level above notebooks/) to sys.path
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))


    out_dir = Path(repo_root) / "output" / "Branched_Crack_Validation"
    out_dir.mkdir(parents=True, exist_ok=True)

    # ----------------------------
    # Problem definition
    # ----------------------------
    mm = 1e-3
    c = 1000.0 * mm
    # c_over_l = np.linspace(1, 201, 10, dtype=float)
    c_over_l = np.array([1,2,3,4,5,6,7,8,9,10,15,20,30,40,50, 75, 100, 200], float)
    theta_deg = 15.0
    theta = np.deg2rad(theta_deg)

    # Material and loading
    E = 200e9
    nu = 0.30
    plane = "strain"
    sig_xx = 0e6
    sig_yy = 100e6
    sig_xy = 0.0

    def print_meta(label, ratio, meta, l):
        print(
            f"[{label}] c/l={ratio:g} | "
            f"keys={list(meta.keys())} | "
            f"rr_size={meta.get('rr_size', 'NA')} | "
            f"rmax/l={meta.get('rmax', np.nan)/l if 'rmax' in meta else np.nan:.3f}"
        )

    # ----------------------------
    # Build branched crack network
    # ----------------------------
    def build_branched_crack(c, l, theta):
        vertices = np.array([
            [0,  -c,  0.0],
            [1,   c,  0.0],
            [2,   c + l*np.cos(theta),   l*np.sin(theta)],
            [3,   c + l*np.cos(theta),  -l*np.sin(theta)],
        ], dtype=float)

        connectivity = np.array([
            [0, 1],   # e0 main
            [1, 2],   # e1 top branch
            [1, 3],   # e2 bottom branch
        ], dtype=int)

        net = CrackNetworkV4.from_vertices_connectivity(
            vertices=vertices,
            connectivity=connectivity,
            Nv_max=4,
            validate=True
        )
        return net, vertices

    # ----------------------------
    # Run sweep over c/l ratios
    # ----------------------------
    KI_top, KII_top = [], []
    KI_bot, KII_bot = [], []
    KI_main, KII_main = [], []

    material = Material(E=E, nu=nu, plane_stress=False)
    applied = AppliedStress(sigma_xx=sig_xx, sigma_yy=sig_yy, sigma_xy=sig_xy)

    mu = float(material.mu)

    # Euclid-fit controls (robust defaults)
    rmax_frac_branch = 0.05   # branches can be short => allow a bit wider
    rmax_frac_main   = 0.05
    min_pts = 4
    two_term = True
    ne_half = 40

    for ratio in c_over_l:
        l = c / ratio

        print(f"Solving for c/l = {ratio:g} (l = {l*1e3:.3f} mm)...")

        net, vertices = build_branched_crack(c, l, theta)

        calc = DCENetworkStaticV4(material, net, applied)
        
        sol = calc.solve(
            ne_half=ne_half,
            crack_mode="half",
            representation="cheb_quad",
            node_distribution="uniform",      # recommended for non-tip segments in this strategy
            nq_col=12,
            nq_stress=16,
            solver_option="parametrized_crack",
            parametrization="polyline",
            junction_model="strict",

            # NEW (tip refinement)
            tip_min_nodes=6,                 # enforce >=6 nodes on each tip segment
            tip_cluster="power",              # "power" (one-sided) or "cheb" (symmetric)
            tip_cluster_power=3.0,            # stronger clustering as p increases (>=2)
            other_min_panels=4,               # baseline per segment
            endpoint_min_nodes=6,        # new
            kink_side_nodes=3,           # new
            refine_junction_endpoints=True,
            refine_kinks=True,
            )
        res = DCEResultsNetworkV4(calc, sol)

        # tip coordinates (global)
        p_top  = vertices[2, 1:3]
        p_bot  = vertices[3, 1:3]
        p_left = vertices[0, 1:3]

        # --- Top branch tip (edge 1, tip at v2)
        try:
            KI_t, KII_t, KI_t_CR, KII_t_CR, meta_t = euclid_tip_fit_from_edge(
                res, edge_index=1, tip_xy=p_top,
                a_fit=l, rmax_frac=rmax_frac_branch, min_pts=min_pts, two_term=two_term, theta=theta,
                material=material
            )
            
            KI_top.append(KI_t)
            KII_top.append(KII_t)
        except Exception as e:
            print(f"  Warning: Top branch fit failed: {e}")
            KI_top.append(np.nan); KII_top.append(np.nan)
        
        # --- Bottom branch tip (edge 2, tip at v3)
        try:
            KI_b, KII_b, KI_b_CR, KII_b_CR, meta_b = euclid_tip_fit_from_edge(
                res, edge_index=2, tip_xy=p_bot,
                a_fit=l, rmax_frac=rmax_frac_branch, min_pts=min_pts, two_term=two_term, theta=-theta,
                material=material
            )
            KI_bot.append(KI_b)
            KII_bot.append(KII_b)
        except Exception as e:
            print(f"  Warning: Bottom branch fit failed: {e}")
            KI_bot.append(np.nan); KII_bot.append(np.nan)

        # --- Main crack left tip (edge 0, tip at v0)
        try:
            KI_m, KII_m, KI_m_CR, KII_m_CR, meta_m = euclid_tip_fit_from_edge(
                res, edge_index=0, tip_xy=p_left,
                a_fit=c,
                rmax_frac=rmax_frac_main, min_pts=min_pts, two_term=two_term, theta=0.0,
                material=material
            )
            KI_main.append(KI_m)
            KII_main.append(KII_m)
        except Exception as e:
            print(f"  Warning: Main crack fit failed: {e}")
            KI_main.append(np.nan); KII_main.append(np.nan)
            
    # Convert to arrays
    KI_top = np.array(KI_top);   KII_top = np.array(KII_top)
    KI_bot = np.array(KI_bot);   KII_bot = np.array(KII_bot)
    KI_main = np.array(KI_main); KII_main = np.array(KII_main)

    # Normalize by reference SIF
    K_ref = sig_yy * np.sqrt(np.pi * c)
    KI_branch_norm = KI_top / K_ref
    KII_branch_norm = KII_top / K_ref
    KI_main_norm = KI_main / K_ref
    KII_main_norm = KII_main / K_ref

    # -------------------------
    # Lo (JAM 1978) digitized data (branch SIFs)
    # -------------------------
    lo_KI = np.array([
        [0.977157764, 0.888603531],
        [1.22156035 , 0.858908507],
        [1.754476973, 0.820385233],
        [2.494862119, 0.792295345],
        [4.855432594, 0.755377207],
        [8.929324781, 0.736918138],
        [20.46800156, 0.728892456],
        [60.6877622 , 0.728089888],
        [178.2899264, 0.727287319],
    ], float)

    lo_KII = np.array([
        [0.995465377, 0.093361835],
        [1.306903865, 0.076523857],
        [1.683851068, 0.061291159],
        [2.450969943, 0.040453082],
        [3.918340794, 0.017222372],
        [10.20229795, -0.021991258],
        [31.44051859, -0.053922155],
        [84.94829929, -0.07220161],
    ], float)

    lo_cl_KI,  lo_KI_norm  = lo_KI[:,0],  lo_KI[:,1]
    lo_cl_KII, lo_KII_norm = lo_KII[:,0], lo_KII[:,1]

    def interp_logx(xq, x, y):
        """
        Interpolate y(x) onto query points xq using linear interpolation in log10(x).
        Assumes x > 0 and sorted ascending.
        """
        xq = np.asarray(xq, float)
        x  = np.asarray(x, float)
        y  = np.asarray(y, float)

        m = (xq > 0) & (xq >= x.min()) & (xq <= x.max())
        yq = np.full_like(xq, np.nan, dtype=float)
        yq[m] = np.interp(np.log10(xq[m]), np.log10(x), y)
        return yq

    loKI_on_grid  = interp_logx(c_over_l, lo_cl_KI,  lo_KI_norm)
    loKII_on_grid = interp_logx(c_over_l, lo_cl_KII, lo_KII_norm)

    # -------------------------
    # Plots
    # -------------------------
    fig1, ax1 = plt.subplots(figsize=(10, 6))
    # Numerical (solid line, no symbols)
    ax1.plot(c_over_l, KI_branch_norm, '-', lw=2.5, label=f'VCM (branch, ne_half={ne_half})')
    # Lo overlay: blue circle edge, red fill, no line
    ax1.plot(
        lo_cl_KI, lo_KI_norm,
        linestyle='None',
        marker='o', markersize=15,
        markerfacecolor='red',
        markeredgecolor='blue',
        markeredgewidth=1.5,
        label="Lo (JAM 1978)"
    )
    ax1.set_xlabel(r'$c/l$', fontsize=12)
    ax1.set_ylabel(r'$K_I / (\sigma_\infty \sqrt{\pi c})$', fontsize=12)
    ax1.set_title(f'Branch tip: Normalized $K_I$ (θ={theta_deg}°)', fontsize=14, weight='bold')
    ax1.set_ylim(bottom=0.0)  # <-- start y-axis at 0 for KI
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    plt.tight_layout()
    fig1.savefig(out_dir / "branched_KI_vs_Lo.png", dpi=150, bbox_inches='tight')
    plt.show()

    fig2, ax2 = plt.subplots(figsize=(10, 6))
    ax2.plot(c_over_l, KII_branch_norm, '-', lw=2.5, label=f'VCM (branch, ne_half={ne_half})')
    ax2.plot(
        lo_cl_KII, lo_KII_norm,
        linestyle='None',
        marker='o', markersize=15,
        markerfacecolor='red',
        markeredgecolor='blue',
        markeredgewidth=1.5,
        label="Lo (JAM 1978)"
    )
    ax2.set_xlabel(r'$c/l$', fontsize=12)
    ax2.set_ylabel(r'$K_{II} / (\sigma_\infty \sqrt{\pi c})$', fontsize=12)
    ax2.set_title(f'Branch tip: Normalized $K_{{II}}$ (θ={theta_deg}°)', fontsize=14, weight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    plt.tight_layout()
    fig2.savefig(out_dir / "branched_KII_vs_Lo.png", dpi=150, bbox_inches='tight')
    plt.show()


    fig3, ax3 = plt.subplots(figsize=(10, 6))
    ax3.plot(c_over_l, KI_main_norm, 'o-', lw=2, markersize=15, 
             markerfacecolor='red',markeredgecolor='blue', markeredgewidth=1.5,
             label=f'Main crack left tip (ne_half={ne_half})')
    ax3.axhline(1.0, color='k', linestyle='--', lw=2, alpha=0.5, label='Unbranched crack: 1.0')
    ax3.set_xlabel(r'$c/l$', fontsize=12)
    ax3.set_ylabel(r'$K_I / (\sigma_\infty \sqrt{\pi c})$', fontsize=12)
    ax3.set_title(f'Main Crack: Normalized $K_I$ vs $c/l$ (θ={theta_deg}°)', fontsize=14, weight='bold')
    ax3.set_xscale('linear'); ax3.grid(True, alpha=0.3); ax3.legend()
    plt.tight_layout()
    fig3.savefig(out_dir / "main_crack_KI_normalized_lin.png", dpi=150, bbox_inches='tight')
    plt.show()

    err_KI_pct  = 100.0 * (KI_branch_norm  - loKI_on_grid)  / loKI_on_grid
    err_KII_pct = 100.0 * (np.abs(KII_branch_norm) - np.abs(loKII_on_grid)) / np.abs(loKII_on_grid)

    # Optional: mask where Lo isn't defined over your c/l range
    mKI  = np.isfinite(err_KI_pct)
    mKII = np.isfinite(err_KII_pct)

    fige4, axe1 = plt.subplots(figsize=(10, 5))
    axe1.plot(c_over_l[mKI], err_KI_pct[mKI], '-', lw=2.5)
    axe1.axhline(0.0, color='k', lw=1, alpha=0.5)
    axe1.set_xlabel(r'$c/l$', fontsize=12)
    axe1.set_ylabel(r'$\%$ error in $K_I$', fontsize=12)
    axe1.set_title(r'Branch tip: $\%$ error vs Lo (JAM 1978) — $K_I$', fontsize=13, weight='bold')
    axe1.grid(True, alpha=0.3)
    plt.tight_layout()
    fige4.savefig(out_dir / "branched_KI_percent_error_vs_Lo.png", dpi=150, bbox_inches='tight')
    plt.show()

    # fige5, axe2 = plt.subplots(figsize=(10, 5))
    # axe2.plot(c_over_l[mKII], err_KII_pct[mKII], '-', lw=2.5)
    # axe2.axhline(0.0, color='k', lw=1, alpha=0.5)
    # axe2.set_xlabel(r'$c/l$', fontsize=12)
    # axe2.set_ylabel(r'$\%$ error in $K_{II}$', fontsize=12)
    # axe2.set_title(r'Branch tip: $\%$ error vs Lo (JAM 1978) — $K_{II}$', fontsize=13, weight='bold')
    # axe2.grid(True, alpha=0.3)
    # plt.tight_layout()
    # fige5.savefig(out_dir / "branched_KII_percent_error_vs_Lo.png", dpi=150, bbox_inches='tight')
    # plt.show()


    print("\n" + "="*60)
    print("Branched crack validation complete!")
    print(f"Results saved to: {out_dir}")
    print("="*60)


    # ---- next cell ----

    # ============================================================
    # Kinked crack propagation demo
    # - single solver_kwargs source of truth
    # - plots only initial and final states
    # ============================================================

    import os, sys
    from pathlib import Path

    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))


    out_dir = Path(repo_root) / "output" / "KinkedCrackPropagation"
    out_dir.mkdir(parents=True, exist_ok=True)

    # ----------------------------
    # Build initial network
    # ----------------------------
    mm = 1e-3
    vertices = np.array([
        [0,   -5.0*mm,  -3.0*mm],
        [1,    5.0*mm,   3.0*mm],
    ], dtype=float)

    connectivity = np.array([[0, 1]], dtype=int)

    net0 = CrackNetworkV4.from_vertices_connectivity(
        vertices=vertices,
        connectivity=connectivity,
        Nv_max=4,
        validate=True
    )

    material = Material(E=200e9, nu=0.30, plane_stress=False)
    applied  = AppliedStress(sigma_xx=100e6, sigma_yy=0.0e6, sigma_xy=0.0)

    # ----------------------------
    # ONE solver kwargs dict (used everywhere)
    # ----------------------------
    solver_kwargs = dict(
        ne_half=80,
        representation="cheb_quad",
        node_distribution="tip_dense",
        solver_option="parametrized_crack",
        parametrization="polyline",
        nq_col=12,
        nq_stress=16,
        crack_mode="half",
        junction_model="core",
        # refinement rules
        refine_kinks=False,
        refine_junction_endpoints=False,
        # how much refinement
        endpoint_min_nodes=4,     # was effectively ~6
        kink_side_nodes=1,         # was 2 in current solver defaults
        # clustering shape
        tip_cluster="power",
        tip_cluster_power=2.5,
        # keep other segments from getting too coarse
        other_min_panels=4,
    )

    # ----------------------------
    # Solve + plot (only when called)
    # ----------------------------
    def solve_and_plot(network, tag: str):
        step_dir = out_dir / tag
        step_dir.mkdir(parents=True, exist_ok=True)

        calc = DCENetworkStaticV4(material, network, applied)
        sol  = calc.solve(**solver_kwargs)
        res  = DCEResultsNetworkV4(calc, sol)

        plotter = DCEPlotterV4(res, out_dir=step_dir)
        _call(plotter, "plot_network_graph")

        opts = StressPlotOptsV4(
            extent_factor=3,
            n_grid=301,
            add_remote=True,
            mask_cracks=False,
            cmap="jet",
            n_bands=40,
            label_contours=False,
            vmax_factor=2,
        )
        _call(plotter, "plot_stress_components_global", opts=opts, components=("sxx", "syy", "sxy"))

        pl_def = DCEPlotterDeformedV4(res, out_dir=step_dir)
        _call(pl_def, "plot_deformed_network", scale=5e1, trim_core_junction_faces=True)

        plt.show()
        print(f"[{tag}] Saved figures to: {step_dir}")
        return res

    # ----------------------------
    # Propagation setup
    # ----------------------------
    cfg = PropagationConfig(f0=0.10, f_fixed=0.1, 
                            step_mode="fixed_step", simultaneous_tip_growth=True)

    Kc_demo = 1e6  # Pa*sqrt(m)
    tough = ConstantToughness(Kc_demo)

    dir_law = MaximumHoopStressLaw()

    evaluator = CandidateEvaluator(
        
        material=material,
        applied=applied,
        solver_kwargs=solver_kwargs,   # <-- same dict
        direction_law=dir_law,
        enable_ne_half_escalation=False,  # <-- allow more accurate SIFs during propagation if needed    
    )

    evaluator.rmax_frac = 0.25
    evaluator.min_pts   = 8


    prop = CrackPropagator(
        cfg=cfg,
        evaluator=evaluator,
        toughness=tough,
        direction_law=dir_law,
    )

    # ----------------------------
    # Plot INITIAL
    # ----------------------------
    solve_and_plot(net0, tag="step_00_initial")

    # ----------------------------
    # Propagate (NO plotting during loop)
    # ----------------------------
    n_steps = 12
    net = net0

    for k in range(1, n_steps + 1):
        result = prop.grow_one_increment(net)
        net = result.network_new

        grew = sum(1 for r in result.reports if bool(getattr(r, "grew", False)))
        print(f"[step_{k:02d}] grown_tips={grew}")

        if grew == 0:
            print("No tips exceeded toughness. Stopping.")
            break

    # ----------------------------
    # Plot FINAL
    # ----------------------------
    solve_and_plot(net, tag=f"step_{k:02d}_final")
    print("Saved figures under:", out_dir)


    # ---- next cell ----

    # ============================================================
    # Kinked crack propagation demo + periodic simplification cycles
    # - uses YOUR CrackNetworkSimplifier module (no re-implementation)
    # - saves/plots at end of each outer cycle (pre/post simplify)
    # ============================================================

    import os, sys
    from pathlib import Path
    from collections import defaultdict, deque

    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))


    # ---- USE YOUR MODULE (no scratch code)
    from fracture_utils.Ugenerator.crack_network_simplifier import CrackNetworkSimplifier, SimplificationConfig

    out_dir = Path(repo_root) / "output" / "KinkedCrackPropagation"
    out_dir.mkdir(parents=True, exist_ok=True)

    # ----------------------------
    # Build initial network
    # ----------------------------
    mm = 1e-3
    vertices = np.array([
        [0,   -5.0*mm,  -3.0*mm],
        [1,    5.0*mm,   3.0*mm],
    ], dtype=float)

    connectivity = np.array([[0, 1]], dtype=int)

    net0 = CrackNetworkV4.from_vertices_connectivity(
        vertices=vertices,
        connectivity=connectivity,
        Nv_max=4,          # NOTE: in your codebase this is NOT a vertex cap (you said it's max junction degree)
        validate=True
    )

    material = Material(E=200e9, nu=0.30, plane_stress=False)
    applied  = AppliedStress(sigma_xx=100e6, sigma_yy=0.0e6, sigma_xy=0.0)

    # ----------------------------
    # ONE solver kwargs dict (used everywhere)
    # ----------------------------
    solver_kwargs = dict(
        ne_half=80,
        representation="cheb_quad",
        node_distribution="tip_dense",
        solver_option="parametrized_crack",
        parametrization="polyline",
        nq_col=12,
        nq_stress=16,
        crack_mode="half",
        junction_model="core",
        # refinement rules
        refine_kinks=False,
        refine_junction_endpoints=False,
        # how much refinement
        endpoint_min_nodes=4,
        kink_side_nodes=1,
        # clustering shape
        tip_cluster="power",
        tip_cluster_power=2.5,
        # keep other segments from getting too coarse
        other_min_panels=4,
    )

    # ----------------------------
    # Solve + plot (only when called)
    # ----------------------------
    def solve_and_plot(network, tag: str):
        step_dir = out_dir / tag
        step_dir.mkdir(parents=True, exist_ok=True)

        calc = DCENetworkStaticV4(material, network, applied)
        sol  = calc.solve(**solver_kwargs)
        res  = DCEResultsNetworkV4(calc, sol)

        plotter = DCEPlotterV4(res, out_dir=step_dir)
        _call(plotter, "plot_network_graph")

        opts = StressPlotOptsV4(
            extent_factor=3,
            n_grid=301,
            add_remote=True,
            mask_cracks=False,
            cmap="jet",
            n_bands=40,
            label_contours=False,
            vmax_factor=2,
        )
        _call(plotter, "plot_stress_components_global", opts=opts, components=("sxx", "syy", "sxy"))

        pl_def = DCEPlotterDeformedV4(res, out_dir=step_dir)
        _call(pl_def, "plot_deformed_network", scale=5e1, trim_core_junction_faces=True)

        plt.show()
        print(f"[{tag}] Saved figures to: {step_dir}")
        return res

    # ----------------------------
    # Propagation setup
    # ----------------------------
    cfg = PropagationConfig(f0=0.10, f_fixed=0.1,
                            step_mode="fixed_step", simultaneous_tip_growth=True)

    Kc_demo = 1e6  # Pa*sqrt(m)
    tough = ConstantToughness(Kc_demo)

    dir_law = MaximumHoopStressLaw()

    evaluator = CandidateEvaluator(
        material=material,
        applied=applied,
        solver_kwargs=solver_kwargs,          # same dict
        direction_law=dir_law,
        enable_ne_half_escalation=False,
    )
    evaluator.rmax_frac = 0.25
    evaluator.min_pts   = 8

    prop = CrackPropagator(
        cfg=cfg,
        evaluator=evaluator,
        toughness=tough,
        direction_law=dir_law,
    )

    # ============================================================
    # Helpers: export network + chain length (single polyline)
    # ============================================================
    def net_to_arrays(net: CrackNetworkV4):
        V = np.array([[int(v.id), float(v.x), float(v.y)] for v in net.vertices], dtype=float)
        C = np.array([[int(e.v0), int(e.v1)] for e in net.edges], dtype=int)
        return V, C

    def chain_order_from_connectivity(C):
        adj = defaultdict(list)
        deg = defaultdict(int)
        for a, b in C:
            a = int(a); b = int(b)
            adj[a].append(b); adj[b].append(a)
            deg[a] += 1; deg[b] += 1
        tips = [vid for vid, d in deg.items() if d == 1]
        start = tips[0] if tips else int(C[0, 0])
        order = []
        prev = None
        cur = start
        while True:
            order.append(cur)
            nbrs = adj[cur]
            nxt = None
            if prev is None:
                nxt = nbrs[0] if nbrs else None
            else:
                cand = [x for x in nbrs if x != prev]
                nxt = cand[0] if cand else None
            if nxt is None:
                break
            prev, cur = cur, nxt
            if len(order) > len(adj) + 5:
                break
        return order

    def polyline_length_mm_from_arrays(V, C):
        if C.size == 0:
            return 0.0
        xy = {int(row[0]): np.array([row[1], row[2]], float) for row in V}
        order = chain_order_from_connectivity(C)
        L = 0.0
        for i in range(len(order) - 1):
            L += float(np.linalg.norm(xy[order[i+1]] - xy[order[i]]))
        return L / mm

    # ============================================================
    # Outer-cycle controls
    # ============================================================
    vertex_limit = 20        # stop inner loop when this many vertices reached
    L_limit_mm   = 200.0     # stop ALL when length reaches this
    max_cycles   = 50       # safety

    simp_cfg = SimplificationConfig(
        max_angle_deviation=3.0,
        min_edge_length=0.25*mm,
        remove_degree2_nodes=False,
        merge_at_degree2=True,
        merge_vertex_tolerance=1e-6,
        max_merged_edge_length=10*mm,
        preserve_tips=True,
        preserve_junctions=True
    )

    # ----------------------------
    # Plot INITIAL
    # ----------------------------
    solve_and_plot(net0, tag="cycle_00_initial")

    # ----------------------------
    # Outer-cycle loop
    # ----------------------------
    net = net0
    global_step = 0


    for cyc in range(1, max_cycles + 1):

        # ---- inner loop: run until vertex_limit or no-growth or length limit
        vertex_high = 20   # trigger simplify when reached/exceeded
        vertex_low  = 12   # desired post-simplify size (guide; simplifier may not hit exactly)

        did_grow_this_cycle = False

        while True:
            # Stop by length first
            V_now, C_now = net_to_arrays(net)
            Lmm = polyline_length_mm_from_arrays(V_now, C_now)
            if Lmm >= L_limit_mm:
                print(f"[cycle_{cyc:02d}] reached length limit: L={Lmm:.2f} mm")
                break

            # --- Always attempt a growth step first ---
            result = prop.grow_one_increment(net)
            net = result.network_new

            grew = sum(1 for r in result.reports if bool(getattr(r, "grew", False)))
            global_step += 1
            did_grow_this_cycle |= (grew > 0)

            # recompute nV AFTER growth
            V_now, C_now = net_to_arrays(net)
            nV = V_now.shape[0]
            Lmm = polyline_length_mm_from_arrays(V_now, C_now)

            print(f"[cycle_{cyc:02d} step_{global_step:04d}] grown_tips={grew} | nV={nV} | L={Lmm:.2f} mm")

            if grew == 0:
                print(f"[cycle_{cyc:02d}] No tips exceeded toughness. Stopping.")
                break

            # Trigger simplification only AFTER we’ve actually grown
            if nV >= vertex_high:
                print(f"[cycle_{cyc:02d}] hit high-water vertex threshold: nV={nV} >= {vertex_high}")
                break

        V_post, C_post = simplifier.to_arrays()
        print(f"[cycle_{cyc:02d}] post-simplify nV={len(V_post)} (target ~{vertex_low})")

        # ---- end-of-cycle outputs (PRE-simplify)
        V_pre, C_pre = net_to_arrays(net)
        Lmm_pre = polyline_length_mm_from_arrays(V_pre, C_pre)

        cyc_dir = out_dir / f"cycle_{cyc:02d}"
        cyc_dir.mkdir(parents=True, exist_ok=True)

        np.savez(cyc_dir / f"network_pre_simplify_step_{global_step:04d}.npz",
                 vertices=V_pre, connectivity=C_pre)

        solve_and_plot(net, tag=f"cycle_{cyc:02d}_step_{global_step:04d}_pre_simplify")

        # stop all cycles if length reached or no growth
        if Lmm_pre >= L_limit_mm:
            print(f"[cycle_{cyc:02d}] terminating outer loop (length reached).")
            break

        # If the last inner step ended due to no-growth, stop (do not simplify)
        if 'result' in locals():
            grew_last = sum(1 for r in result.reports if bool(getattr(r, "grew", False)))
            if grew_last == 0:
                print(f"[cycle_{cyc:02d}] terminating outer loop (no growth).")
                break

        # ---- simplify using YOUR MODULE
        simplifier = CrackNetworkSimplifier(config=simp_cfg)
        simplifier.load_from_arrays(V_pre, C_pre)
        simp_stats = simplifier.simplify(verbose=True)
        V_post, C_post = simplifier.to_arrays()

        np.savez(cyc_dir / f"network_post_simplify_step_{global_step:04d}.npz",
                 vertices=V_post, connectivity=C_post)

        # rebuild CrackNetworkV4 from simplified arrays and continue
        net = CrackNetworkV4.from_vertices_connectivity(
            vertices=V_post,
            connectivity=C_post,
            Nv_max=4,        # keep your junction-degree cap consistent
            validate=True
        )

        solve_and_plot(net, tag=f"cycle_{cyc:02d}_step_{global_step:04d}_post_simplify")

    print("Saved figures under:", out_dir)


    # ---- next cell ----

    # ============================================================
    # Two kinked cracks propagation demo
    # - single solver_kwargs source of truth
    # - plots only initial and final states
    # ============================================================

    import os, sys
    from pathlib import Path

    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))


    out_dir = Path(repo_root) / "output" / "KinkedCrackPropagation"
    out_dir.mkdir(parents=True, exist_ok=True)

    # ----------------------------
    # Build initial network
    # ----------------------------
    mm = 1e-3
    vertices = np.array([
        [0,  -5.0*mm,  -3.0*mm],
        [1 ,  65.0*mm, -10*mm],
        [2,  80.0*mm, -40.0*mm],
        [3,  120.0*mm, -80.0*mm],
        [4, 60.0*mm, -60.0*mm]
    ], dtype=float)

    connectivity = np.array([
        [0, 1],   # e0
        [1, 2],
        [2, 3],
        [2, 4],
    ], dtype=int)


    net0 = CrackNetworkV4.from_vertices_connectivity(
        vertices=vertices,
        connectivity=connectivity,
        Nv_max=4,
        validate=True
    )

    material = Material(E=200e9, nu=0.30, plane_stress=False)
    applied  = AppliedStress(sigma_xx=100e6, sigma_yy=0.0e6, sigma_xy=0.0)

    # ----------------------------
    # ONE solver kwargs dict (used everywhere)
    # ----------------------------
    solver_kwargs = dict(
        ne_half=60,
        representation="cheb_quad",
        node_distribution="tip_dense",
        solver_option="parametrized_crack",
        parametrization="polyline",
        nq_col=12,
        nq_stress=16,
        crack_mode="half",
        junction_model="core",
        # refinement rules
        refine_kinks=True,
        refine_junction_endpoints=True,
        # how much refinement
        endpoint_min_nodes=4,     # was effectively ~6
        kink_side_nodes=1,         # was 2 in current solver defaults
        # clustering shape
        tip_cluster="power",
        tip_cluster_power=2.5,
        # keep other segments from getting too coarse
        other_min_panels=4,
    )

    # ----------------------------
    # Solve + plot (only when called)
    # ----------------------------
    def solve_and_plot(network, tag: str):
        step_dir = out_dir / tag
        step_dir.mkdir(parents=True, exist_ok=True)

        calc = DCENetworkStaticV4(material, network, applied)
        sol  = calc.solve(**solver_kwargs)
        res  = DCEResultsNetworkV4(calc, sol)

        plotter = DCEPlotterV4(res, out_dir=step_dir)
        _call(plotter, "plot_network_graph")

        opts = StressPlotOptsV4(
            extent_factor=3,
            n_grid=301,
            add_remote=True,
            mask_cracks=False,
            cmap="jet",
            n_bands=40,
            label_contours=False,
            vmax_factor=2,
        )
        _call(plotter, "plot_stress_components_global", opts=opts, components=("sxx", "syy", "sxy"))

        pl_def = DCEPlotterDeformedV4(res, out_dir=step_dir)
        _call(pl_def, "plot_deformed_network", scale=5e1, trim_core_junction_faces=True)

        plt.show()
        print(f"[{tag}] Saved figures to: {step_dir}")
        return res

    # ----------------------------
    # Propagation setup
    # ----------------------------
    cfg = PropagationConfig(f0=0.10, f_fixed=0.1, 
                            step_mode="fixed_step", simultaneous_tip_growth=True)

    Kc_demo = 1e6  # Pa*sqrt(m)
    tough = ConstantToughness(Kc_demo)

    dir_law = MaximumHoopStressLaw()


    evaluator = CandidateEvaluator(
        
        material=material,
        applied=applied,
        solver_kwargs=solver_kwargs,   # <-- same dict
        direction_law=dir_law,
        enable_ne_half_escalation=False,  # <-- allow more accurate SIFs during propagation if needed    
    )

    prop = CrackPropagator(
        cfg=cfg,
        evaluator=evaluator,
        toughness=tough,
        direction_law=dir_law,
    )

    # ----------------------------
    # Plot INITIAL
    # ----------------------------
    solve_and_plot(net0, tag="step_00_initial")

    # ----------------------------
    # Propagate (NO plotting during loop)
    # ----------------------------
    n_steps = 15
    net = net0

    for k in range(1, n_steps + 1):
        result = prop.grow_one_increment(net)
        net = result.network_new

        grew = sum(1 for r in result.reports if bool(getattr(r, "grew", False)))
        print(f"[step_{k:02d}] grown_tips={grew}")

        if grew == 0:
            print("No tips exceeded toughness. Stopping.")
            break

    # ----------------------------
    # Plot FINAL
    # ----------------------------
    solve_and_plot(net, tag=f"step_{k:02d}_final")
    print("Saved figures under:", out_dir)


    # ---- next cell ----

    # ============================================================
    # Two kinked cracks propagation demo
    # - multi-component, K-scaled fixed-step growth
    # ============================================================

    import os, sys
    from pathlib import Path
    import numpy as np

    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))


    # IMPORTANT: use your updated network_ops
    from fracture_utils.Upropagation.network_ops import extract_open_polylines

    out_dir = Path(repo_root) / "output" / "KinkedCrackPropagation"
    out_dir.mkdir(parents=True, exist_ok=True)

    # ----------------------------
    # Build initial network
    # ----------------------------
    mm = 1e-3
    vertices = np.array([
        [0,  -5.0*mm,  -3.0*mm],
        [1 ,  65.0*mm, -10*mm],
        [2,  80.0*mm, -40.0*mm],
        [3, 120.0*mm, -80.0*mm],
        [4,  60.0*mm, -60.0*mm],
    ], dtype=float)

    connectivity = np.array([
        [0, 1],
        [1, 2],
        [2, 3],
        [2, 4],
    ], dtype=int)

    net = CrackNetworkV4.from_vertices_connectivity(
        vertices=vertices,
        connectivity=connectivity,
        Nv_max=4,
        validate=True
    )

    material = Material(E=200e9, nu=0.30, plane_stress=False)
    applied  = AppliedStress(sigma_xx=100e6, sigma_yy=0.0, sigma_xy=0.0)

    # ----------------------------
    # Solver kwargs
    # ----------------------------
    solver_kwargs = dict(
        ne_half=60,
        representation="cheb_quad",
        node_distribution="tip_dense",
        solver_option="parametrized_crack",
        parametrization="polyline",
        nq_col=12,
        nq_stress=16,
        crack_mode="half",
        junction_model="core",
        refine_kinks=True,
        refine_junction_endpoints=True,
        endpoint_min_nodes=4,
        kink_side_nodes=1,
        tip_cluster="power",
        tip_cluster_power=2.5,
        other_min_panels=4,
    )

    # ----------------------------
    # Plot helper
    # ----------------------------
    def solve_and_plot(network, tag):
        step_dir = out_dir / tag
        step_dir.mkdir(parents=True, exist_ok=True)

        calc = DCENetworkStaticV4(material, network, applied)
        sol  = calc.solve(**solver_kwargs)
        res  = DCEResultsNetworkV4(calc, sol)

        plotter = DCEPlotterV4(res, out_dir=step_dir)
        _call(plotter, "plot_network_graph")

        pl_def = DCEPlotterDeformedV4(res, out_dir=step_dir)
        _call(pl_def, "plot_deformed_network", scale=5e1, trim_core_junction_faces=True)

        plt.show()
        return res

    # ----------------------------
    # Propagation setup
    # ----------------------------
    cfg = PropagationConfig(
        f0=0.10,
        f_fixed=0.10,
        step_mode="fixed_step",
        simultaneous_tip_growth=True
    )

    tough = ConstantToughness(1e6)
    dir_law = MaximumHoopStressLaw()

    evaluator = CandidateEvaluator(
        material=material,
        applied=applied,
        solver_kwargs=solver_kwargs,
        direction_law=dir_law,
        enable_ne_half_escalation=False,
    )

    prop = CrackPropagator(
        cfg=cfg,
        evaluator=evaluator,
        toughness=tough,
        direction_law=dir_law,
    )

    # ----------------------------
    # INITIAL
    # ----------------------------
    solve_and_plot(net, "step_00_initial")

    # ============================================================
    # MULTI-COMPONENT SCALED GROWTH LOOP
    # ============================================================
    n_steps = 6

    from fracture_utils.Upropagation.network_ops import get_netops

    netops = get_netops()

    for k in range(1, n_steps + 1):

        deg = netops.degree_map(net)
        polylines = netops.extract_open_polylines(net)

        if not polylines:
            print("No polylines found.")
            break

        # ---- baseline solve ONCE
        base_res = evaluator.solve_results(net)

        poly_data = []

        for pl in polylines:
            tips = netops.extract_deg1_tips(net, [pl], deg) or []
            if not tips:
                continue

            # there can be 1 or 2 tips associated with a polyline (tip–tip or tip–junction)
            # We'll evaluate both and pick the larger Keff as that polyline’s representative driver.
            tip_keffs = []
            for tip in tips:
                tev = evaluator.eval_tip(base_res, tip)
                tip_keffs.append((tip, float(tev.keff), float(tev.theta)))

            tip, keff, theta = max(tip_keffs, key=lambda x: x[1])

            # polyline length from vids
            L = 0.0
            vmap = {int(v.id): np.array([float(v.x), float(v.y)]) for v in net.vertices}
            for i in range(len(pl.vids) - 1):
                a = int(pl.vids[i]); b = int(pl.vids[i+1])
                L += float(np.linalg.norm(vmap[b] - vmap[a]))

            poly_data.append(dict(poly=pl, tip=tip, keff=keff, theta=theta, length=L))

        if not poly_data:
            print("No active tips.")
            break

        # ---- reference component = longest
        ref = max(poly_data, key=lambda d: d["length"])
        L_ref = float(ref["length"])
        K_ref = max(float(ref["keff"]), 1e-12)

        ds_ref = cfg.f_fixed * L_ref

        # ---- apply growth (scaled by Keff ratio)
        updater = prop._get_updater()  # uses your internal updater
        net_new = net
        grew_any = False

        for d in poly_data:
            rel = float(d["keff"]) / K_ref
            ds = ds_ref * rel
            if ds <= 0.0:
                continue

            net_new = updater.extend_tip(
                net_new,
                d["tip"],
                theta=float(d["theta"]),
                delta_a=float(ds)
            )
            grew_any = True

        net = net_new

        print(
            f"[step_{k:02d}] grew={grew_any} | "
            f"L_ref={L_ref/mm:.2f} mm | ds_ref={ds_ref/mm:.3f} mm"
        )

        if not grew_any:
            print("No growth accepted. Stopping.")
            break


    # ----------------------------
    # FINAL
    # ----------------------------
    solve_and_plot(net, f"step_{k:02d}_final")
    print("Saved under:", out_dir)


    # ---- next cell ----

    # ============================================================
    # Kinked crack propagation demo (via network_growth.py)
    # + generator.py intersection/junction update at each outer cycle
    # - plots/saves at each outer step: pre-simplify, post-simplify, post-intersections
    # ============================================================

    import os, sys
    from pathlib import Path
    import numpy as np

    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))


    out_dir = Path(repo_root) / "output" / "KinkedCrackPropagation"
    out_dir.mkdir(parents=True, exist_ok=True)

    # ----------------------------
    # Build initial network (3 separate polylines)
    # ----------------------------
    mm = 1e-3
    vertices = np.array([
        [0,   -5.0*mm,  -3.0*mm],
        [1,    5.0*mm,   3.0*mm],
        [2,   -7.0*mm, -10*mm],
        [3,    8.0*mm,   4.0*mm],
        [4,    0.0*mm,   8.0*mm],
        [5,   10.0*mm,  -5.0*mm],
    ], dtype=float)

    connectivity = np.array([[0, 1], [2, 3], [4, 5]], dtype=int)

    net0 = CrackNetworkV4.from_vertices_connectivity(
        vertices=vertices,
        connectivity=connectivity,
        Nv_max=4,
        validate=True
    )

    material = Material(E=200e9, nu=0.30, plane_stress=False)
    applied  = AppliedStress(sigma_xx=100e6, sigma_yy=0.0e6, sigma_xy=0.0)

    # ----------------------------
    # ONE solver kwargs dict (used everywhere)
    # ----------------------------
    solver_kwargs = dict(
        ne_half=40,
        representation="cheb_quad",
        node_distribution="tip_dense",
        solver_option="parametrized_crack",
        parametrization="polyline",
        nq_col=12,
        nq_stress=16,
        crack_mode="half",
        junction_model="core",
        refine_kinks=False,
        refine_junction_endpoints=False,
        endpoint_min_nodes=4,
        kink_side_nodes=1,
        tip_cluster="power",
        tip_cluster_power=2.5,
        other_min_panels=4,
    )

    # ----------------------------
    # Solve + plot (unchanged)
    # ----------------------------
    def solve_and_plot(network, tag: str):
        step_dir = out_dir / tag
        step_dir.mkdir(parents=True, exist_ok=True)

        calc = DCENetworkStaticV4(material, network, applied)
        sol  = calc.solve(**solver_kwargs)
        res  = DCEResultsNetworkV4(calc, sol)

        plotter = DCEPlotterV4(res, out_dir=step_dir)
        _call(plotter, "plot_network_graph")

        opts = StressPlotOptsV4(
            extent_factor=3,
            n_grid=301,
            add_remote=True,
            mask_cracks=False,
            cmap="jet",
            n_bands=40,
            label_contours=False,
            vmax_factor=2,
        )
        _call(plotter, "plot_stress_components_global", opts=opts, components=("sxx", "syy", "sxy"))

        pl_def = DCEPlotterDeformedV4(res, out_dir=step_dir)
        _call(pl_def, "plot_deformed_network", scale=5e1, trim_core_junction_faces=True)

        plt.show()
        print(f"[{tag}] Saved figures to: {step_dir}")
        return res

    # ----------------------------
    # Propagation setup
    # ----------------------------
    cfg = PropagationConfig(
        f0=0.1,
        f_fixed=0.08,
        step_mode="fixed_step",
        simultaneous_tip_growth=True
    )

    tough = ConstantToughness(1e6)
    dir_law = MaximumHoopStressLaw()

    evaluator = CandidateEvaluator(
        material=material,
        applied=applied,
        solver_kwargs=solver_kwargs,
        direction_law=dir_law,
        enable_ne_half_escalation=False,
    )
    evaluator.rmax_frac = 0.25
    evaluator.min_pts   = 8

    prop = CrackPropagator(
        cfg=cfg,
        evaluator=evaluator,
        toughness=tough,
        direction_law=dir_law,
    )

    # ============================================================
    # NEW: intersection/junction update using generator.py
    # ============================================================
    from fracture_utils.Ugenerator.generator import CrackNetworkGenerator
    import networkx as nx

    def _net_to_arrays(net: CrackNetworkV4):
        V = np.array([[int(v.id), float(v.x), float(v.y)] for v in net.vertices], dtype=float)
        C = np.array([[int(e.v0), int(e.v1)] for e in net.edges], dtype=int)
        return V, C

    def update_network_with_intersections(net: CrackNetworkV4,
                                          detect_mode: str = "single_pass",
                                          verbose: bool = True) -> CrackNetworkV4:
        """
        Adapt CrackNetworkV4 -> CrackNetworkGenerator.G, run detect_and_split_intersections(),
        then convert back to CrackNetworkV4.
        """
        V, C = _net_to_arrays(net)

        # pick a domain that safely contains the current geometry
        xs = V[:, 1]; ys = V[:, 2]
        w = float(xs.max() - xs.min()) if V.shape[0] else 1.0
        h = float(ys.max() - ys.min()) if V.shape[0] else 1.0
        pad = 0.25 * max(w, h) + 1e-6
        domain_size = (w + 2*pad, h + 2*pad)

        gen = CrackNetworkGenerator(domain_size=domain_size, seed=None)
        gen.G = nx.Graph()

        # load nodes (keep original ids as node keys)
        for row in V:
            vid = int(row[0])
            gen.G.add_node(vid, pos=(float(row[1]), float(row[2])))

        # load edges + lengths
        pos = nx.get_node_attributes(gen.G, "pos")
        for a, b in C:
            a = int(a); b = int(b)
            pa = np.array(pos[a], float)
            pb = np.array(pos[b], float)
            L = float(np.linalg.norm(pb - pa))
            gen.G.add_edge(a, b, length=L)

        gen._update_vertex_types()

        # split intersections -> adds new junction nodes
        n_int = gen.detect_and_split_intersections(verbose=verbose, mode=detect_mode)
        if verbose:
            print(f"Intersection update: processed {n_int} intersection(s)")

        Vg, Cg, _types = gen.to_arrays()
        C2 = Cg[:, 1:3].astype(int) if Cg.size else np.zeros((0, 2), dtype=int)

        # rebuild CrackNetworkV4
        net_new = CrackNetworkV4.from_vertices_connectivity(
            vertices=Vg,
            connectivity=C2,
            Nv_max=4,
            validate=True
        )
        return net_new

    # ============================================================
    # NEW: Simplification (your module)
    # ============================================================
    from fracture_utils.Ugenerator.crack_network_simplifier import CrackNetworkSimplifier, SimplificationConfig

    # Use conservative simplification to preserve features & junctions
    simp_cfg = SimplificationConfig(
        max_angle_deviation=5,      # << more conservative than 5°
        min_edge_length=0.50*mm,      # << keep small features unless truly tiny
        merge_at_degree2=True,
        remove_degree2_nodes=False,  # << keep degree-2 nodes to preserve geometry (unless merged by proximity)
        merge_vertex_tolerance=1*mm,  # << only merge if very close (helps preserve geometry)
        max_merged_edge_length=20.0*mm,   # << prevent collapsing long-ish geometry
        preserve_tips=True,
        preserve_junctions=True
    )

    def simplify_network(net: CrackNetworkV4, verbose=True) -> CrackNetworkV4:
        V, C = _net_to_arrays(net)
        simplifier = CrackNetworkSimplifier(config=simp_cfg)
        simplifier.load_from_arrays(V, C)
        simplifier.simplify(verbose=verbose)
        V2, C2 = simplifier.to_arrays()

        C2 = np.asarray(C2)

        # --- normalize to shape (Ne,2) = [v0,v1]
        if C2.ndim != 2:
            raise ValueError(f"simplifier connectivity must be 2D, got shape {C2.shape}")

        if C2.shape[1] == 2:
            C_use = C2.astype(int)
        elif C2.shape[1] == 3:
            # assume (eid, v0, v1) and drop eid
            C_use = C2[:, 1:3].astype(int)
        else:
            raise ValueError(f"simplifier connectivity has unsupported shape {C2.shape}")

        net2 = CrackNetworkV4.from_vertices_connectivity(
            vertices=V2,
            connectivity=C_use,
            Nv_max=4,
            validate=True
        )
        return net2


    # ============================================================
    # Outer-cycle loop controls
    # ============================================================
    max_cycles  = 6
    max_inner_steps = 6

    vertex_high = 25     # trigger simplify when reached/exceeded
    # (vertex_low is not enforced automatically unless your simplifier has a target; we just report)
    vertex_low  = 15

    # ----------------------------
    # INITIAL: run generator intersection discovery once
    # ----------------------------
    net = update_network_with_intersections(net0, detect_mode="iterative", verbose=True)
    solve_and_plot(net, tag="cycle_00_initial_post_intersections")

    global_step = 0

    for cyc in range(1, max_cycles + 1):

        # ----------------------------
        # INNER growth loop
        # ----------------------------
        did_grow = False

        for _ in range(max_inner_steps):
            result = prop.grow_one_increment(net)
            net = result.network_new

            grew = sum(1 for r in result.reports if bool(getattr(r, "grew", False)))
            global_step += 1
            did_grow |= (grew > 0)

            V_now, C_now = _net_to_arrays(net)
            nV = V_now.shape[0]

            print(f"[cycle_{cyc:02d} step_{global_step:04d}] grown_tips={grew} | nV={nV}")

            if grew == 0:
                print(f"[cycle_{cyc:02d}] No tips exceeded toughness. Stopping.")
                break

            if nV >= vertex_high:
                print(f"[cycle_{cyc:02d}] hit vertex_high: nV={nV} >= {vertex_high}")
                break

        # ----------------------------
        # PRE-SIMPLIFY plot/save
        # ----------------------------
        solve_and_plot(net, tag=f"cycle_{cyc:02d}_step_{global_step:04d}_pre_simplify")

        if not did_grow:
            print(f"[cycle_{cyc:02d}] no growth in this cycle -> stopping outer loop.")
            break

        # ----------------------------
        # SIMPLIFY (then plot/save)
        # ----------------------------
        net = simplify_network(net, verbose=True)
        V_s, _ = _net_to_arrays(net)
        print(f"[cycle_{cyc:02d}] post-simplify nV={V_s.shape[0]} (target ~{vertex_low})")

        solve_and_plot(net, tag=f"cycle_{cyc:02d}_step_{global_step:04d}_post_simplify")

        # ----------------------------
        # GENERATOR intersection/junction update again (then plot/save)
        # ----------------------------
        net = update_network_with_intersections(net, detect_mode="iterative", verbose=True)
        solve_and_plot(net, tag=f"cycle_{cyc:02d}_step_{global_step:04d}_post_intersections")

    print("Saved figures under:", out_dir)


    # ---- next cell ----

    # ============================================================
    # Kinked crack propagation demo (via network_growth.py)
    # - plots + saves at each OUTER cycle (pre/post simplify)
    # - simplifies component-by-component each outer cycle
    # ============================================================

    import os, sys
    from pathlib import Path
    import numpy as np

    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))


    out_dir = Path(repo_root) / "output" / "KinkedCrackPropagation"
    out_dir.mkdir(parents=True, exist_ok=True)

    # ----------------------------
    # Build initial network
    # ----------------------------
    mm = 1e-3
    vertices = np.array([
        [0,   -5.0*mm,  -3.0*mm],
        [1,    5.0*mm,   3.0*mm],
    ], dtype=float)

    connectivity = np.array([[0, 1]], dtype=int)

    net0 = CrackNetworkV4.from_vertices_connectivity(
        vertices=vertices,
        connectivity=connectivity,
        Nv_max=4,
        validate=True
    )

    material = Material(E=200e9, nu=0.30, plane_stress=False)
    applied  = AppliedStress(sigma_xx=100e6, sigma_yy=0.0e6, sigma_xy=0.0)

    # ----------------------------
    # ONE solver kwargs dict (used everywhere)
    # ----------------------------
    solver_kwargs = dict(
        ne_half=80,
        representation="cheb_quad",
        node_distribution="tip_dense",
        solver_option="parametrized_crack",
        parametrization="polyline",
        nq_col=12,
        nq_stress=16,
        crack_mode="half",
        junction_model="core",
        refine_kinks=False,
        refine_junction_endpoints=False,
        endpoint_min_nodes=4,
        kink_side_nodes=1,
        tip_cluster="power",
        tip_cluster_power=2.5,
        other_min_panels=4,
    )

    # ----------------------------
    # Solve + plot (keep unchanged)
    # ----------------------------
    def solve_and_plot(network, tag: str):
        step_dir = out_dir / tag
        step_dir.mkdir(parents=True, exist_ok=True)

        calc = DCENetworkStaticV4(material, network, applied)
        sol  = calc.solve(**solver_kwargs)
        res  = DCEResultsNetworkV4(calc, sol)

        plotter = DCEPlotterV4(res, out_dir=step_dir)
        _call(plotter, "plot_network_graph")

        opts = StressPlotOptsV4(
            extent_factor=3,
            n_grid=301,
            add_remote=True,
            mask_cracks=False,
            cmap="jet",
            n_bands=40,
            label_contours=False,
            vmax_factor=2,
        )
        _call(plotter, "plot_stress_components_global", opts=opts, components=("sxx", "syy", "sxy"))

        pl_def = DCEPlotterDeformedV4(res, out_dir=step_dir)
        _call(pl_def, "plot_deformed_network", scale=5e1, trim_core_junction_faces=True)

        plt.show()
        print(f"[{tag}] Saved figures to: {step_dir}")
        return res

    # ----------------------------
    # Propagation setup
    # ----------------------------
    cfg = PropagationConfig(
        f0=0.10,
        f_fixed=0.10,
        step_mode="fixed_step",
        simultaneous_tip_growth=True
    )

    Kc_demo = 1e6  # Pa*sqrt(m)
    tough = ConstantToughness(Kc_demo)

    dir_law = MaximumHoopStressLaw()

    evaluator = CandidateEvaluator(
        material=material,
        applied=applied,
        solver_kwargs=solver_kwargs,
        direction_law=dir_law,
        enable_ne_half_escalation=False,
    )
    evaluator.rmax_frac = 0.25
    evaluator.min_pts   = 8

    prop = CrackPropagator(
        cfg=cfg,
        evaluator=evaluator,
        toughness=tough,
        direction_law=dir_law,
    )

    # ============================================================
    # NEW: network_growth.py orchestration + simplification
    # ============================================================
    from fracture_utils.Upropagation.network_ops import get_netops
    from fracture_utils.Upropagation.network_growth import (
        NetworkGrower, GrowthLimits, GrowthPolicy, SimplifyPolicy
    )
    from fracture_utils.Ugenerator.crack_network_simplifier import CrackNetworkSimplifier, SimplificationConfig

    netops = get_netops()

    # ---- outer/inner limits
    max_cycles  = 50
    inner_max   = 200          # safety bound per outer cycle
    L_limit_mm  = 200.0        # stop when LONGEST component reaches this

    limits = GrowthLimits(
        max_outer_cycles=max_cycles,
        max_inner_steps=inner_max,
        max_longest_component_length_mm=L_limit_mm,
        stop_if_no_growth=True
    )

    # ---- growth policy: keep simple fixed_step (single polyline also works)
    policy = GrowthPolicy(
        f_ref=None,        # None => uses prop.cfg.f_fixed
        keff_floor=1e-12
    )

    # ---- simplification policy (component-by-component)
    # IMPORTANT: these defaults preserve shape; adjust as needed
    simp_cfg = SimplificationConfig(
        max_angle_deviation=3.0,     # smaller => preserves kinks
        min_edge_length=0.0*mm,      # don't prune by length here
        merge_at_degree2=True,
        remove_degree2_nodes=False,  # CRITICAL: avoid collapsing to a chord
        merge_vertex_tolerance=1e-6,
        max_merged_edge_length=8.0*mm,
        preserve_tips=True,
        preserve_junctions=True
    )

    simplify = SimplifyPolicy(
        enabled=True,
        simplifier_config=simp_cfg,
    )

    grower = NetworkGrower(
        prop=prop,
        evaluator=evaluator,
        netops=netops,
        limits=limits,
        policy=policy,
        simplify=simplify,
        CrackNetworkV4=CrackNetworkV4,
        CrackNetworkSimplifier=CrackNetworkSimplifier,
    )

    # ----------------------------
    # Plot INITIAL
    # ----------------------------
    solve_and_plot(net0, tag="cycle_00_initial")

    # ----------------------------
    # Run OUTER cycles manually so we can plot/save per cycle
    # ----------------------------
    net = net0
    global_step = 0

    for cyc in range(1, max_cycles + 1):

        # ---- run exactly ONE outer cycle worth of work
        # (NetworkGrower supports doing the full run; here we step cycle-by-cycle for plotting)
        net_pre, net_post, cyc_report, step_reports = grower.run_one_cycle(net, outer_cycle=cyc, global_step_start=global_step)

        # update global_step
        if step_reports:
            global_step = step_reports[-1].global_step
        else:
            # no steps attempted
            global_step = global_step

        # ---- Save PRE + plot
        cyc_dir = out_dir / f"cycle_{cyc:02d}"
        cyc_dir.mkdir(parents=True, exist_ok=True)

        V_pre, C_pre = grower.net_to_arrays(net_pre)
        np.savez(
            cyc_dir / f"network_pre_simplify_step_{global_step:04d}.npz",
            vertices=V_pre,
            connectivity=C_pre
        )
        solve_and_plot(net_pre, tag=f"cycle_{cyc:02d}_step_{global_step:04d}_pre_simplify")

        # ---- termination checks before simplify plotting
        if cyc_report.stop_reason is not None and cyc_report.stop_reason in ("length_limit", "no_growth"):
            print(f"[cycle_{cyc:02d}] terminating outer loop ({cyc_report.stop_reason}).")
            break

        # ---- Save POST + plot
        V_post, C_post = grower.net_to_arrays(net_post)
        np.savez(
            cyc_dir / f"network_post_simplify_step_{global_step:04d}.npz",
            vertices=V_post,
            connectivity=C_post
        )
        solve_and_plot(net_post, tag=f"cycle_{cyc:02d}_step_{global_step:04d}_post_simplify")

        # ---- advance network
        net = net_post

        # ---- print cycle summary
        print(
            f"[cycle_{cyc:02d}] steps={cyc_report.n_inner_steps} | "
            f"grew_any={cyc_report.grew_any} | "
            f"L_longest={cyc_report.L_longest_mm:.2f} mm | "
            f"n_polylines={cyc_report.n_polylines}"
        )

        # ---- stop if length reached (post-simplify)
        if cyc_report.stop_reason == "length_limit":
            break

    print("Saved figures under:", out_dir)

    return locals()


### Case `validation` (from `validation.ipynb`)


In [ ]:
def run_validation():
    """Case extracted from validation.ipynb."""
    import os, sys
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt

    # --- repo root on sys.path (one level above notebooks/)
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    out_dir = Path(repo_root) / "output" / "SingleHorizontalCrack"
    out_dir.mkdir(parents=True, exist_ok=True)


    # -------------------------
    # Define Y network: vertices + connectivity
    # Units: meters
    # -------------------------
    mm = 1e-3
    vertices = np.array([
        [0,   -5.0*mm,  0.0*mm],   
        [1,   5.0*mm,  0*mm], 
    ], dtype=float)

    # edges: (edge_id, v0, v1)
    connectivity = np.array([
        [0, 1],
    ], dtype=int)

    net = CrackNetworkV4.from_vertices_connectivity(
        vertices=vertices,
        connectivity=connectivity,
        Nv_max=3,
        validate=True,
    )

    material = Material(E=200e9, nu=0.30, plane_stress=False)
    applied  = AppliedStress(sigma_xx=0e6, sigma_yy=100e6, sigma_xy=0.0)

    calc = DCENetworkStaticV4(material, net, applied)

    sol  = calc.solve(
        ne_half=40,
        representation="cheb_quad",
        node_distribution="uniform",
        solver_option="parametrized_crack",
        parametrization="polyline",
        nq_col=3,
        nq_stress=6,
        crack_mode="half",
        enforce_cod_nonnegative=True,
        cod_tol=1e-10,
        cod_max_iter=25,
    )


    res = DCEResultsNetworkV4(calc, sol)
    plotter = DCEPlotterV4(res, out_dir=out_dir)

    # Graph topology
    plotter.plot_network_graph()

    # Displacement vector field
    plotter.plot_displacement_vector_field(
        extent_factor=1.2,
        n_grid=41,
        scale=1.0,
        disp_scale=5e4,
        mask_cracks=False,
    )

    # Stress fields (global)
    opts = StressPlotOptsV4(
        extent_factor=3,
        n_grid=400,
        add_remote=True,
        mask_cracks=False,
        cmap="jet",
        n_bands=40,
        label_contours=False,
        vmax_factor=2
    )
    plotter.plot_stress_components_global(opts=opts, components=("sxx","syy","sxy"))

    plt.show()
    print("Saved figures to:", out_dir)


    # ----------------------------
    # Plot: deformed network (trim junction faces; smooth COD/CSD if desired)
    # ----------------------------
    pl_def = DCEPlotterDeformedV4(res, out_dir=out_dir)
    pl_def.plot_deformed_network(
        n_pts_per_edge=200,
        scale=1e2,
        show_faces=True,
        units="mm",
        cod_smooth_window=5,
        csd_smooth_window=5,
        junction_model="core",
    )

    # ---- next cell ----

    # --- COD Validation Cell (fixed calc/network wiring) ---
    import os, sys
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt

    # --- repo root on sys.path (one level above notebooks/)
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    out_dir = Path(repo_root) / "output" / "COD_validation"
    out_dir.mkdir(parents=True, exist_ok=True)
    # -------------------------
    # Crack/physics metadata (benchmark)
    # IMPORTANT: keep 'a' consistent with the geometric half-length used below.
    # -------------------------
    mm = 1e-3
    a_geom = 5.0 * mm  # half-length (meters)

    crack_meta = dict(
        a=a_geom,
        sigma=100e6,       # magnitude used in analytical COD (Pa)
        E=200e9,
        nu=0.30,
        plane="strain",    # "strain" or "stress"
        edge_index=0,
    )

    # -------------------------
    # Define a single straight crack network (one edge)
    # Use a HORIZONTAL crack for Mode I under sigma_yy tension.
    # -------------------------
    vertices = np.array([
        [0, -a_geom, 0.0],   # [id, x, y]
        [1, +a_geom, 0.0],
    ], dtype=float)

    connectivity = np.array([
        [0, 1],              # edge from v0 -> v1
    ], dtype=int)

    network = CrackNetworkV4.from_vertices_connectivity(
        vertices=vertices,
        connectivity=connectivity,
        Nv_max=3,
        validate=True,
    )

    material = Material(E=crack_meta["E"], nu=crack_meta["nu"], plane_stress=("stress" in crack_meta["plane"].lower()))

    # Remote loading: uniform tension in y gives Mode I for a horizontal crack
    applied  = AppliedStress(sigma_xx=0.0, sigma_yy=+crack_meta["sigma"], sigma_xy=0.0)

    # IMPORTANT:
    #   - "network" is the geometry
    #   - "calc" is the solver object that HAS calc.network, calc.material, calc.applied
    calc = DCENetworkStaticV4(material=material, network=network, applied=applied)

    # -------------------------
    # COD solver hook for validation_cod.run_cod_sweep
    # -------------------------
    def run_solver_cod(knobs, crack_meta):
        """
        Return CODProfile(s, COD) for one edge using panel-midpoint reconstruction.
        COD is stored in MICRONS for plotting convenience.
        """
        sol = calc.solve(**knobs)

        # DCEResultsNetworkV4 expects (calc, sol), not (network, sol)
        res = DCEResultsNetworkV4(calc, sol)

        edge_index = int(crack_meta.get("edge_index", 0))
        x, COD, CSD, extra = res.reconstruct_cod_csd_panel_midpoints(
            edge_index=edge_index,
            enforce_global_tip_zero=True,
        )

        x = np.asarray(x, float).reshape(-1)
        COD = np.asarray(COD, float).reshape(-1) * 1e6  # Convert meters to microns
        CSD = np.asarray(CSD, float).reshape(-1)
        extra = dict(extra) if isinstance(extra, dict) else {}

        a_edge = float(extra.get("a_edge", np.max(np.abs(x))))
        s = (x + a_edge) / (2.0 * a_edge)  # map x in [-a_edge, a_edge] -> s in [0,1]

        return CODProfile(
            s=s,
            cod=COD,  # Now in microns
            meta=dict(x=x, CSD=CSD, **extra),
        )

    # -------------------------
    # Analytical COD: infinite plate, straight central crack under uniform tension
    # delta(x) = (4*sigma/E')*sqrt(a^2 - x^2)
    # Returns COD in MICRONS
    # -------------------------
    def analytical_cod(s, crack_meta):
        a = float(crack_meta["a"])
        sigma = float(crack_meta["sigma"])
        E = float(crack_meta["E"])
        nu = float(crack_meta["nu"])
        plane = str(crack_meta.get("plane", "strain")).lower()
        Eprime = E / (1.0 - nu**2) if "strain" in plane else E

        s = np.asarray(s, float)
        x = (2.0 * s - 1.0) * a
        cod_m = (4.0 * sigma / Eprime) * np.sqrt(np.maximum(a * a - x * x, 0.0))
        return cod_m * 1e6  # Convert meters to microns

    # -------------------------
    # Knob sweep
    # -------------------------
    ne_half_list = list(range(25, 230, 25))

    knob_sweep = []
    for ne in ne_half_list:
        knob_sweep.append(dict(
            ne_half=ne,
            crack_mode="full",
            representation="cheb_quad",   # or "panel", "singular", "cheb_quad"
            node_distribution="tip_dense",
            nq_col=12,
            nq_stress=16,
            ridge=1e-10,
            r0_factor=1e-3,
            solver_option="parametrized_crack",
            parametrization="polyline",
        ))

    # -------------------------
    # Run COD sweep
    # -------------------------
    sweep = run_cod_sweep(
        crack_meta=crack_meta,
        knob_sweep=knob_sweep,
        run_solver=run_solver_cod,
        analytical_cod=analytical_cod,
        s_sample=201,
    )

    # -------------------------
    # Plot and save figures
    # -------------------------

    # Figure 1: COD overlay
    result1 = plot_cod_overlay(
        sweep,
        knob_label_keys=("ne_half",),
        max_curves=5,
        title="Mode I COD profile: numerical vs analytical",
    )
    if result1 is not None:
        fig1, ax1 = result1  # Unpack the tuple
        filepath1 = out_dir / "cod_overlay_ne_half.png"

        # Update y-axis label to reflect microns
        ax1.set_ylabel(r"COD ($\mu$m)")
        ax1.set_xlabel("Normalized position s")
        ax1.grid(True)
        ax1.legend()
        fig1.savefig(filepath1, dpi=300, bbox_inches="tight")
        print(f"Saved: {filepath1}")
        plt.close(fig1)

    # Figure 2: Combined L2 and Linf COD errors on log-log axes
    # (algebraic convergence e ~ N^{-p} shows as a straight line of slope -p)
    fig2, ax2 = plt.subplots(figsize=(8, 6))

    ne_values = []
    l2_errors = []
    linf_errors = []

    for item in sweep.results:
        ne_values.append(item.knobs["ne_half"])
        l2_errors.append(item.l2_rel_pct)
        linf_errors.append(item.linf_rel_pct)

    ne_arr = np.asarray(ne_values, dtype=float)
    l2_arr = np.asarray(l2_errors, dtype=float)
    linf_arr = np.asarray(linf_errors, dtype=float)

    # Fit slope p in log-log: log(e) = log(C) - p * log(N)
    def _fit_slope(x, y):
        mask = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
        if int(mask.sum()) < 2:
            return np.nan, np.nan
        lx = np.log(x[mask]); ly = np.log(y[mask])
        p, c = np.polyfit(lx, ly, 1)
        return float(p), float(c)

    p_l2, c_l2 = _fit_slope(ne_arr, l2_arr)
    p_linf, c_linf = _fit_slope(ne_arr, linf_arr)

    ax2.loglog(ne_arr, l2_arr, marker="o", linewidth=2,
               label=r"$L_2$ relative error  (slope $\approx %.2f$)" % p_l2)
    ax2.loglog(ne_arr, linf_arr, marker="s", linewidth=2,
               label=r"$L_\infty$ relative error  (slope $\approx %.2f$)" % p_linf)

    # Reference slope guides (-1 and -2) anchored to the L2 curve at ne_arr[0]
    if np.isfinite(p_l2):
        n_ref = np.array([ne_arr.min(), ne_arr.max()], dtype=float)
        e0 = l2_arr[0]
        n0 = ne_arr[0]
        e_ref1 = e0 * (n_ref / n0) ** (-1.0)
        e_ref2 = e0 * (n_ref / n0) ** (-2.0)
        ax2.loglog(n_ref, e_ref1, ls="--", color="gray", lw=1,
                   label=r"reference slope $-1$")
        ax2.loglog(n_ref, e_ref2, ls=":", color="gray", lw=1,
                   label=r"reference slope $-2$")

    ax2.set_xlabel(r"$N^{e}_{\mathrm{half}}$", fontsize=16)
    ax2.set_ylabel("Relative error (%)", fontsize=16)
    ax2.grid(True, which="both", alpha=0.3)
    ax2.legend(fontsize=16)
    ax2.tick_params(axis="both", which="major", labelsize=16)

    filepath2 = out_dir / "cod_combined_errors_vs_ne_half.png"
    fig2.savefig(filepath2, dpi=300, bbox_inches="tight")
    print(f"Saved: {filepath2}  |  L2 slope = {p_l2:.3f}  Linf slope = {p_linf:.3f}")
    plt.close(fig2)

    print(f"\nAll plots saved to: {out_dir}")


    # ---- next cell ----

    #### VALIDATION OF A MODE II CRACK UNDER SHEAR LOADING (CSD)

    import os, sys
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt

    # --- repo root on sys.path (one level above notebooks/)
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    out_dir = Path(repo_root) / "output" / "CSD_validation_ModeII"
    out_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # Crack/physics metadata (benchmark)
    # -------------------------
    mm = 1e-3
    a_geom = 5.0 * mm  # half-length (m)
    tau0   = 100e6      # remote shear (Pa)

    crack_meta = dict(
        a=a_geom,
        tau=tau0,         # remote shear magnitude (Pa)
        E=200e9,
        nu=0.30,
        plane="strain",   # "strain" or "stress"
        edge_index=0,
    )

    # -------------------------
    # Geometry: single horizontal crack (-a,0) -> (+a,0)
    # Mode II: apply uniform remote shear sigma_xy = tau
    # -------------------------
    vertices = np.array([
        [0, -a_geom, 0.0],
        [1, +a_geom, 0.0],
    ], dtype=float)

    connectivity = np.array([[0, 1]], dtype=int)

    network = CrackNetworkV4.from_vertices_connectivity(
        vertices=vertices,
        connectivity=connectivity,
        Nv_max=3,
        validate=True,
    )

    material = Material(
        E=crack_meta["E"],
        nu=crack_meta["nu"],
        plane_stress=("stress" in crack_meta["plane"].lower()),
    )

    applied = AppliedStress(
        sigma_xx=0.0,
        sigma_yy=0.0,
        sigma_xy=+crack_meta["tau"],
    )

    calc = DCENetworkStaticV4(material=material, network=network, applied=applied)

    # -------------------------
    # Solver hook: return CSD profile using the CODProfile container
    # -------------------------
    def run_solver_csd(knobs, crack_meta):
        """
        Returns CODProfile(s, csd_um) for one edge.
        We store CSD (sliding displacement jump) in the 'cod' field for reuse of plotting utilities.
        """
        sol = calc.solve(**knobs)
        res = DCEResultsNetworkV4(calc, sol)

        edge_index = int(crack_meta.get("edge_index", 0))
        x, COD, CSD, extra = res.reconstruct_cod_csd_panel_midpoints(
            edge_index=edge_index,
            enforce_global_tip_zero=True,
        )

        x   = np.asarray(x, float).reshape(-1)
        CSD = np.asarray(CSD, float).reshape(-1) * 1e6  # meters -> microns
        extra = dict(extra) if isinstance(extra, dict) else {}

        a_edge = float(extra.get("a_edge", np.max(np.abs(x))))
        s = (x + a_edge) / (2.0 * a_edge)  # map x in [-a_edge, a_edge] -> s in [0,1]

        return CODProfile(
            s=s,
            cod=CSD,  # store CSD in microns
            meta=dict(x=x, **extra),
        )

    # -------------------------
    # Analytical CSD (Mode II) for infinite plate, central crack under remote shear tau
    # delta_t(x) = (4*tau/E') * sqrt(a^2 - x^2)
    # Returns CSD in MICRONS.
    # -------------------------
    def analytical_csd(s, crack_meta):
        a   = float(crack_meta["a"])
        tau = float(crack_meta["tau"])
        E   = float(crack_meta["E"])
        nu  = float(crack_meta["nu"])
        plane = str(crack_meta.get("plane", "strain")).lower()
        Eprime = E / (1.0 - nu**2) if "strain" in plane else E

        s = np.asarray(s, float)
        x = (2.0 * s - 1.0) * a
        csd_m = (4.0 * tau / Eprime) * np.sqrt(np.maximum(a*a - x*x, 0.0))
        return csd_m * 1e6  # meters -> microns

    # -------------------------
    # Knob sweep
    # -------------------------
    ne_half_list = list(range(25, 230, 25))

    knob_sweep = []
    for ne in ne_half_list:
        knob_sweep.append(dict(
            ne_half=int(ne),
            crack_mode="full",              # set "half" if validating half-crack formulation
            representation="cheb_quad",     # "panel", "singular", "cheb_quad", "cheb_spectral"
            node_distribution="tip_dense",
            nq_col=12,
            nq_stress=16,
            ridge=1e-10,
            r0_factor=1e-3,
            solver_option="parametrized_crack",
            parametrization="polyline",
        ))

    # -------------------------
    # Run sweep
    # -------------------------
    sweep = run_cod_sweep(
        crack_meta=crack_meta,
        knob_sweep=knob_sweep,
        run_solver=run_solver_csd,
        analytical_cod=analytical_csd,   # reuse API; returns CSD
        s_sample=201,
    )

    # -------------------------
    # Plot and save figures
    # -------------------------

    # Figure 1: CSD overlay
    result1 = plot_cod_overlay(
        sweep,
        knob_label_keys=("ne_half",),
        max_curves=5,
        title="Mode II CSD profile: numerical vs analytical (remote shear)",
    )
    if result1 is not None:
        fig1, ax1 = result1
        filepath1 = out_dir / "csd_overlay_ne_half.png"
        ax1.set_ylabel(r"CSD ($\mu$m)")
        ax1.set_xlabel("Normalized position s")
        ax1.grid(True)
        ax1.legend()
        fig1.savefig(filepath1, dpi=300, bbox_inches='tight')
        print(f"Saved: {filepath1}")
        plt.close(fig1)

    # Figure 2: Combined L2 and Linf errors on same plot
    fig2, ax2 = plt.subplots(figsize=(8, 6))

    # Extract error data from sweep.results
    ne_values = []
    l2_errors = []
    linf_errors = []

    for item in sweep.results:  # Access .results attribute
        ne_values.append(item.knobs['ne_half'])  # item.knobs is a dict
        l2_errors.append(item.l2_rel_pct)        # Use attribute access
        linf_errors.append(item.linf_rel_pct)    # Use attribute access

    # Plot both error metrics
    ax2.plot(ne_values, l2_errors, marker='o', label=r'$L_2$ Error', linewidth=2)
    ax2.plot(ne_values, linf_errors, marker='s', label=r'$L_\infty$ Error', linewidth=2)

    ax2.set_xlabel('ne_half', fontsize=12)
    ax2.set_ylabel('Relative Error (%)', fontsize=12)
    ax2.set_title('Mode II CSD Convergence: Error vs Discretization', fontsize=13)
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=11)

    filepath2 = out_dir / "csd_combined_errors_vs_ne_half.png"
    fig2.savefig(filepath2, dpi=300, bbox_inches='tight')
    print(f"Saved: {filepath2}")
    plt.close(fig2)

    print(f"\nAll plots saved to: {out_dir}")

    # ---- next cell ----

    # --- SIF Validation Cell (4 estimators; PK uses window_frac + normal_offset_frac sweep) ---
    import os, sys
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt

    # --- repo root on sys.path (one level above notebooks/)
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    out_dir = Path(repo_root) / "output" / "SIF-validation"
    out_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # Benchmark: straight crack, Mode I
    # -------------------------
    mm = 1e-3
    a_geom = 5.0 * mm
    sigma0 = 100e6

    crack_meta = dict(
        a=a_geom,
        sigma=sigma0,
        E=200e9,
        nu=0.30,
        plane="strain",  # "strain" or "stress"
        edge_index=0,
    )

    # -------------------------
    # Build a single-edge network
    # -------------------------
    vertices = np.array([
        [0, -a_geom, 0.0],
        [1, +a_geom, 0.0],
    ], dtype=float)

    connectivity = np.array([[0, 1]], dtype=int)

    network = CrackNetworkV4.from_vertices_connectivity(
        vertices=vertices,
        connectivity=connectivity,
        Nv_max=3,
        validate=True,
    )

    plane_stress = ("stress" in str(crack_meta["plane"]).lower())
    material = Material(E=crack_meta["E"], nu=crack_meta["nu"], plane_stress=plane_stress)
    applied  = AppliedStress(sigma_xx=0.0, sigma_yy=+crack_meta["sigma"], sigma_xy=0.0)
    calc = DCENetworkStaticV4(material=material, network=network, applied=applied)

    # -------------------------
    # Analytical SIF
    # -------------------------
    def KI_analytic(a, sigma):
        return float(sigma * np.sqrt(np.pi * a))

    # -------------------------
    # Run one solve and return the 3 "good" estimators + PK (with tunables)
    # -------------------------
    def solve_and_estimate(ne_half: int, *, window_frac: float, normal_offset_frac: float, representation: str = "cheb_quad"):
        knobs = dict(
            ne_half=int(ne_half),
            crack_mode="half",
            representation=str(representation),
            node_distribution="tip_dense",
            nq_col=12,
            nq_stress=16,
            ridge=1e-10,
            r0_factor=1e-3,
            solver_option="parametrized_crack",
            parametrization="polyline",
        )
        sol = calc.solve(**knobs)
        res = DCEResultsNetworkV4(calc, sol)

        edge_index = int(crack_meta.get("edge_index", 0))

        x, COD, CSD, extra = res.reconstruct_cod_csd_panel_midpoints(edge_index=edge_index, enforce_global_tip_zero=True)
        extra = dict(extra) if isinstance(extra, dict) else {}
        a_fit = float(extra.get("a_edge", float(crack_meta["a"])))

        E  = float(res.calc.material.E)
        nu = float(res.calc.material.nu)
        mu = E / (2.0 * (1.0 + nu))
        plane_stress_local = bool(getattr(res.calc.material, "plane_stress", False))
        kappa  = (3.0 - nu) / (1.0 + nu) if plane_stress_local else (3.0 - 4.0 * nu)
        Eprime = E if plane_stress_local else (E / (1.0 - nu**2))

        # Tip-fit
        KI_tip, KII_tip, _ = sif_from_cod_fit(
            x=x, COD=COD, CSD=CSD,
            a=a_fit, mu=mu, kappa=kappa,
            tip="right",
            window="fixed",
            rho_min=1e-6,
            rho_max=0.80,
            min_pts=10,
            n_fit=400,
            two_term=True,
        )

        # Jump
        KI_jump, KII_jump = estimate_KI_KII_from_jumps_near_tip(
            x=x, COD=COD, CSD=CSD,
            a=a_fit,
            Eprime=Eprime,
            side="right",
            frac_window=0.08,
        )

        # PK (Mode I: KI = sqrt(E' * J1))
        pk = PKProcessor(res)
        F = pk.F_PK_window(
            edge_index=edge_index,
            at="end",
            exclude_self=True,
            window_panels=0,               # force frac-mode
            window_frac=float(window_frac),
            min_panels=12,
            normal_offset_frac=float(normal_offset_frac),
        )
        if F is None:
            KI_pk = float("nan")
            J1 = float("nan")
        else:
            # local tangent from reconstructor
            _, _, _, extra2 = res.reconstruct_cod_csd_panel_midpoints(edge_index=edge_index, enforce_global_tip_zero=False)
            extra2 = dict(extra2)
            ex = np.asarray(extra2["ex"], float).reshape(2,)
            J1 = float(np.dot(np.asarray(F, float).reshape(2,), ex))
            KI_pk = float(np.sqrt(max(E * J1, 0.0)))

        # Mode I analytic
        KI_ref = KI_analytic(a_fit, float(crack_meta["sigma"]))

        return dict(
            ne_half=int(ne_half),
            a_fit=a_fit,
            Eprime=Eprime,
            KI_ref=KI_ref,
            KI_tip=float(KI_tip),
            KI_jump=float(KI_jump),
            KI_pk=float(KI_pk),
            J1=float(J1),
        )

    # -------------------------
    # Sweep ne_half with a small grid of PK tunables and plot
    # -------------------------
    ne_half_list = list(range(20, 420, 40))

    # Choose a few combinations (3–5) to avoid crowding
    pk_variants = [
        # dict(window_frac=0.20, normal_offset_frac=0.0,    label="wf=0.20, off=0"),
        dict(window_frac=0.50, normal_offset_frac=5e-6, label="wf=0.50"),
    ]

    # Precompute baseline tip/jump once per ne_half (reuse first variant result)
    baseline = {}
    pk_results = {v["label"]: [] for v in pk_variants}

    for ne in ne_half_list:
        r0 = solve_and_estimate(ne, window_frac=pk_variants[0]["window_frac"], normal_offset_frac=pk_variants[0]["normal_offset_frac"])
        baseline[ne] = dict(KI_ref=r0["KI_ref"], KI_tip=r0["KI_tip"], KI_jump=r0["KI_jump"])

        for v in pk_variants:
            r = solve_and_estimate(ne, window_frac=v["window_frac"], normal_offset_frac=v["normal_offset_frac"])
            pk_results[v["label"]].append(r)

    # -------------------------
    # Plot error vs ne_half
    # -------------------------
    def err_pct(K, Kref):
        return 100.0 * (K - Kref) / Kref

    # First figure: linear x-axis
    fig1, ax1 = plt.subplots()
    ax1.plot(ne_half_list, [err_pct(baseline[ne]["KI_tip"], baseline[ne]["KI_ref"]) for ne in ne_half_list],
            marker="o", label="K_tip")
    ax1.plot(ne_half_list, [err_pct(baseline[ne]["KI_jump"], baseline[ne]["KI_ref"]) for ne in ne_half_list],
            marker="o", label="K_jump")
    # PK variants
    for lab, arr in pk_results.items():
        ax1.plot(ne_half_list, [err_pct(a["KI_pk"], a["KI_ref"]) for a in arr], marker="o", label=f"K_pk ({lab})")
    ax1.set_xlabel(r"$N^{e}_{\mathrm{half}}$", fontsize=16)
    ax1.set_ylabel("Error (%)", fontsize=16)
    ax1.grid(True)
    ax1.legend(fontsize=16)
    ax1.tick_params(axis="both", which="major", labelsize=16)
    filepath1 = out_dir / 'sif_error_vs_ne_half_ncol12_nsig16_linear.png'
    fig1.savefig(filepath1, dpi=300, bbox_inches='tight')
    print(f"Saved: {filepath1}")

    # Second figure: log x-axis
    fig2, ax2 = plt.subplots()
    ax2.plot(ne_half_list, [err_pct(baseline[ne]["KI_tip"], baseline[ne]["KI_ref"]) for ne in ne_half_list],
            marker="o", label="K_tip")
    ax2.plot(ne_half_list, [err_pct(baseline[ne]["KI_jump"], baseline[ne]["KI_ref"]) for ne in ne_half_list],
            marker="o", label="K_jump")
    # PK variants
    for lab, arr in pk_results.items():
        ax2.plot(ne_half_list, [err_pct(a["KI_pk"], a["KI_ref"]) for a in arr], marker="o", label=f"K_pk ({lab})")
    ax2.set_xlabel("ne_half")
    ax2.set_ylabel("Error (%)")
    ax2.set_title("Mode I SIF error vs ne_half (log scale)")
    ax2.set_xscale('log')
    ax2.grid(True, which='both')
    ax2.legend()
    filepath2 = out_dir / 'sif_error_vs_ne_half_ncol12_nsig16_log.png'
    fig2.savefig(filepath2, dpi=300, bbox_inches='tight')
    print(f"Saved: {filepath2}")

    plt.show()


    # ---- next cell ----

    # --- Mixed-Mode SIF validation: inclined crack under biaxial loading (plots like Fig. B/C) ---
    import os, sys
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt

    # --- repo root on sys.path (one level above notebooks/)
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    out_dir = Path(repo_root) / "output" / "MixedMode_SIF_validation"
    out_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # Problem definition (match the paper figure)
    # Crack length = 2a, inclined by beta from loading axis.
    # Analytical (Rooke & Cartwright):
    #   KI  = sqrt(pi a) ( σ22 cos^2β + σ11 sin^2β )
    #   KII = sqrt(pi a) ( σ22 - σ11 ) sinβ cosβ
    # -------------------------
    mm = 1e-3
    a = 5.0 * mm                  # half-length (m)
    sig11 = 60e6                  # Pa  (σ11)
    sig22 = 100e6                 # Pa  (σ22)
    plane = "strain"              # "strain" or "stress"
    edge_index = 0

    # Sweep β in degrees (0..90)
    beta_deg = np.linspace(0.0, 90.0, 19)
    beta_rad = np.deg2rad(beta_deg)

    # Numerical resolutions
    ne_list = [20, 40, 60, 120]

    # Solver knobs (common)
    base_knobs = dict(
        crack_mode="full",
        representation="cheb_quad",     # you can switch to "cheb_spectral" once stable
        node_distribution="tip_dense",
        nq_col=12,
        nq_stress=16,
        ridge=1e-10,
        r0_factor=1e-3,
        solver_option="parametrized_crack",
        parametrization="polyline",
    )

    # -------------------------
    # Helpers
    # -------------------------
    def analytical_K(beta):
        """Return (KI, KII) for beta (radians)."""
        KI  = np.sqrt(np.pi * a) * (sig22 * (np.cos(beta)**2) + sig11 * (np.sin(beta)**2))
        KII = np.sqrt(np.pi * a) * ((sig22 - sig11) * np.sin(beta) * np.cos(beta))
        return KI, KII

    def crack_endpoints(beta):
        """Horizontal crack rotated by beta about origin."""
        c = np.cos(beta); s = np.sin(beta)
        p0 = np.array([-a, 0.0])
        p1 = np.array([+a, 0.0])
        R = np.array([[c, -s],[s, c]])
        q0 = R @ p0
        q1 = R @ p1
        return q0, q1

    def solve_K_for_beta(beta, ne_half):
        """Solve for a given beta and ne_half; return (KI_tip, KII_tip)."""
        q0, q1 = crack_endpoints(beta)

        vertices = np.array([
            [0, q0[0], q0[1]],
            [1, q1[0], q1[1]],
        ], dtype=float)

        connectivity = np.array([[0, 1]], dtype=int)

        network = CrackNetworkV4.from_vertices_connectivity(
            vertices=vertices,
            connectivity=connectivity,
            Nv_max=3,
            validate=True,
        )

        material = Material(
            E=200e9,
            nu=0.30,
            plane_stress=("stress" in plane.lower()),
        )

        applied = AppliedStress(
            sigma_xx=float(sig11),
            sigma_yy=float(sig22),
            sigma_xy=0.0,
        )

        calc = DCENetworkStaticV4(material=material, network=network, applied=applied)

        knobs = dict(base_knobs)
        knobs["ne_half"] = int(ne_half)

        sol = calc.solve(**knobs)
        res = DCEResultsNetworkV4(calc, sol)

        x, COD, CSD, extra = res.reconstruct_cod_csd_panel_midpoints(
            edge_index=edge_index,
            enforce_global_tip_zero=True,
        )
        extra = dict(extra) if isinstance(extra, dict) else {}
        a_fit = float(extra.get("a_edge", a))

        # elastic constants for sif_from_cod_fit
        E  = float(res.calc.material.E)
        nu = float(res.calc.material.nu)
        mu = E / (2.0 * (1.0 + nu))
        plane_stress_local = bool(getattr(res.calc.material, "plane_stress", False))
        kappa = (3.0 - nu) / (1.0 + nu) if plane_stress_local else (3.0 - 4.0 * nu)

        KI, KII, meta = sif_from_cod_fit(
            x=x, COD=COD, CSD=CSD,
            a=a_fit,
            mu=mu,
            kappa=kappa,
            tip="right",
            window="fixed",
            rho_min=1e-6,
            rho_max=0.60,
            min_pts=10,
            n_fit=400,
            two_term=True,
        )
        return float(KI), float(KII)

    # -------------------------
    # Compute curves
    # -------------------------
    KI_ana = np.zeros_like(beta_deg, float)
    KII_ana = np.zeros_like(beta_deg, float)
    for i, b in enumerate(beta_rad):
        KI_ana[i], KII_ana[i] = analytical_K(b)

    KI_num = {ne: np.zeros_like(beta_deg, float) for ne in ne_list}
    KII_num = {ne: np.zeros_like(beta_deg, float) for ne in ne_list}

    for ne in ne_list:
        print(f"\nSolving ne_half = {ne}")
        for i, b in enumerate(beta_rad):
            KI, KII = solve_K_for_beta(b, ne_half=ne)
            KI_num[ne][i] = KI
            KII_num[ne][i] = KII

    # Convert to MPa*sqrt(m) for plotting
    scale = 1e-6  # Pa*sqrt(m) -> MPa*sqrt(m)
    KI_ana_p = KI_ana * scale
    KII_ana_p = KII_ana * scale
    KI_num_p = {ne: KI_num[ne] * scale for ne in ne_list}
    KII_num_p = {ne: KII_num[ne] * scale for ne in ne_list}

    # -------------------------
    # Plot (B): KI vs beta
    # -------------------------
    figB, axB = plt.subplots()
    axB.plot(beta_deg, KI_ana_p, label="Analytical", linewidth=2)
    markers = {20:"o", 40:"s", 60:"^"}
    for ne in ne_list:
        axB.plot(beta_deg, KI_num_p[ne], marker=markers.get(ne,"o"), linestyle="None", label=f"Numerical ne_half={ne}")
    axB.set_xlabel(r"$\beta$ [deg]")
    axB.set_ylabel(r"$K_I$ [MPa $\sqrt{m}$]")
    axB.set_title(r"Mode I SIF vs crack angle $\beta$")
    axB.grid(True)
    axB.legend()
    figB.savefig(out_dir / "KI_vs_beta.png", dpi=300, bbox_inches="tight")
    plt.show()

    # -------------------------
    # Plot (C): KII vs beta
    # -------------------------
    figC, axC = plt.subplots()
    axC.plot(beta_deg, KII_ana_p, label="Analytical", linewidth=2)
    for ne in ne_list:
        axC.plot(beta_deg, KII_num_p[ne], marker=markers.get(ne,"o"), linestyle="None", label=f"Numerical ne_half={ne}")
    axC.set_xlabel(r"$\beta$ [deg]")
    axC.set_ylabel(r"$K_{II}$ [MPa $\sqrt{m}$]")
    axC.set_title(r"Mode II SIF vs crack angle $\beta$")
    axC.grid(True)
    axC.legend()
    figC.savefig(out_dir / "KII_vs_beta.png", dpi=300, bbox_inches="tight")
    plt.show()

    print(f"\nSaved plots to: {out_dir}")


    # ---- next cell ----

    import os, sys
    from pathlib import Path

    # Add repo root (one level above notebooks/) to sys.path
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    # --- Preamble re-exports
    out_dir = Path(repo_root) / "output" / "ArcField_Results"
    out_dir.mkdir(parents=True, exist_ok=True)


    # ----------------------------
    # Problem definition
    # ----------------------------
    mm = 1e-3
    a = 10.0 * mm  # crack half-length
    alpha_deg = 30.0  # half arc angle in degrees
    R = a / np.sin(np.deg2rad(alpha_deg))

    V = np.array([
        [0, -a,    0.0],     # endpoint (node)
        [1, a,  0.0],     # endpoint (node)
        [2,  0*mm,  R],    # center (control) 
    ], float)

    E = np.array([
        [0, 1],  
    ], int)


    net = CrackNetworkV4.from_vertices_connectivity(
        vertices=V,
        connectivity=E,
        Nv_max=4,
        validate=True,
        edge_kinds=["arc"],      # one per edge in E, same order
        edge_ctrl_vids=[(2,)],           # one per edge; only arc gets a center
        vertex_roles={2: "control"},
    )

    material = Material(E=200e9, nu=0.30, plane_stress=False)
    applied  = AppliedStress(sigma_xx=100.0e6, sigma_yy=200.0e6, sigma_xy=0.0)

    calc = DCENetworkStaticV4(material, net, applied)
    sol  = calc.solve(ne_half=40, parametrization="polyline", 
                      crack_mode="half",nq_col=12, nq_stress=16, junction_model="strict")
    res = DCEResultsNetworkV4(calc, sol)

    # ----------------------------
    # Plot: network graph + stress + displacements
    # ----------------------------
    plotter = DCEPlotterV4(res, out_dir=out_dir)

    _call(plotter, "plot_network_graph")

    opts = StressPlotOptsV4(
        extent_factor=3,
        n_grid=301,
        add_remote=True,
        mask_cracks=False,
        cmap="jet",
        n_bands=40,
        label_contours=False,
        vmax_factor=2,
    )
    _call(plotter, "plot_stress_components_global", opts=opts, components=("sxx","syy","sxy"))

    # If your core plotter has displacement methods, this will use them automatically.
    # Otherwise, it will just skip.
    _call(plotter, "plot_displacement_vector_field", disp_scale=3e4, mask_cracks=False)

    # ----------------------------
    # Plot: deformed network (trim junction faces; smooth COD/CSD if desired)
    # ----------------------------
    pl_def = DCEPlotterDeformedV4(res, out_dir=out_dir)
    pl_def.plot_deformed_network(
        n_pts_per_edge=200,
        scale=2e1,
        show_faces=True,
        units="mm",
        junction_model="strict",
    )

    # ----------------------------
    # Vectors: B-content and PK forces
    # ----------------------------
    pkplt = DCEPlotterPK(res, out_dir=out_dir)

    _call(
        pkplt,
        "plot_b_content_vectors",
        style=VecPlotStyle(max_len_factor=0.18),
        user_scale=1e-3,
        units="mm",
        show_tips=True,
        show_junctions=True,
        show_sum_at_junction=True,
        annotate=True,
    )

    _call(
        pkplt,
        "plot_pk_force_vectors",
        style=VecPlotStyle(max_len_factor=0.18),
        user_scale=1e-3,
        units="mm",
        exclude_self=True,
        show_tips=True,
        show_junctions=True,
        n_samples_tip=32,
        annotate=True,
    )

    plt.show()
    print("Saved figures to:", out_dir)


    # ---- next cell ----

    # ============================================================
    # Cotterell–Rice (IJF 1980) validation for circular arc crack
    # ============================================================

    import os, sys
    from pathlib import Path

    # Add repo root (one level above notebooks/) to sys.path
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    # --- Preamble re-exports
    out_dir = Path(repo_root) / "output" / "Arc_CotterellRice_Validation"
    out_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # Problem inputs
    # -------------------------
    mm = 1e-3
    a_chord = 10.0 * mm     # HALF chord length
    E = 200e9
    nu = 0.30
    plane = "strain"
    edge_index = 0

    sig_xx = 0e6
    sig_yy = 100e6
    sig_xy = 0.0

    # alpha = half-angle; total arc angle = 2*alpha
    alpha_deg = np.linspace(1.0, 30.0, 6)
    alpha = np.deg2rad(alpha_deg)

    # solver knobs
    base_knobs = dict(
        crack_mode="half",
        representation="cheb_quad",
        node_distribution="tip_dense",
        nq_col=12,
        nq_stress=16,
        ridge=1e-10,
        r0_factor=1e-3,
        solver_option="parametrized_crack",
        junction_model="strict",
        parametrization="polyline",
    )
    # -------------------------
    # Arc geometry (TRUE circle center)
    # -------------------------
    def arc_network(alpha):
        R = a_chord / np.sin(alpha)
        y_c = R * np.cos(alpha)     # TRUE center

        V = np.array([
            [0, -a_chord, 0.0],
            [1,  a_chord, 0.0],
            [2,  0.0,     y_c],     # CONTROL = CENTER
        ], float)

        E = np.array([[0, 1]], int)

        return CrackNetworkV4.from_vertices_connectivity(
            vertices=V,
            connectivity=E,
            Nv_max=4,
            validate=True,
            edge_kinds=["arc"],
            edge_ctrl_vids=[(2,)],
            vertex_roles={2: "control"},
        )

    # -------------------------
    # Solver wrapper
    # -------------------------
    def solve_K(net, ne_half):
        material = Material(E=E, nu=nu, plane_stress=("stress" in plane))
        applied  = AppliedStress(sig_xx, sig_yy, sig_xy)

        calc = DCENetworkStaticV4(material, net, applied)
        knobs = dict(base_knobs)
        knobs["ne_half"] = int(ne_half)

        sol = calc.solve(**knobs)
        res = DCEResultsNetworkV4(calc, sol)

        x, COD, CSD, extra = res.reconstruct_cod_csd_panel_midpoints(
            edge_index=edge_index,
            enforce_global_tip_zero=True,
        )

        a_arc = float(extra["a_edge"])  # HALF ARC LENGTH

        mu = E / (2.0 * (1.0 + nu))
        kappa = (3.0 - 4.0 * nu)

        KI, KII, _ = sif_from_cod_fit(
            x=x, COD=COD, CSD=CSD,
            a=a_arc,
            mu=mu,
            kappa=kappa,
            tip="right",
            window="fixed",
            rho_min=1e-6,
            rho_max=0.2,
            min_pts=10,
            n_fit=400,
            two_term=True,
        )

        return float(KI), float(KII), a_arc

    # ============================================================
    # Run: ARC geometry, ne_half = 80
    # ============================================================
    ne_half = 100
    scale = 1e-6

    KI_num = []
    KII_num = []
    KI_ana = []
    KII_ana = []

    for al in alpha:
        R = a_chord / np.sin(al)
        a_arc = R * al

        KIa, KIIa = cotterell_rice_K(al, a_chord, sig_xx, sig_yy, sig_xy)
        KI_ana.append(KIa)
        KII_ana.append(KIIa)

        net = arc_network(al)
        KIn, KIIn, _ = solve_K(net, ne_half)
        KI_rn, KII_rn = rotate_sifs(KIn, KIIn, -2*al)

        KI_num.append(KI_rn)
        KII_num.append(KII_rn)
        
    KI_ana = np.array(KI_ana)
    KII_ana = np.array(KII_ana)
    KI_num = np.array(KI_num)
    KII_num =-np.array(KII_num)

    # -------------------------
    # Plots
    # -------------------------
    fig1, ax1 = plt.subplots()
    ax1.plot(alpha_deg, KI_ana*scale, "k--", lw=3, label="Analytical (Cotterell–Rice)")
    ax1.plot(alpha_deg, KI_num*scale, lw=2, label=f"Numerical arc ne_half={ne_half}")
    ax1.set_xlabel(r"$\alpha$ [deg]  (total arc angle $=2\alpha$)")
    ax1.set_ylabel(r"$K_I$ [MPa $\sqrt{m}$]")
    ax1.set_title("Arc crack: $K_I$ vs $\\alpha$")
    ax1.grid(True); ax1.legend()
    ax1.set_ylim(bottom=0)
    fig1.savefig(out_dir / "KI_vs_alpha_1term_ne80.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig2, ax2 = plt.subplots()
    ax2.plot(alpha_deg, KII_ana*scale, "k--", lw=3, label="Analytical (Cotterell–Rice)")
    ax2.plot(alpha_deg,KII_num*scale, lw=2, label=f"Numerical arc ne_half={ne_half}")
    ax2.set_xlabel(r"$\alpha$ [deg]  (total arc angle $=2\alpha$)")
    ax2.set_ylabel(r"$K_{II}$ [MPa $\sqrt{m}$]")
    ax2.set_title("Arc crack: $K_{II}$ vs $\\alpha$")
    ax2.grid(True); ax2.legend()
    ax2.set_ylim(bottom=0)
    fig2.savefig(out_dir / "KII_vs_alpha_1term_ne80.png", dpi=300, bbox_inches="tight")
    plt.show()



    # ---- next cell ----

    # ============================================================
    # Cotterell–Rice (IJF 1980) validation for circular arc crack
    # Sweep ne_half and plot L2 error norms in K_I and K_II
    # ============================================================

    import os, sys
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt

    # Add repo root (one level above notebooks/) to sys.path
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    out_dir = Path(repo_root) / "output" / "Arc_CotterellRice_Validation"
    out_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # Problem inputs
    # -------------------------
    mm = 1e-3
    a_chord = 10.0 * mm     # HALF chord length
    E = 200e9
    nu = 0.30
    plane = "strain"
    edge_index = 0

    sig_xx = 0e6
    sig_yy = 100e6
    sig_xy = 0.0

    # alpha = half-angle; total arc angle = 2*alpha
    alpha_deg = np.linspace(1.0, 30.0, 6)
    alpha = np.deg2rad(alpha_deg)

    # solver knobs
    base_knobs = dict(
        crack_mode="half",
        representation="cheb_quad",
        node_distribution="tip_dense",
        nq_col=12,
        nq_stress=16,
        ridge=1e-10,
        r0_factor=1e-3,
        solver_option="parametrized_crack",
        junction_model="strict",
        parametrization="polyline",
    )

    # -------------------------
    # Arc geometry (TRUE circle center)
    # -------------------------
    def arc_network(alpha):
        R = a_chord / np.sin(alpha)
        y_c = R * np.cos(alpha)     # TRUE center

        V = np.array([
            [0, -a_chord, 0.0],
            [1,  a_chord, 0.0],
            [2,  0.0,     y_c],     # CONTROL = CENTER
        ], float)

        Econn = np.array([[0, 1]], int)

        return CrackNetworkV4.from_vertices_connectivity(
            vertices=V,
            connectivity=Econn,
            Nv_max=4,
            validate=True,
            edge_kinds=["arc"],
            edge_ctrl_vids=[(2,)],
            vertex_roles={2: "control"},
        )

    # -------------------------
    # Solver wrapper
    # -------------------------
    def solve_K(net, ne_half):
        material = Material(E=E, nu=nu, plane_stress=("stress" in plane))
        applied  = AppliedStress(sig_xx, sig_yy, sig_xy)

        calc = DCENetworkStaticV4(material, net, applied)
        knobs = dict(base_knobs)
        knobs["ne_half"] = int(ne_half)

        sol = calc.solve(**knobs)
        res = DCEResultsNetworkV4(calc, sol)

        x, COD, CSD, extra = res.reconstruct_cod_csd_panel_midpoints(
            edge_index=edge_index,
            enforce_global_tip_zero=True,
        )

        a_arc = float(extra["a_edge"])  # HALF ARC LENGTH

        mu = E / (2.0 * (1.0 + nu))
        kappa = (3.0 - 4.0 * nu) if ("strain" in plane.lower()) else (3.0 - nu) / (1.0 + nu)

        KI, KII, _ = sif_from_cod_fit(
            x=x, COD=COD, CSD=CSD,
            a=a_arc,
            mu=mu,
            kappa=kappa,
            tip="right",
            window="fixed",
            rho_min=1e-6,
            rho_max=0.2,
            min_pts=10,
            n_fit=400,
            two_term=True,
        )

        return float(KI), float(KII), a_arc

    # -------------------------
    # L2 error utility (RMS or plain L2; pick one)
    # -------------------------
    def l2_error(y_num, y_ref, *, mode="rms"):
        y_num = np.asarray(y_num, float).reshape(-1)
        y_ref = np.asarray(y_ref, float).reshape(-1)
        e = y_num - y_ref
        if mode == "rms":
            return 100.0 * float(np.sqrt(np.mean(e**2)))/np.sqrt(np.mean(y_ref**2))
        elif mode == "l2":
            return 100.0 * float(np.sqrt(np.sum(e**2)))/np.sqrt(np.sum(y_ref**2))
        else:
            raise ValueError("mode must be 'rms' or 'l2'")

    # ============================================================
    # Sweep ne_half and compute errors
    # ============================================================
    ne_list = list(range(40, 301, 20))  # 40, 60, 80, 100, 120, 140, 160
    err_KI = []
    err_KII = []

    # Precompute analytical once (depends only on alpha)
    KI_ana = []
    KII_ana = []
    for al in alpha:
        R = a_chord / np.sin(al)
        a_arc = R * al  # HALF ARC LENGTH (CR reference)
        KIa, KIIa = cotterell_rice_K(al, a_arc, sig_xx, sig_yy, sig_xy)
        KI_ana.append(KIa)
        KII_ana.append(KIIa)
    KI_ana = np.asarray(KI_ana, float)
    KII_ana = np.asarray(KII_ana, float)

    # Sweep
    for ne_half in ne_list:
        KI_num = []
        KII_num = []

        for al in alpha:
            net = arc_network(al)
            KIn, KIIn, _ = solve_K(net, ne_half)
            KI_rn, KII_rn = rotate_sifs(KIn, KIIn, -2.0 * al)

            KI_num.append(KI_rn)
            KII_num.append(KII_rn)

        KI_num = np.asarray(KI_num, float)

        # keep your historical sign convention for KII (you had KII_num = -KII_num)
        KII_num = -np.asarray(KII_num, float)

        err_KI.append(l2_error(KI_num, KI_ana, mode="rms"))
        err_KII.append(l2_error(KII_num, KII_ana, mode="rms"))

    err_KI = np.asarray(err_KI, float)
    err_KII = np.asarray(err_KII, float)


    # ============================================================
    # Plots: error norms vs ne_half
    # ============================================================
    figE1, axE1 = plt.subplots()
    axE1.plot(ne_list, err_KI, "o-", lw=2)
    axE1.set_xlabel(r"$n_{e,\mathrm{half}}$")
    axE1.set_ylabel(r"% RMS Error in $K_{I}$")
    axE1.set_title(r"Arc crack: $K_I$ RMS error vs $n_{e,\mathrm{half}}$")
    axE1.grid(True)
    figE1.savefig(out_dir / "KI_RMS_error_vs_ne_half.png", dpi=300, bbox_inches="tight")
    plt.show()

    figE2, axE2 = plt.subplots()
    axE2.plot(ne_list, err_KII, "o-", lw=2)
    axE2.set_xlabel(r"$n_{e,\mathrm{half}}$")
    axE2.set_ylabel(r"% RMS Error in $K_{II}$")
    axE2.set_title(r"Arc crack: $K_{II}$ RMS error vs $n_{e,\mathrm{half}}$")
    axE2.grid(True)
    figE2.savefig(out_dir / "KII_RMS_error_vs_ne_half.png", dpi=300, bbox_inches="tight")
    plt.show()


    # ---- next cell ----

    # ============================================================
    # Cotterell–Rice validation for circular arc crack using a POLYLINE approximation
    # FIT OVER MULTIPLE TIP-NEAR SEGMENTS (module-based pooled Euclid fit)
    # ============================================================

    import os, sys
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt

    # Add repo root (one level above notebooks/) to sys.path
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    out_dir = Path(repo_root) / "output" / "ArcPolyline_CotterellRice_Validation"
    out_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # Inputs
    # -------------------------
    mm = 1e-3
    a_chord = 10.0 * mm
    E = 200e9
    nu = 0.30
    plane = "stress"   # or "strain"

    sig_xx = 0e6
    sig_yy = 100e6
    sig_xy = 0.0

    alpha_deg = np.linspace(1.0, 45.0, 12)
    alpha = np.deg2rad(alpha_deg)

    ne_half = 40
    nseg_test = 25
    scale = 1e-6

    # pooled-fit controls (for titles + passing into solve)
    min_segs = 6
    fit_frac = 0.30      # fraction of total polyline arclength pooled (module semantics)
    rmax_frac = 0.30     # rmax = rmax_frac * a_arc
    two_term = True

    base_knobs = dict(
        crack_mode="half",
        representation="cheb_quad",
        node_distribution="tip_dense",
        nq_col=12,
        nq_stress=16,
        ridge=1e-10,
        r0_factor=1e-3,
        solver_option="parametrized_crack",
        junction_model="strict",
        parametrization="polyline",
    )

    # -------------------------
    # Use module pooled polyline SIF
    # -------------------------
    from fracture_utils.Uprocessor import SIF_cod as sifcod

    def solve_arc_polyline(al, *, ne_half, nseg, fit_frac, min_segs, rmax_frac, two_term):
        """
        Returns:
          a_arc (half arc length),
          KI_CR, KII_CR (in CR comparison frame),
          meta (dict)
        """
        # geometry / reference length for Cotterell–Rice
        R = a_chord / np.sin(al)
        a_arc = R * al  # HALF arc length (CR uses this)

        # build polyline network
        net, V = polyline_arc_network(al, a_chord, nseg=nseg)

        # last segment is the physical right tip segment
        edge_tip = int(nseg - 1)

        KI, KII, KI_CR, KII_CR, meta = sifcod.solve_K_polyline(
            net=net,
            V=V,
            ne_half=int(ne_half),
            edge_index_use=edge_tip,
            a_fit_global=float(a_arc),
            E=E, nu=nu, plane=plane,
            sig_xx=sig_xx, sig_yy=sig_yy, sig_xy=sig_xy,
            base_knobs=base_knobs,

            # pooled-window controls (module)
            fit_frac=float(fit_frac),
            min_segs=int(min_segs),
            rmin_frac=1e-6,
            rmax_frac=float(rmax_frac),
            min_pts=12,
            two_term=bool(two_term),

            # rotate to CR frame
            theta_rot_override=float(-2.0 * al),
        )

        return float(a_arc), float(KI_CR), float(KII_CR), meta

    # -------------------------
    # Run sweep
    # -------------------------
    KI_ana, KII_ana = [], []
    KI_poly, KII_poly = [], []
    meta_list = []

    for al in alpha:
        # analytical (must use a_arc = R*al)
        R = a_chord / np.sin(al)
        a_arc = R * al
        KIa, KIIa = cotterell_rice_K(al, a_arc, sig_xx, sig_yy, sig_xy)
        KI_ana.append(KIa)
        KII_ana.append(KIIa)

        # numerical pooled polyline fit (module)
        a_used, KIn, KIIn, meta = solve_arc_polyline(
            al,
            ne_half=ne_half,
            nseg=nseg_test,
            fit_frac=fit_frac,
            min_segs=min_segs,
            rmax_frac=rmax_frac,
            two_term=two_term,
        )
        KI_poly.append(KIn)
        KII_poly.append(KIIn)
        meta_list.append(meta)

    KI_ana  = np.asarray(KI_ana, float)
    KII_ana = np.asarray(KII_ana, float)
    KI_poly = np.asarray(KI_poly, float)
    KII_poly = np.asarray(KII_poly, float)

    # If your historical convention requires flipping KII, do it here (otherwise comment out)
    KII_poly = -KII_poly

    print("Example meta (last alpha):", meta_list[-1])

    # -------------------------
    # Plots
    # -------------------------
    fig1, ax1 = plt.subplots()
    ax1.plot(alpha_deg, KI_ana*scale, "k--", lw=3, label="Analytical (Cotterell–Rice)")
    ax1.plot(alpha_deg, KI_poly*scale, lw=2, label=f"Numerical POLYLINE pooled (nseg={nseg_test}, ne_half={ne_half})")
    ax1.set_xlabel(r"$\alpha$ [deg]  (total arc angle $=2\alpha$)")
    ax1.set_ylabel(r"$K_I$ [MPa $\sqrt{m}$]")
    ax1.set_title(rf"$K_I$ vs $\alpha$ (pooled fit: min_segs={min_segs}, fit_frac={fit_frac}, rmax_frac={rmax_frac})")
    ax1.grid(True); ax1.legend()
    ax1.set_ylim(bottom=0)
    fig1.savefig(out_dir / f"K_I_vs_alpha_polyline_pooled_ne{ne_half}_nseg{nseg_test}.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig2, ax2 = plt.subplots()
    ax2.plot(alpha_deg, KII_ana*scale, "k--", lw=3, label="Analytical (Cotterell–Rice)")
    ax2.plot(alpha_deg, KII_poly*scale, lw=2, label=f"Numerical POLYLINE pooled (nseg={nseg_test}, ne_half={ne_half})")
    ax2.set_xlabel(r"$\alpha$ [deg]  (total arc angle $=2\alpha$)")
    ax2.set_ylabel(r"$K_{II}$ [MPa $\sqrt{m}$]")
    ax2.set_title(rf"$K_{{II}}$ vs $\alpha$ (pooled fit: min_segs={min_segs}, fit_frac={fit_frac}, rmax_frac={rmax_frac})")
    ax2.grid(True); ax2.legend()
    ax2.set_ylim(bottom=0)
    fig2.savefig(out_dir / f"K_II_vs_alpha_polyline_pooled_ne{ne_half}_nseg{nseg_test}.png", dpi=300, bbox_inches="tight")
    plt.show()


    # ---- next cell ----

    # ============================================================
    # Diagnostics: L2 error vs ne_half and vs rmax_frac
    # ============================================================

    def compute_K_arrays(ne_half_val, rmax_frac_val, *, fit_frac, min_segs, two_term):
        KI_ana, KII_ana = [], []
        KI_num, KII_num = [], []

        for al in alpha:
            R = a_chord / np.sin(al)
            a_fit_global = R * al  # half arc length

            KIa, KIIa = cotterell_rice_K(al, a_fit_global, sig_xx, sig_yy, sig_xy)
            KI_ana.append(KIa); KII_ana.append(KIIa)

            net, V = polyline_arc_network(al, a_chord, nseg=nseg_test)

            KI_loc, KII_loc, KI_CR, KII_CR, meta = solve_K_polyline(
                net, V, int(ne_half_val), edge_tip, a_fit_global,
                E=E, nu=nu, plane=plane,
                sig_xx=sig_xx, sig_yy=sig_yy, sig_xy=sig_xy,
                base_knobs=base_knobs,
                fit_frac=float(fit_frac),
                min_segs=int(min_segs),
                rmax_frac=float(rmax_frac_val),
                two_term=bool(two_term),
            )

            # keep your convention (KII positive)
            if KII_CR < 0.0:
                KII_CR = -KII_CR

            KI_num.append(KI_CR)
            KII_num.append(KII_CR)

        KI_ana = np.asarray(KI_ana, float)
        KII_ana = np.asarray(KII_ana, float)
        KI_num = np.asarray(KI_num, float)
        KII_num = np.asarray(KII_num, float)
        return KI_ana, KII_ana, KI_num, KII_num


    def rel_L2(num, ana):
        num = np.asarray(num, float)
        ana = np.asarray(ana, float)
        denom = np.linalg.norm(ana)
        if denom == 0:
            return np.nan
        return np.linalg.norm(num - ana) / denom


    # -------------------------
    # (1) L2 error vs ne_half
    # -------------------------
    ne_list = np.arange(30, 61, 2)  # 2..40 in steps of 2 (change if you want every integer)
    errKI_ne = []
    errKII_ne = []

    # Use the SAME knobs you currently used in the main loop
    fit_frac_fixed = 0.3
    min_segs_fixed = 4
    rmax_fixed = 0.1
    two_term_fixed = False

    for ne_val in ne_list:
        KIa, KIIa, KIn, KIIn = compute_K_arrays(
            ne_val, rmax_fixed,
            fit_frac=fit_frac_fixed,
            min_segs=min_segs_fixed,
            two_term=two_term_fixed
        )
        errKI_ne.append(rel_L2(KIn, KIa))
        errKII_ne.append(rel_L2(KIIn, KIIa))

    errKI_ne = np.asarray(errKI_ne, float)
    errKII_ne = np.asarray(errKII_ne, float)

    fig, ax = plt.subplots()
    ax.plot(ne_list, errKI_ne, "o-", label=r"$\epsilon_{K_I}$ (rel. L2)")
    ax.plot(ne_list, errKII_ne, "s-", label=r"$\epsilon_{K_{II}}$ (rel. L2)")
    ax.set_xlabel(r"$n_e$ (ne\_half)")
    ax.set_ylabel("Relative L2 error")
    ax.set_title(f"L2 error vs ne_half  (nseg={nseg_test}, rmax_frac={rmax_fixed})")
    ax.grid(True)
    ax.legend()
    fig.savefig(out_dir / f"L2_error_vs_ne_half_nseg{nseg_test}_rmax{rmax_fixed}.png", dpi=300, bbox_inches="tight")
    plt.show()



    # ---- next cell ----


    # -------------------------
    # (2) L2 error vs rmax_frac
    # -------------------------
    rmax_list = np.linspace(0.05, 0.20, 20)
    errKI_r = []
    errKII_r = []

    ne_fixed = 40  # per your request

    for rmax_val in rmax_list:
        KIa, KIIa, KIn, KIIn = compute_K_arrays(
            ne_fixed, rmax_val,
            fit_frac=fit_frac_fixed,
            min_segs=min_segs_fixed,
            two_term=two_term_fixed
        )
        errKI_r.append(rel_L2(KIn, KIa))
        errKII_r.append(rel_L2(KIIn, KIIa))

    errKI_r = np.asarray(errKI_r, float)
    errKII_r = np.asarray(errKII_r, float)

    fig, ax = plt.subplots()
    ax.plot(rmax_list, errKI_r, "o-", label=r"$\epsilon_{K_I}$ (rel. L2)")
    ax.plot(rmax_list, errKII_r, "s-", label=r"$\epsilon_{K_{II}}$ (rel. L2)")
    ax.set_xlabel(r"$r_{\max,\mathrm{frac}}$")
    ax.set_ylabel("Relative L2 error")
    ax.set_title(f"L2 error vs rmax_frac  (nseg={nseg_test}, ne_half={ne_fixed})")
    ax.grid(True)
    ax.legend()
    fig.savefig(out_dir / f"L2_error_vs_rmax_nseg{nseg_test}_ne{ne_fixed}.png", dpi=300, bbox_inches="tight")
    plt.show()


    # ---- next cell ----

    ## STRAIGHT CRACK VALIDATION: Lo Journal of Applied Mechanics 1978, vol 45-p797
    # ============================================================
    ## (1) Single Straight Crack
    import os, sys
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt

    # Add repo root (one level above notebooks/) to sys.path
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    # --- Preamble re-exports

    out_dir = Path(repo_root) / "output" / "Straight_Crack_Validation"
    out_dir.mkdir(parents=True, exist_ok=True)

    # ----------------------------
    # Problem definition
    # ----------------------------
    mm = 1e-3
    c = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]) * mm

    # Material and loading
    E = 200e9
    nu = 0.30
    plane = "strain"
    sig_xx = 0e6
    sig_yy = 100e6
    sig_xy = 0.0

    # Solver parameters
    ne_half = 100
    base_knobs = dict(
        crack_mode="half",
        representation="cheb_quad",
        node_distribution="tip_dense",
        nq_col=12,
        nq_stress=16,
        ridge=1e-10,
        r0_factor=1e-3,
        solver_option="parametrized_crack",
        junction_model="strict",
        parametrization="polyline",
    )

    # ----------------------------
    # Build straight crack network
    # ----------------------------
    def build_straight_crack(crack_half_length):
        """
        Build straight crack centered at origin
        
        Parameters
        ----------
        crack_half_length : float
            Half-length of crack (c)
        
        Returns
        -------
        net : CrackNetworkV4
            Crack network
        vertices : ndarray
            Vertex array
        """
        vertices = np.array([
            [0,  -crack_half_length,  0.0],  # v0 (left tip)
            [1,   crack_half_length,  0.0],  # v1 (right tip)
        ], dtype=float)
        
        connectivity = np.array([
            [0, 1],   # e0: main crack
        ], dtype=int)
        
        net = CrackNetworkV4.from_vertices_connectivity(
            vertices=vertices,
            connectivity=connectivity,
            Nv_max=2,
            validate=True
        )
        
        return net, vertices

    # ----------------------------
    # Run sweep over crack lengths
    # ----------------------------
    KI_right = []
    KII_right = []
    KI_left = []
    KII_left = []

    material = Material(E=E, nu=nu, plane_stress=False)
    applied = AppliedStress(sigma_xx=sig_xx, sigma_yy=sig_yy, sigma_xy=sig_xy)

    # Extract material properties
    mu = material.mu
    kappa = material.kappa

    print(f"Material properties:")
    print(f"  E = {E/1e9:.1f} GPa")
    print(f"  nu = {nu}")
    print(f"  mu = {mu/1e9:.2f} GPa")
    print(f"  kappa = {kappa:.2f}")
    print()

    for length in c:    
        print(f"Solving for c = {length*1e3:.2f} mm...")
        
        # Build network
        net, vertices = build_straight_crack(length)
        
        # Solve
        calc = DCENetworkStaticV4(material, net, applied)
        knobs = dict(base_knobs)
        knobs["ne_half"] = ne_half
        
        sol = calc.solve(**knobs)
        res = DCEResultsNetworkV4(calc, sol)
        
        # Extract COD/CSD for the crack edge
        x, COD, CSD, extra = res.reconstruct_cod_csd_panel_midpoints(
            edge_index=0,
            enforce_global_tip_zero=True,
        )
        
        # Extract SIFs for right tip (tip="right" means measuring from right end)
        try:
            KI_r, KII_r, meta_r = sif_from_cod_fit(
                x=x,
                COD=COD,
                CSD=CSD,
                a=length,           # crack half-length
                mu=mu,              # shear modulus from material
                kappa=kappa,        # Kolosov constant from material
                tip="right",        # measuring from right tip
                window="auto",
                rho_min=1e-3,
                rho_max=0.05,
                n_fit=80,
                two_term=False,
            )
            KI_right.append(KI_r)
            KII_right.append(abs(KII_r))
        except Exception as e:
            print(f"  Warning: Right tip fit failed: {e}")
            KI_right.append(np.nan)
            KII_right.append(np.nan)

        # Extract SIFs for left tip (tip="left" means measuring from left end)
        try:
            KI_l, KII_l, meta_l = sif_from_cod_fit(
                x=x,
                COD=COD,
                CSD=CSD,
                a=length,           # crack half-length
                mu=mu,
                kappa=kappa,
                tip="left",         # measuring from left tip
                window="auto",
                rho_min=1e-3,
                rho_max=0.05,
                n_fit=80,
                two_term=False,
            )
            KI_left.append(KI_l)
            KII_left.append(abs(KII_l))
        except Exception as e:
            print(f"  Warning: Left tip fit failed: {e}")
            KI_left.append(np.nan)
            KII_left.append(np.nan)

    # Convert to arrays
    KI_right = np.array(KI_right)
    KII_right = np.array(KII_right)
    KI_left = np.array(KI_left)
    KII_left = np.array(KII_left)

    # Average left and right (should be symmetric)
    KI_avg = (KI_right + KI_left) / 2.0
    KII_avg = (KII_right + KII_left) / 2.0

    # Analytical solution for straight crack under remote tension
    K_analytical = sig_yy * np.sqrt(np.pi * c)

    # Calculate percent error
    error_KI = 100.0 * (KI_avg - K_analytical) / K_analytical

    # -------------------------
    # Plots
    # -------------------------
    scale = 1e-6  # Convert to MPa√m

    # Plot 1: K_I comparison (numerical vs analytical)
    fig1, ax1 = plt.subplots(figsize=(10, 6))
    ax1.plot(c*1e3, K_analytical*scale, 'k--', lw=3, label='Analytical: $\sigma_\infty\sqrt{\pi c}$')
    ax1.plot(c*1e3, KI_avg*scale, 'o-', lw=2, markersize=8, label=f'Numerical (ne_half={ne_half})')
    ax1.set_xlabel(r'$c$ [mm] (crack half-length)', fontsize=12)
    ax1.set_ylabel(r'$K_I$ [MPa$\sqrt{m}$]', fontsize=12)
    ax1.set_title('Straight Crack: $K_I$ vs Crack Size', fontsize=14, weight='bold')
    ax1.set_xscale('log')
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=11)
    plt.tight_layout()
    fig1.savefig(out_dir / "straight_KI_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()

    # Plot 2: Percent error
    fig2, ax2 = plt.subplots(figsize=(10, 6))
    ax2.axhline(0.0, color='k', linestyle='--', lw=2, alpha=0.5, label='Zero error')
    ax2.plot(c*1e3, error_KI, 'o-', lw=2, markersize=8, color='C1', label=f'Error (ne_half={ne_half})')
    ax2.set_xlabel(r'$c$ [mm] (crack half-length)', fontsize=12)
    ax2.set_ylabel(r'Error in $K_I$ [%]', fontsize=12)
    ax2.set_title('Straight Crack: Percent Error in $K_I$', fontsize=14, weight='bold')
    ax2.set_xscale('log')
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=11)
    plt.tight_layout()
    fig2.savefig(out_dir / "straight_KI_error.png", dpi=150, bbox_inches='tight')
    plt.show()

    # Plot 3: K_II (should be near zero for Mode I loading)
    fig3, ax3 = plt.subplots(figsize=(10, 6))
    ax3.plot(c*1e3, KII_avg*scale, 's-', lw=2, markersize=8, label=f'Numerical (ne_half={ne_half})')
    ax3.axhline(0.0, color='k', linestyle='--', lw=1, alpha=0.5, label='Expected: $K_{II} = 0$')
    ax3.set_xlabel(r'$c$ [mm] (crack half-length)', fontsize=12)
    ax3.set_ylabel(r'$K_{II}$ [MPa$\sqrt{m}$]', fontsize=12)
    ax3.set_title('Straight Crack: $K_{II}$ vs Crack Size (should be ≈ 0)', fontsize=14, weight='bold')
    ax3.set_xscale('log')
    ax3.grid(True, alpha=0.3)
    ax3.legend(fontsize=11)
    plt.tight_layout()
    fig3.savefig(out_dir / "straight_KII.png", dpi=150, bbox_inches='tight')
    plt.show()

    # Print summary table
    print("\n" + "="*70)
    print("STRAIGHT CRACK VALIDATION SUMMARY")
    print("="*70)
    print(f"{'c [mm]':<12} {'K_I Analytical':<18} {'K_I Numerical':<18} {'Error [%]':<12}")
    print("-"*70)
    for i, length in enumerate(c):
        print(f"{length*1e3:<12.2f} {K_analytical[i]*scale:<18.4f} {KI_avg[i]*scale:<18.4f} {error_KI[i]:<12.2f}")
    print("="*70)

    print("\n" + "="*60)
    print("Straight crack validation complete!")
    print(f"Results saved to: {out_dir}")
    print("="*60)

    # ---- next cell ----

    ## BRANCHED CRACK VALIDATION: Lo J. Applied Mechanics (1978) 45, p797
    # ============================================================
    # Symmetric Branched Crack (robust Euclidean near-tip fit)
    # ============================================================

    import os, sys
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt

    # Add repo root (one level above notebooks/) to sys.path
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))


    out_dir = Path(repo_root) / "output" / "Branched_Crack_Validation"
    out_dir.mkdir(parents=True, exist_ok=True)

    # ----------------------------
    # Problem definition
    # ----------------------------
    mm = 1e-3
    c = 1000.0 * mm
    # c_over_l = np.linspace(1, 201, 10, dtype=float)
    c_over_l = np.array([1,2,3,4,5,6,7,8,9,10,15,20,30,40,50, 75, 100, 200], float)
    theta_deg = 15.0
    theta = np.deg2rad(theta_deg)

    # Material and loading
    E = 200e9
    nu = 0.30
    plane = "strain"
    sig_xx = 0e6
    sig_yy = 100e6
    sig_xy = 0.0

    def print_meta(label, ratio, meta, l):
        print(
            f"[{label}] c/l={ratio:g} | "
            f"keys={list(meta.keys())} | "
            f"rr_size={meta.get('rr_size', 'NA')} | "
            f"rmax/l={meta.get('rmax', np.nan)/l if 'rmax' in meta else np.nan:.3f}"
        )

    # ----------------------------
    # Build a symmetric branched crack network
    # ----------------------------
    def build_branched_crack(c, l, theta):
        vertices = np.array([
            [0,  -c,  0.0],
            [1,   c,  0.0],
            [2,   c + l*np.cos(theta),   l*np.sin(theta)],
            [3,   c + l*np.cos(theta),  -l*np.sin(theta)],
        ], dtype=float)

        connectivity = np.array([
            [0, 1],   # e0 main
            [1, 2],   # e1 top branch
            [1, 3],   # e2 bottom branch
        ], dtype=int)

        net = CrackNetworkV4.from_vertices_connectivity(
            vertices=vertices,
            connectivity=connectivity,
            Nv_max=4,
            validate=True
        )
        return net, vertices

    # ----------------------------
    # Run sweep over c/l ratios
    # ----------------------------
    KI_top, KII_top = [], []
    KI_bot, KII_bot = [], []
    KI_main, KII_main = [], []

    material = Material(E=E, nu=nu, plane_stress=False)
    applied = AppliedStress(sigma_xx=sig_xx, sigma_yy=sig_yy, sigma_xy=sig_xy)

    mu = float(material.mu)

    # Euclid-fit controls (robust defaults)
    rmax_frac_branch = 0.05   # branches can be short => allow a bit wider
    rmax_frac_main   = 0.05
    min_pts = 4
    two_term = True
    ne_half = 40

    for ratio in c_over_l:
        l = c / ratio

        print(f"Solving for c/l = {ratio:g} (l = {l*1e3:.3f} mm)...")

        net, vertices = build_branched_crack(c, l, theta)

        calc = DCENetworkStaticV4(material, net, applied)
        
        sol = calc.solve(
            ne_half=ne_half,
            crack_mode="half",
            representation="cheb_quad",
            node_distribution="uniform",      # recommended for non-tip segments in this strategy
            nq_col=12,
            nq_stress=16,
            solver_option="parametrized_crack",
            parametrization="polyline",
            junction_model="strict",

            # NEW (tip refinement)
            tip_min_nodes=6,                 # enforce >=6 nodes on each tip segment
            tip_cluster="power",              # "power" (one-sided) or "cheb" (symmetric)
            tip_cluster_power=3.0,            # stronger clustering as p increases (>=2)
            other_min_panels=4,               # baseline per segment
            endpoint_min_nodes=6,        # new
            kink_side_nodes=3,           # new
            refine_junction_endpoints=True,
            refine_kinks=True,
            )
        res = DCEResultsNetworkV4(calc, sol)

        # tip coordinates (global)
        p_top  = vertices[2, 1:3]
        p_bot  = vertices[3, 1:3]
        p_left = vertices[0, 1:3]

        # --- Top branch tip (edge 1, tip at v2)
        try:
            KI_t, KII_t, KI_t_CR, KII_t_CR, meta_t = euclid_tip_fit_from_edge(
                res, edge_index=1, tip_xy=p_top,
                a_fit=l, rmax_frac=rmax_frac_branch, min_pts=min_pts, two_term=two_term, theta=theta,
                material=material
            )
            
            KI_top.append(KI_t)
            KII_top.append(KII_t)
        except Exception as e:
            print(f"  Warning: Top branch fit failed: {e}")
            KI_top.append(np.nan); KII_top.append(np.nan)
        
        # --- Bottom branch tip (edge 2, tip at v3)
        try:
            KI_b, KII_b, KI_b_CR, KII_b_CR, meta_b = euclid_tip_fit_from_edge(
                res, edge_index=2, tip_xy=p_bot,
                a_fit=l, rmax_frac=rmax_frac_branch, min_pts=min_pts, two_term=two_term, theta=-theta,
                material=material
            )
            KI_bot.append(KI_b)
            KII_bot.append(KII_b)
        except Exception as e:
            print(f"  Warning: Bottom branch fit failed: {e}")
            KI_bot.append(np.nan); KII_bot.append(np.nan)

        # --- Main crack left tip (edge 0, tip at v0)
        try:
            KI_m, KII_m, KI_m_CR, KII_m_CR, meta_m = euclid_tip_fit_from_edge(
                res, edge_index=0, tip_xy=p_left,
                a_fit=c,
                rmax_frac=rmax_frac_main, min_pts=min_pts, two_term=two_term, theta=0.0,
                material=material
            )
            KI_main.append(KI_m)
            KII_main.append(KII_m)
        except Exception as e:
            print(f"  Warning: Main crack fit failed: {e}")
            KI_main.append(np.nan); KII_main.append(np.nan)
            
    # Convert to arrays
    KI_top = np.array(KI_top);   KII_top = np.array(KII_top)
    KI_bot = np.array(KI_bot);   KII_bot = np.array(KII_bot)
    KI_main = np.array(KI_main); KII_main = np.array(KII_main)

    # Normalize by reference SIF
    K_ref = sig_yy * np.sqrt(np.pi * c)
    KI_branch_norm = KI_top / K_ref
    KII_branch_norm = KII_top / K_ref
    KI_main_norm = KI_main / K_ref
    KII_main_norm = KII_main / K_ref

    # -------------------------
    # Lo (JAM 1978) digitized data (branch SIFs)
    # -------------------------
    lo_KI = np.array([
        [0.977157764, 0.888603531],
        [1.22156035 , 0.858908507],
        [1.754476973, 0.820385233],
        [2.494862119, 0.792295345],
        [4.855432594, 0.755377207],
        [8.929324781, 0.736918138],
        [20.46800156, 0.728892456],
        [60.6877622 , 0.728089888],
        [178.2899264, 0.727287319],
    ], float)

    lo_KII = np.array([
        [0.995465377, 0.093361835],
        [1.306903865, 0.076523857],
        [1.683851068, 0.061291159],
        [2.450969943, 0.040453082],
        [3.918340794, 0.017222372],
        [6.626457855, -0.005190409],
        [10.20229795, -0.021991258],
        [31.44051859, -0.053922155],
        [84.94829929, -0.07220161],
    ], float)

    lo_cl_KI,  lo_KI_norm  = lo_KI[:,0],  lo_KI[:,1]
    lo_cl_KII, lo_KII_norm = lo_KII[:,0], lo_KII[:,1]

    def interp_logx(xq, x, y):
        """
        Interpolate y(x) onto query points xq using linear interpolation in log10(x).
        Assumes x > 0 and sorted ascending.
        """
        xq = np.asarray(xq, float)
        x  = np.asarray(x, float)
        y  = np.asarray(y, float)

        m = (xq > 0) & (xq >= x.min()) & (xq <= x.max())
        yq = np.full_like(xq, np.nan, dtype=float)
        yq[m] = np.interp(np.log10(xq[m]), np.log10(x), y)
        return yq

    loKI_on_grid  = interp_logx(c_over_l, lo_cl_KI,  lo_KI_norm)
    loKII_on_grid = interp_logx(c_over_l, lo_cl_KII, lo_KII_norm)

    # -------------------------
    # Plots
    # -------------------------
    fig1, ax1 = plt.subplots(figsize=(10, 6))
    # Numerical (solid line, no symbols)
    ax1.plot(c_over_l, KI_branch_norm, '-', lw=2.5, label=f'VCM (branch, ne_half={ne_half})')
    # Lo overlay: blue circle edge, red fill, no line
    ax1.plot(
        lo_cl_KI, lo_KI_norm,
        linestyle='None',
        marker='o', markersize=7,
        markerfacecolor='red',
        markeredgecolor='blue',
        markeredgewidth=1.5,
        label="Lo (JAM 1978)"
    )
    ax1.set_xlabel(r'$c/l$', fontsize=12)
    ax1.set_ylabel(r'$K_I / (\sigma_\infty \sqrt{\pi c})$', fontsize=12)
    ax1.set_title(f'Branch tip: Normalized $K_I$ (θ={theta_deg}°)', fontsize=14, weight='bold')
    ax1.set_ylim(bottom=0.0)  # <-- start y-axis at 0 for KI
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    plt.tight_layout()
    fig1.savefig(out_dir / "branched_KI_vs_Lo.png", dpi=150, bbox_inches='tight')
    plt.show()

    fig2, ax2 = plt.subplots(figsize=(10, 6))
    ax2.plot(c_over_l, KII_branch_norm, '-', lw=2.5, label=f'VCM (branch, ne_half={ne_half})')
    ax2.plot(
        lo_cl_KII, lo_KII_norm,
        linestyle='None',
        marker='o', markersize=7,
        markerfacecolor='red',
        markeredgecolor='blue',
        markeredgewidth=1.5,
        label="Lo (JAM 1978)"
    )
    ax2.set_xlabel(r'$c/l$', fontsize=12)
    ax2.set_ylabel(r'$K_{II} / (\sigma_\infty \sqrt{\pi c})$', fontsize=12)
    ax2.set_title(f'Branch tip: Normalized $K_{{II}}$ (θ={theta_deg}°)', fontsize=14, weight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    plt.tight_layout()
    fig2.savefig(out_dir / "branched_KII_vs_Lo.png", dpi=150, bbox_inches='tight')
    plt.show()


    fig3, ax3 = plt.subplots(figsize=(10, 6))
    ax3.plot(c_over_l, KI_main_norm, 'o-', lw=2, markersize=7, label=f'Main crack left tip (ne_half={ne_half})')
    ax3.axhline(1.0, color='k', linestyle='--', lw=2, alpha=0.5, label='Unbranched crack: 1.0')
    ax3.set_xlabel(r'$c/l$', fontsize=12)
    ax3.set_ylabel(r'$K_I / (\sigma_\infty \sqrt{\pi c})$', fontsize=12)
    ax3.set_title(f'Main Crack: Normalized $K_I$ vs $c/l$ (θ={theta_deg}°)', fontsize=14, weight='bold')
    ax3.set_xscale('linear'); ax3.grid(True, alpha=0.3); ax3.legend()
    plt.tight_layout()
    fig3.savefig(out_dir / "main_crack_KI_normalized_lin.png", dpi=150, bbox_inches='tight')
    plt.show()

    err_KI_pct  = 100.0 * (KI_branch_norm  - loKI_on_grid)  / loKI_on_grid
    err_KII_pct = 100.0 * (KII_branch_norm - loKII_on_grid) / loKII_on_grid

    # Optional: mask where Lo isn't defined over your c/l range
    mKI  = np.isfinite(err_KI_pct)
    mKII = np.isfinite(err_KII_pct)

    fige4, axe1 = plt.subplots(figsize=(10, 5))
    axe1.plot(c_over_l[mKI], err_KI_pct[mKI], '-', lw=2.5)
    axe1.axhline(0.0, color='k', lw=1, alpha=0.5)
    axe1.set_xlabel(r'$c/l$', fontsize=12)
    axe1.set_ylabel(r'$\%$ error in $K_I$', fontsize=12)
    axe1.set_title(r'Branch tip: $\%$ error vs Lo (JAM 1978) — $K_I$', fontsize=13, weight='bold')
    axe1.grid(True, alpha=0.3)
    plt.tight_layout()
    fige4.savefig(out_dir / "branched_KI_percent_error_vs_Lo.png", dpi=150, bbox_inches='tight')
    plt.show()

    fige5, axe2 = plt.subplots(figsize=(10, 5))
    axe2.plot(c_over_l[mKII], err_KII_pct[mKII], '-', lw=2.5)
    axe2.axhline(0.0, color='k', lw=1, alpha=0.5)
    axe2.set_xlabel(r'$c/l$', fontsize=12)
    axe2.set_ylabel(r'$\%$ error in $K_{II}$', fontsize=12)
    axe2.set_title(r'Branch tip: $\%$ error vs Lo (JAM 1978) — $K_{II}$', fontsize=13, weight='bold')
    axe2.grid(True, alpha=0.3)
    plt.tight_layout()
    fige5.savefig(out_dir / "branched_KII_percent_error_vs_Lo.png", dpi=150, bbox_inches='tight')
    plt.show()


    print("\n" + "="*60)
    print("Branched crack validation complete!")
    print(f"Results saved to: {out_dir}")
    print("="*60)


    # ---- next cell ----

    ## BRANCHED CRACK VALIDATION: Lo J. Applied Mechanics (1978) 45, p797
    # ============================================================
    # Asymmetric Branched Crack (robust Euclidean near-tip fit)
    # ============================================================

    import os, sys
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt

    # Add repo root (one level above notebooks/) to sys.path
    nb_dir = Path(os.getcwd()).resolve()
    repo_root = nb_dir.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))


    out_dir = Path(repo_root) / "output" / "Branched_Crack_Validation"
    out_dir.mkdir(parents=True, exist_ok=True)

    # ----------------------------
    # Problem definition
    # ----------------------------
    mm = 1e-3
    c = 1000.0 * mm
    # c_over_l = np.linspace(1, 201, 10, dtype=float)
    # c_over_l = np.array([1,2,3,4,5,6,7,8,9,10,15,20,30,40,50, 75, 100, 200], float)
    c_over_l = np.array([1,2,5,10, 20, 50, 100], float)
    theta_deg = 15.0
    theta = np.deg2rad(theta_deg)

    # Material and loading
    E = 200e9
    nu = 0.30
    plane = "strain"
    sig_xx = 0e6
    sig_yy = 100e6
    sig_xy = 0.0

    def print_meta(label, ratio, meta, l):
        print(
            f"[{label}] c/l={ratio:g} | "
            f"keys={list(meta.keys())} | "
            f"rr_size={meta.get('rr_size', 'NA')} | "
            f"rmax/l={meta.get('rmax', np.nan)/l if 'rmax' in meta else np.nan:.3f}"
        )

    # ----------------------------
    # Build a symmetric branched crack network
    # ----------------------------
    def build_branched_crack(c, l, theta):
        vertices = np.array([
            [0,  -2*c,  0.0],
            [1,   0.0,  0.0],
            [2,   l*np.cos(theta),   l*np.sin(theta)],
        ], dtype=float)

        connectivity = np.array([
            [0, 1],   # e0 main
            [1, 2],   # e1 top branch
        ], dtype=int)

        net = CrackNetworkV4.from_vertices_connectivity(
            vertices=vertices,
            connectivity=connectivity,
            Nv_max=4,
            validate=True
        )
        return net, vertices

    # ----------------------------
    # Run sweep over c/l ratios
    # ----------------------------
    KI_top, KII_top = [], []
    KI_bot, KII_bot = [], []
    KI_main, KII_main = [], []

    material = Material(E=E, nu=nu, plane_stress=False)
    applied = AppliedStress(sigma_xx=sig_xx, sigma_yy=sig_yy, sigma_xy=sig_xy)

    mu = float(material.mu)

    # Euclid-fit controls (robust defaults)
    rmax_frac_branch = 0.1   # branches can be short => allow a bit wider
    rmax_frac_main   = 0.1
    min_pts = 8
    two_term = False
    ne_half = 60

    for ratio in c_over_l:
        l = c / ratio

        print(f"Solving for c/l = {ratio:g} (l = {l*1e3:.3f} mm)...")

        net, vertices = build_branched_crack(c, l, theta)

        calc = DCENetworkStaticV4(material, net, applied)
        
        sol = calc.solve(
            ne_half=ne_half,
            crack_mode="full",
            representation="cheb_quad",
            node_distribution="tip_dense",      # recommended for non-tip segments in this strategy
            nq_col=12,
            nq_stress=16,
            solver_option="parametrized_crack",
            parametrization="polyline",
            junction_model="strict",

            # NEW (tip refinement)
            tip_min_nodes=6,                 # enforce >=6 nodes on each tip segment
            tip_cluster="power",              # "power" (one-sided) or "cheb" (symmetric)
            tip_cluster_power=3.0,            # stronger clustering as p increases (>=2)
            other_min_panels=4,               # baseline per segment
            endpoint_min_nodes=6,        # new
            kink_side_nodes=3,           # new
            refine_junction_endpoints=True,
            refine_kinks=True,
            )
        res = DCEResultsNetworkV4(calc, sol)

        # tip coordinates (global)
        p_top  = vertices[2, 1:3]
        p_left = vertices[0, 1:3]

        # --- Top branch tip (edge 1, tip at v2)
        try:
            KI_t, KII_t, KI_t_CR, KII_t_CR, meta_t = euclid_tip_fit_from_edge(
                res, edge_index=1, tip_xy=p_top,
                a_fit=l, rmax_frac=rmax_frac_branch, min_pts=min_pts, two_term=two_term, theta=theta,
                material=material
            )
            
            KI_top.append(KI_t_CR)
            KII_top.append(KII_t_CR)
        except Exception as e:
            print(f"  Warning: Top branch fit failed: {e}")
            KI_top.append(np.nan); KII_top.append(np.nan)


        # --- Main crack left tip (edge 0, tip at v0)
        try:
            KI_m, KII_m, KI_m_CR, KII_m_CR, meta_m = euclid_tip_fit_from_edge(
                res, edge_index=0, tip_xy=p_left,
                a_fit=c,
                rmax_frac=rmax_frac_main, min_pts=min_pts, two_term=two_term, theta=.0,
                material=material
            )
            KI_main.append(KI_m)
            KII_main.append(KII_m)
        except Exception as e:
            print(f"  Warning: Main crack fit failed: {e}")
            KI_main.append(np.nan); KII_main.append(np.nan)
            
    # Convert to arrays
    KI_top = np.array(KI_top);   KII_top = np.array(KII_top)
    KI_main = np.array(KI_main); KII_main = np.array(KII_main)

    # Normalize by reference SIF
    K_ref = sig_yy * np.sqrt(np.pi * c)
    KI_branch_norm = KI_top / K_ref
    KII_branch_norm = KII_top / K_ref
    KI_main_norm = KI_main / K_ref
    KII_main_norm = KII_main / K_ref

    # -------------------------
    # Lo (JAM 1978) digitized data (branch SIFs)
    # -------------------------
    lo_KI = np.array([
        [0.976147056,	1.152648475],
        [1.143765727,	1.126966292],
        [1.378120901,	1.097271268],
        [1.89014178,	1.059550562],
        [2.450847066,	1.033868379],
        [3.326275879,	1.013804173],
        [4.903693017,	0.996147673],
        [7.565839828,	0.985714286],
        [12.56285313,	0.978491172],
        [44.80162225,	0.972070626],
        # [285.3622159,	0.968860353],
    ], float)

    lo_KII = np.array([
        [0.990222752,	0.318712717],
        [1.36272909,	0.29142294],
        [2.002533967,	0.26414845],
        [3.26237831,	0.236093161],
        [5.782060166,	0.212081651],
        [13.3241222,	0.184911996],
        [33.4000804,	0.165810244],
        [127.6576459,	0.145197125],
        # [329.0243998,	0.139783942],
        # [976.0579,	    0.133598696],
    ], float)

    lo_cl_KI,  lo_KI_norm  = lo_KI[:,0],  lo_KI[:,1]
    lo_cl_KII, lo_KII_norm = lo_KII[:,0], lo_KII[:,1]

    def interp_logx(xq, x, y):
        """
        Interpolate y(x) onto query points xq using linear interpolation in log10(x).
        Assumes x > 0 and sorted ascending.
        """
        xq = np.asarray(xq, float)
        x  = np.asarray(x, float)
        y  = np.asarray(y, float)

        m = (xq > 0) & (xq >= x.min()) & (xq <= x.max())
        yq = np.full_like(xq, np.nan, dtype=float)
        yq[m] = np.interp(np.log10(xq[m]), np.log10(x), y)
        return yq

    loKI_on_grid  = interp_logx(c_over_l, lo_cl_KI,  lo_KI_norm)
    loKII_on_grid = interp_logx(c_over_l, lo_cl_KII, lo_KII_norm)

    # -------------------------
    # Plots
    # -------------------------
    fig1, ax1 = plt.subplots(figsize=(10, 6))
    # Numerical (solid line, no symbols)
    ax1.plot(c_over_l, KI_branch_norm, '-', lw=2.5, label=f'VCM (branch, ne_half={ne_half})')
    # Lo overlay: blue circle edge, red fill, no line
    ax1.plot(
        lo_cl_KI, lo_KI_norm,
        linestyle='None',
        marker='o', markersize=7,
        markerfacecolor='red',
        markeredgecolor='blue',
        markeredgewidth=1.5,
        label="Lo (JAM 1978)"
    )
    ax1.set_xlabel(r'$c/l$', fontsize=12)
    ax1.set_ylabel(r'$K_I / (\sigma_\infty \sqrt{\pi c})$', fontsize=12)
    ax1.set_title(f'Branch tip: Normalized $K_I$ (θ={theta_deg}°)', fontsize=14, weight='bold')
    ax1.set_ylim(bottom=0.0)  # <-- start y-axis at 0 for KI
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    plt.tight_layout()
    fig1.savefig(out_dir / "branched_KI_vs_Lo.png", dpi=150, bbox_inches='tight')
    plt.show()

    fig2, ax2 = plt.subplots(figsize=(10, 6))
    ax2.plot(c_over_l, KII_branch_norm, '-', lw=2.5, label=f'VCM (branch, ne_half={ne_half})')
    ax2.plot(
        lo_cl_KII, lo_KII_norm,
        linestyle='None',
        marker='o', markersize=7,
        markerfacecolor='red',
        markeredgecolor='blue',
        markeredgewidth=1.5,
        label="Lo (JAM 1978)"
    )
    ax2.set_xlabel(r'$c/l$', fontsize=12)
    ax2.set_ylabel(r'$K_{II} / (\sigma_\infty \sqrt{\pi c})$', fontsize=12)
    ax2.set_title(f'Branch tip: Normalized $K_{{II}}$ (θ={theta_deg}°)', fontsize=14, weight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    plt.tight_layout()
    fig2.savefig(out_dir / "branched_KII_vs_Lo.png", dpi=150, bbox_inches='tight')
    plt.show()


    fig3, ax3 = plt.subplots(figsize=(10, 6))
    ax3.plot(c_over_l, KI_main_norm, 'o-', lw=2, markersize=7, label=f'Main crack left tip (ne_half={ne_half})')
    ax3.axhline(1.0, color='k', linestyle='--', lw=2, alpha=0.5, label='Unbranched crack: 1.0')
    ax3.set_xlabel(r'$c/l$', fontsize=12)
    ax3.set_ylabel(r'$K_I / (\sigma_\infty \sqrt{\pi c})$', fontsize=12)
    ax3.set_title(f'Main Crack: Normalized $K_I$ vs $c/l$ (θ={theta_deg}°)', fontsize=14, weight='bold')
    ax3.set_xscale('linear'); ax3.grid(True, alpha=0.3); ax3.legend()
    plt.tight_layout()
    fig3.savefig(out_dir / "main_crack_KI_normalized_lin.png", dpi=150, bbox_inches='tight')
    plt.show()

    err_KI_pct  = 100.0 * (KI_branch_norm  - loKI_on_grid)  / loKI_on_grid
    err_KII_pct = 100.0 * (KII_branch_norm - loKII_on_grid) / loKII_on_grid

    # Optional: mask where Lo isn't defined over your c/l range
    mKI  = np.isfinite(err_KI_pct)
    mKII = np.isfinite(err_KII_pct)

    fige4, axe1 = plt.subplots(figsize=(10, 5))
    axe1.plot(c_over_l[mKI], err_KI_pct[mKI], '-', lw=2.5)
    axe1.axhline(0.0, color='k', lw=1, alpha=0.5)
    axe1.set_xlabel(r'$c/l$', fontsize=12)
    axe1.set_ylabel(r'$\%$ error in $K_I$', fontsize=12)
    axe1.set_title(r'Branch tip: $\%$ error vs Lo (JAM 1978) — $K_I$', fontsize=13, weight='bold')
    axe1.grid(True, alpha=0.3)
    plt.tight_layout()
    fige4.savefig(out_dir / "branched_KI_percent_error_vs_Lo.png", dpi=150, bbox_inches='tight')
    plt.show()

    fige5, axe2 = plt.subplots(figsize=(10, 5))
    axe2.plot(c_over_l[mKII], err_KII_pct[mKII], '-', lw=2.5)
    axe2.axhline(0.0, color='k', lw=1, alpha=0.5)
    axe2.set_xlabel(r'$c/l$', fontsize=12)
    axe2.set_ylabel(r'$\%$ error in $K_{II}$', fontsize=12)
    axe2.set_title(r'Branch tip: $\%$ error vs Lo (JAM 1978) — $K_{II}$', fontsize=13, weight='bold')
    axe2.grid(True, alpha=0.3)
    plt.tight_layout()
    fige5.savefig(out_dir / "branched_KII_percent_error_vs_Lo.png", dpi=150, bbox_inches='tight')
    plt.show()


    print("\n" + "="*60)
    print("Branched crack validation complete!")
    print(f"Results saved to: {out_dir}")
    print("="*60)

    return locals()


## 4. Dispatch


In [ ]:
CASES = {
    "bem_displacement": run_bem_displacement,
    "crack_net": run_crack_net,
    "direct_method_verification": run_direct_method_verification,
    "disk_compression_2": run_disk_compression_2,
    "disk_compression_inclined": run_disk_compression_inclined,
    "disk_energetics": run_disk_energetics,
    "disk_experiments": run_disk_experiments,
    "function_diagnostics": run_function_diagnostics,
    "graph_net": run_graph_net,
    "propagation": run_propagation,
    "validation": run_validation,
}

if CASE not in CASES:
    raise ValueError(f"Unknown CASE={CASE!r}. Expected one of {list(CASES)}.")

if DRY_RUN:
    print(f'DRY RUN — case functions defined but not executed.')
    print(f'Set DRY_RUN = False (in the selector cell) and re-run to invoke run_{CASE}().')
    result = None
else:
    result = CASES[CASE]()
    if isinstance(result, dict):
        print(f'Finished case {CASE!r}; returned-locals keys: {list(result)[:8]}...')
    else:
        print(f'Finished case {CASE!r}.')
